# Welcome to Bridging Research-Grade Models and Consumer Wearables: A Personalized Federated Learning Approach to Stress Prediction 

## Getting Started:

First, activate your virtual environment, if necessary, and download all necessary packages: 

In [ ]:
pip install -r requirements.txt

## Verify Installation:
Check that all core libraries are properly installed:

In [ ]:
# Test core imports
import gc
import sys

import numpy as np
import pandas as pd
import torch

print("All packages imported successfully!")
print(f"\nPython: {sys.version.split()[0]}")
print(f"PyTorch: {torch.version}")
print(f"NumPy: {np.version}")
print(f"Pandas: {pd.__version__}")
print(f"\nDevice: {'CUDA' if torch.cuda.is_available() else 'CPU'}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## Data Installation
### Option A: Automatic Download (Linux/MacOS/Bash present) <br>
If you're on Linux or Mac or Git Bash is available, run the cell below to automatically download and extract the WESAD dataset:

In [ ]:
from pathlib import Path
import shutil
import subprocess
import sys

data_dir = Path("data/raw/wesad")

# If already present, skip
if data_dir.exists() and any(data_dir.iterdir()):
    print(f"✅ Dataset already present at: {data_dir.resolve()}")
else:
    # try bash if it exists (Linux/macOS/Git Bash/WSL)
    bash = shutil.which("bash")
    sh_script = Path("scripts/download_dataset.sh")

    if bash and sh_script.exists():
        subprocess.run([bash, str(sh_script)], check=True)
        print("Dataset downloaded via bash script!")
    else:
        print(" 'bash' not found (likely native Windows).")
        print("Proceeding with manual instructions.")

### Option B: Manual Download (All Platforms)

If automatic download doesn't work:

- Download: [WESAD Dataset](https://uni-siegen.sciebo.de/s/HGdUkoNlW1Ub0Gx/download)
- Extract the downloaded zip file
- Place the WESAD folder in: data/raw/wesad/

- Expected structure:

```
data/
└── raw/
    └── wesad/
        └── WESAD/
            ├── S2/
            ├── S3/
            └── ...

```

## Verify Dataset Location

Check if WESAD dataset is present: 

In [ ]:
from pathlib import Path

wesad_path = Path("data/raw/wesad/WESAD")

if wesad_path.exists():
    subjects = sorted(
        [d.name for d in wesad_path.iterdir() if d.is_dir() and d.name.startswith("S")]
    )
    print("WESAD dataset found!")
    print(f"Location: {wesad_path.absolute()}")
    print(f"Subjects ({len(subjects)}): {', '.join(subjects)}")
else:
    print("WESAD dataset NOT found!")
    print(f"Expected at: {wesad_path.absolute()}")
    print("\nPlease download and place the dataset in the correct location.")

## Configuration Setup
Set up your environment configuration:

In [ ]:
# Create .env file if it doesn't exist

from pathlib import Path

from dotenv import load_dotenv

env_file = Path(".env")

if not env_file.exists():
    env_example = Path(".env.example")

    if env_example.exists():
        with open(env_example) as f:
            content = f.read()

        with open(env_file, "w") as f:
            f.write(content)

        print("Created .env from .env.example")

    else:
        print("ℹ️ No .env.example found, using defaults")

else:
    print("ℹ️ .env already exists")

# Load environment variables
load_dotenv()
print("Configuration loaded")

You can now proceed with:

Data Preprocessing - Process raw WESAD signals
Feature Extraction - Extract time/frequency domain features
Model Training - Train baseline and federated models
Evaluation - Analyze results

Continue to the next section below to start preprocessing the data.

## Data Preprocessing

### Load and Merge WESAD Dataset

During this step, the biosensor data from the WESAD directory will be automatically extracted into several merged files in `.pkl` format.<br>

This may take a few minutes depending on the machine.

In [ ]:
from src.data import WESADResampleMerge

print("Initializing WESAD preprocessor...")
preprocessor = WESADResampleMerge(
    dataset_path="data/raw/wesad/WESAD/",
    IDs=[2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 13, 14, 15, 16, 17],
)

# Loads metadata, merges wrist data (resample to BVP rate),chest data and filters labels and invalid temperatures
print("\nStarting preprocessing pipeline...")
preprocessor.resample_merge()

# Load the merged data
print("\nLoading merged datasets...")
chest_df = pd.read_pickle("data/raw/wesad/WESAD/merged/merged_chest.pkl")
wrist_df = pd.read_pickle("data/raw/wesad/WESAD/merged/merged_wrist.pkl")

# Display info
print("\n" + "=" * 70)
print("PREPROCESSING COMPLETE")
print("=" * 70)
print("\nChest Data:")
print(f"  Shape: {chest_df.shape}")
print(f"  Columns: {list(chest_df.columns)}")
print(f"  Memory: {chest_df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

print("\nWrist Data:")
print(f"  Shape: {wrist_df.shape}")
print(f"  Columns: {list(wrist_df.columns)}")
print(f"  Memory: {wrist_df.memory_usage(deep=True).sum() / 10242:.2f} MB")

print("\nLabel Distribution (Chest):")
print(chest_df["label"].value_counts().sort_index())

### Data Validation

In [ ]:
print("QUICK DATA VALIDATION CHECKS:\n")

#  Basic Info
print("\n1. CHEST DATA")
print(f"   Shape: {chest_df.shape}")
print(f"   Memory: {chest_df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
print(f"   Columns: {list(chest_df.columns)}")

print("\n2. WRIST DATA")
print(f"   Shape: {wrist_df.shape}")
print(f"   Memory: {wrist_df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
print(f"   Columns: {list(wrist_df.columns)}")

print("\n3. LABEL DISTRIBUTION")
print("\n   Chest:")
print(chest_df["label"].value_counts().sort_index())
print("\n   Wrist:")
print(wrist_df["label"].value_counts().sort_index())

# Check for Invalid Temps (discovered in EDA notebook 2)
print("\n4. TEMPERATURE VALIDATION")
chest_temp_col = "c_temp" if "c_temp" in chest_df.columns else "temp"
wrist_temp_col = "w_temp"

chest_invalid = (chest_df[chest_temp_col] <= 0).sum()
wrist_invalid = (wrist_df[wrist_temp_col] <= 0).sum()

print(f"   Chest temp ≤0: {chest_invalid} (should be 0)")
print(f"   Wrist temp ≤0: {wrist_invalid} (should be 0)")

if chest_invalid == 0 and wrist_invalid == 0:
    print(" Temperature interpolation successful!")
else:
    print(" Warning: Still have invalid temperatures")

# Check for Invalid eda values (discovered in EDA notebook 2)
print("\n5. EDA VALIDATION")
chest_eda_col = "c_eda" if "c_eda" in chest_df.columns else "eda"
wrist_eda_col = "w_eda"

chest_invalid = (chest_df[chest_eda_col] < 0).sum()
wrist_invalid = (wrist_df[wrist_eda_col] < 0).sum()

print(f"   Chest eda <0: {chest_invalid} (should be 0)")
print(f"   Wrist eda <0: {wrist_invalid} (should be 0)")

if chest_invalid == 0 and wrist_invalid == 0:
    print(" EDA interpolation successful! No negative artifacts.")
else:
    print(" Warning: Still have invalid eda artifacts")


print("\n6. MISSING DATA")
chest_nans = chest_df.isnull().sum()
wrist_nans = wrist_df.isnull().sum()

if chest_nans.sum() > 0:
    print(f"   Chest NaNs:\n{chest_nans[chest_nans > 0]}")
else:
    print("   Chest: No missing values")

if wrist_nans.sum() > 0:
    print(f"   Wrist NaNs:\n{wrist_nans[wrist_nans > 0]}")
else:
    print("   Wrist: No missing values")

print("\nData validation complete!")

## Signal Preprocessing 

We will preprocess each signal with their respective preprocessing steps, using NeuroKit2 for signals with peer-reviewed processing modules, and manual filters (Savitzky–Golay, FIR) for signals without standardised processing modules in the NK2 library.

### Chest Signals
*(All chest signals are at 700 Hz native sampling rate)*

#### ECG
- **Filtering**: 5th-order Butterworth high-pass at 0.5 Hz + 50 Hz notch filter (via NeuroKit2 `ecg_clean()`, "neurokit" method). Preserves full QRS morphology for downstream R-peak detection and HRV feature extraction.

#### EMG
- **Filtering**: 4th-order Butterworth high-pass at 100 Hz + constant detrending (via NeuroKit2 `emg_clean()`, "biosppy" method). The 100 Hz cutoff intentionally removes the lower EMG register to fully eliminate ECG contamination, leaving only muscle activation in the 100–500 Hz range.

#### EDA (Chest)
- **Filtering**: Butterworth low-pass at 3 Hz (via NeuroKit2 `eda_clean()`, "neurokit" method). At 700 Hz the 3 Hz cutoff is well within Nyquist, enabling proper tonic/phasic decomposition via `eda_phasic()` downstream.

#### TEMP (Chest)
- **Smoothing**: Savitzky–Golay filter with window size 11 and polynomial order 3. Temperature is a very slow-changing signal (changes over minutes) and no dedicated temperature module exists in NeuroKit2 or BioSPPy.

#### RESP
- **Filtering**: 2nd-order Butterworth band-pass at 0.05–3 Hz (via NeuroKit2 `rsp_clean()`, "khodadad2018" method). This preserves breathing rates from 3 to 180 breaths/min, which is preferable for stress detection over the tighter BioSPPy range (0.1–0.35 Hz ≈ 6–21 breaths/min), since stress-induced tachypnea can exceed 30 breaths/min (0.5 Hz).

#### ACC (Chest)
- **Smoothing**: Savitzky–Golay filter with window size 31 and polynomial order 5 (applied per axis: X, Y, Z). Removes high-frequency sensor noise while preserving body motion dynamics.

---

### Wrist Signals
*(All wrist signals are upsampled to 64 Hz in the resampling stage)*

#### BVP
- **Filtering**: Butterworth low-pass filter (via NeuroKit2 `ppg_clean()`, "elgendi" method, 64 Hz). BVP and PPG are physiologically identical signals, so the PPG pipeline is the correct processing family. The cleaned signal is suitable for downstream peak detection and HRV/IBI feature extraction.

#### EDA (Wrist)
- **Filtering**: Butterworth low-pass at 3 Hz (via NeuroKit2 `eda_clean()`, "neurokit" method, 64 Hz). At the native 4 Hz rate, Nyquist (2 Hz) sat below the 3 Hz cutoff, making any filter a no-op. After upsampling to 64 Hz (Nyquist = 32 Hz), the 3 Hz low-pass preserves all genuine EDA content while removing interpolation artifacts introduced by resampling.

#### TEMP (Wrist)
- **Smoothing**: Savitzky–Golay filter with polynomial order 3, window size scaled proportionally from the native-rate window (native 4 Hz → window 11, upsampled 64 Hz → window 176 + 1 = 177 to enforce odd-length requirement). Window scaling is necessary because Savitzky–Golay is sample-count-based rather than frequency-based.

#### ACC (Wrist)
- **Filtering**: FIR low-pass filter with 128 taps and a cutoff of 0.4 Hz normalised to Nyquist, applied via `filtfilt()` for zero-phase distortion (per axis: X, Y, Z). Designed to isolate slow orientation changes from hand tremor and high-frequency vibrations.

In [ ]:
# set config_files
CHEST_CONFIG = "config_files/dataset/wesad_chest_configuration.json"
WRIST_CONFIG = "config_files/dataset/wesad_wrist_configuration.json"

In [ ]:
from src.data import SignalPreprocessor

# preprocess the chest data
signal_preprocessor = SignalPreprocessor(
    "data/raw/wesad/WESAD/merged/merged_chest.pkl",
    "data/processed/wesad/chest_preprocessed.pkl",
    CHEST_CONFIG,
)
signal_preprocessor.preprocess_signals()

# preprocess the wrist data
signal_preprocessor = SignalPreprocessor(
    "data/raw/wesad/WESAD/merged/merged_wrist.pkl",
    "data/processed/wesad/wrist_preprocessed.pkl",
    WRIST_CONFIG,
    wrist=True,
)
signal_preprocessor.preprocess_signals()

## Data Windowing & Segmentation

Segments preprocessed signals into fixed-length windows for feature extraction.

Design decisions (justified by literature):

### Window size:
-   60s for all signals, Schmidt et al. (2018) WESAD benchmark.
-   This is sufficient for time-domain HRV, Poincaré, and LF/HF frequency-domain features.
-   VLF/ULF frequency-domain HRV features require ≥300s and are dropped (as in multiple WESAD studies the dataset conditions are too short for reliable VLF estimation).

### ACC sub-windows:
-   5s sub-windows within each 60s window, per Schmidt et al. (2018).
-   ACC is high-frequency motion data so 60s averaging washes out patterns.
-   Feature extraction will aggregate sub-window features (mean/std) into the 60s window.
 
### Slide / overlap:
-   Configurable. Schmidt benchmark uses 0.25s shift.
-  For RF, high overlap introduces correlated samples which according to literature is not ideal for tree-based models.
-   Therefore ablation is supported: 0.25s, 2s, 5s, 20s, 60s (no overlap).
 
### Label purity:
-   Windows spanning multiple labels are rejected. This avoids mixed label windows where part of the data is baseline and part is stress. We check this strictly.

### Subject isolation:
-   Windowing is always done per-subject. No window ever crosses subject boundaries.

In [ ]:
from src.windowing.wesad_windowing import WESADWindowing, summarize_ablation

# Paths
CHEST_PREPROCESSED = "data/processed/wesad/chest_preprocessed.pkl"
WRIST_PREPROCESSED = "data/processed/wesad/wrist_preprocessed.pkl"

CHEST_CONFIG = "config_files/dataset/wesad_chest_configuration.json"
WRIST_CONFIG = "config_files/dataset/wesad_wrist_configuration.json"

WINDOWED_OUTPUT_DIR = "data/processed/wesad/windowed/"

# Windowing parameters
WINDOW_SIZE_SEC = (
    60.0  # 60s window, sufficient for HRV time/freq/nonlinear features (Schmidt 2018)
)
ACC_SUBWINDOW_SEC = 5.0  # ACC sub-windowed at 5s per Schmidt et al. (2018)
SLIDE_SEC = 5.0  # changed from 0.25 for RF baseline, ablation below tests alternatives

# Chest windowing
print("\nCHEST DEVICE\n")

chest_windower = WESADWindowing(
    data_path=CHEST_PREPROCESSED,
    config_path=CHEST_CONFIG,
    output_dir=WINDOWED_OUTPUT_DIR,
    window_size_sec=WINDOW_SIZE_SEC,
    slide_sec=SLIDE_SEC,
    acc_subwindow_sec=ACC_SUBWINDOW_SEC,
    wrist=False,
)

chest_windower.load_data()
chest_windows = chest_windower.segment()
chest_windows_path, chest_meta_path = chest_windower.save_windows(chest_windows)

# Verify: materialise one window on demand to confirm the indexing works
sample = chest_windows[0]
sample_df = chest_windower.get_window_df(sample)
print(
    f"Sample window: sid: {sample['sid']}, label: {sample['label']}, "
    f"shape: {sample_df.shape}, is_overlapped: {sample['is_overlapped']}"
)

# ACC sub-windows extracted on demand example
acc_subs = chest_windower.extract_acc_subwindows(sample_df)
print(f"ACC sub-windows: {len(acc_subs)} × {acc_subs[0].shape if acc_subs else 'N/A'}")

# Wrist windowing
print("\nWRIST DEVICE\n")

wrist_windower = WESADWindowing(
    data_path=WRIST_PREPROCESSED,
    config_path=WRIST_CONFIG,
    output_dir=WINDOWED_OUTPUT_DIR,
    window_size_sec=WINDOW_SIZE_SEC,
    slide_sec=SLIDE_SEC,
    acc_subwindow_sec=ACC_SUBWINDOW_SEC,
    wrist=True,
)

wrist_windower.load_data()
wrist_windows = wrist_windower.segment()
wrist_windows_path, wrist_meta_path = wrist_windower.save_windows(wrist_windows)

# verify
sample_w = wrist_windows[0]
sample_w_df = wrist_windower.get_window_df(sample_w)
print(
    f"Sample window: sid: {sample_w['sid']}, label: {sample_w['label']}, "
    f"shape: {sample_w_df.shape}, is_overlapped: {sample_w['is_overlapped']}"
)
acc_subs_w = wrist_windower.extract_acc_subwindows(sample_w_df)
print(
    f"ACC sub-windows: {len(acc_subs_w)} × {acc_subs_w[0].shape if acc_subs_w else 'N/A'}"
)

# Ablation study (RF slide sensitivity)
# For the Random Forest, high overlap introduces correlated training samples, according to literature.
# We test five values to identify the best trade-off between sample count
# and inter-sample independence. The benchmark 0.25s is included.
#
# is_overlapped flag ensures validation sets only ever use independent windows

print("\nABLATION — CHEST\n")

chest_ablation_results = chest_windower.run_ablation(
    slide_values=[0.25, 2.0, 5.0, 20.0, 60.0]
)
summarize_ablation(chest_ablation_results, window_size_sec=WINDOW_SIZE_SEC)

# Release chest data before wrist ablation to avoid holding both in memory
del chest_windower.df
gc.collect()

print("\nABLATION — WRIST\n")

wrist_ablation_results = wrist_windower.run_ablation(
    slide_values=[0.25, 2.0, 5.0, 20.0, 60.0]
)
summarize_ablation(wrist_ablation_results, window_size_sec=WINDOW_SIZE_SEC)

## Manual Feature Extraction

Extracts handcrafted features from each windowed segment for the Random Forest baseline model.

**Strategy per sensor:**
- **ECG** (700 Hz): HRV time-domain (RMSSD, SDNN, pNN50), frequency-domain (LF, HF — VLF/ULF dropped, require ≥300s), nonlinear (SD1/SD2, SampEn), wave morphology (P/Q/S/T), ECG-derived respiration (EDR)
- **EMG** (700 Hz): Statistical features + spectral band energy (100–350 Hz, above the 100 Hz high-pass cutoff applied in preprocessing)
- **EDA** (700 Hz chest / 64 Hz wrist): tonic/phasic decomposition via cvxEDA (SCL mean/std, SCR peak count and amplitude)
- **BVP** (64 Hz): PPG-derived HRV time, frequency, and nonlinear features — primary bridge signal to the Fitbit
- **RESP** (700 Hz): breathing rate, inhalation/exhalation durations, I/E ratio
- **TEMP** (both): mean, std, min, max, range, slope
- **ACC** (both): features on 5s sub-windows aggregated (mean/std) per Schmidt et al. (2018)

**Post-extraction:**
- Per-subject min-max normalization using baseline condition (label 1.0) as reference - follows Schmidt (2018) and prevents inter-subject leakage
- Median imputation for NaN values produced by NeuroKit2 on short/irregular windows

In [ ]:
import gc

from src.ml_pipeline.feature_extraction import run_feature_pipeline_streaming

# Output paths
CHEST_FEATURES_OUT = "data/processed/wesad/features/chest_features.pkl"
WRIST_FEATURES_OUT = "data/processed/wesad/features/wrist_features.pkl"

# Paths to the preprocessed merged pickles (NOT loaded into RAM)
CHEST_DATA_PATH = "data/processed/wesad/chest_preprocessed.pkl"
WRIST_DATA_PATH = "data/processed/wesad/wrist_preprocessed.pkl"

# Chest:
chest_features = run_feature_pipeline_streaming(
    data_path=CHEST_DATA_PATH,
    windows=chest_windows,
    config_path=CHEST_CONFIG,
    output_path=CHEST_FEATURES_OUT,
    wrist=False,
    baseline_label=1.0,
    normalize=True,
    impute=False,
    impute_strategy="median",
)
gc.collect()

# Wrist:
wrist_features = run_feature_pipeline_streaming(
    data_path=WRIST_DATA_PATH,
    windows=wrist_windows,
    config_path=WRIST_CONFIG,
    output_path=WRIST_FEATURES_OUT,
    wrist=True,
    baseline_label=1.0,
    normalize=True,
    impute=False,
    impute_strategy="median",
)
gc.collect()

### Verification

In [ ]:
import pandas as pd

CHEST_FEATURES_OUT = "data/processed/wesad/features/chest_features.pkl"
WRIST_FEATURES_OUT = "data/processed/wesad/features/wrist_features.pkl"

chest_features = pd.read_pickle(CHEST_FEATURES_OUT)
wrist_features = pd.read_pickle(WRIST_FEATURES_OUT)

meta_cols = ["sid", "label", "window_idx", "is_overlapped"]
for name, df in [("CHEST", chest_features), ("WRIST", wrist_features)]:
    feat_cols = [c for c in df.columns if c not in meta_cols]
    print(f"\n{name} FEATURE MATRIX")
    print(f"  Shape                : {df.shape}")
    print(f"  Feature count        : {len(feat_cols)}")
    print(f"  Subjects             : {sorted(df['sid'].unique())}")
    print(
        f"  Label distribution   : {df['label'].value_counts().sort_index().to_dict()}"
    )
    print(f"  NaN count            : {df[feat_cols].isna().sum().sum()}")
    print(
        f"  Inf count            : {(df[feat_cols].abs() == float('inf')).sum().sum()}"
    )
    print(f"  Independent windows  : {(~df['is_overlapped']).sum()}")
    print(f"  Overlapped windows   : {df['is_overlapped'].sum()}")
    print("\n  Feature value ranges (first 10 features):")
    print(
        df[feat_cols].describe().T[["mean", "std", "min", "max"]].head(10).to_string()
    )

print("\nSave/load integrity check passed.")

# Model Training & Evaluation : WESAD RF Pipeline  


## 1. Load Configs

In [ ]:
# Setup
import gc
import os
import pickle
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid", font_scale=1.1)
plt.rcParams["figure.dpi"] = 120

# Paths
CHEST_FEATURES = "data/processed/wesad/features/chest_features.pkl"
WRIST_FEATURES = "data/processed/wesad/features/wrist_features.pkl"

CHEST_PREPROCESSED = "data/processed/wesad/chest_preprocessed.pkl"
WRIST_PREPROCESSED = "data/processed/wesad/wrist_preprocessed.pkl"

CHEST_CONFIG = "config_files/dataset/wesad_chest_configuration.json"
WRIST_CONFIG = "config_files/dataset/wesad_wrist_configuration.json"

WINDOWED_OUTPUT_DIR = "data/processed/wesad/windowed/"

# Results directories
RESULTS_BASE = "results/binary/"
os.makedirs(RESULTS_BASE, exist_ok=True)

META_COLS = ["sid", "label", "window_idx", "is_overlapped"]

print("RF Baseline training configuration ready.")

In [ ]:
# Load feature matrices
df_chest = pd.read_pickle(CHEST_FEATURES)
df_wrist = pd.read_pickle(WRIST_FEATURES)

for name, df in [("CHEST", df_chest), ("WRIST", df_wrist)]:
    fcols = [c for c in df.columns if c not in META_COLS]
    print(
        f"{name}: {df.shape[0]:,} windows, {len(fcols)} features, "
        f"{len(df['sid'].unique())} subjects"
    )
    print(f"  Labels: {df['label'].value_counts().sort_index().to_dict()}")
print()

## 2. Feature Selection Ablation: Chest

Runs full LOSOCV (15 folds, nested RandomizedSearchCV) with each FS method:
- **none**: all features, no selection
- **tree**: RF-based importance, keep features above mean importance
- **kbest**: univariate ANOVA F-test, top-k features
- **pca**: PCA to 95% variance explained


In [ ]:
from src.training.ml_classifier import LABEL_MAP_BINARY, run_fs_ablation

# Chest FS ablation:
print("CHEST — Feature Selection Ablation (Binary)")
print("=" * 70)

chest_fs_results = run_fs_ablation(
    df_features=df_chest,
    label_map=LABEL_MAP_BINARY,
    methods=["none", "tree", "kbest", "pca"],
    output_base=os.path.join(RESULTS_BASE, "chest_fs_ablation"),
)

## 3. Feature Selection Ablation: Wrist

Same ablation on wrist features. A lower performance than chest is expected because of fewer modalities, noisier signals and a lower sampling rate.

In [ ]:
print("WRIST — Feature Selection Ablation (Binary)")
print("=" * 70)

wrist_fs_results = run_fs_ablation(
    df_features=df_wrist,
    label_map=LABEL_MAP_BINARY,
    methods=["none", "tree", "kbest", "pca"],
    output_base=os.path.join(RESULTS_BASE, "wrist_fs_ablation"),
)

## Normalization Ablation 

Per-subject baseline normalization scales each subject's features so that
baseline sits at [0, 1]. Stress features then naturally exceed 1.0, making
classification near-trivial. This ablation extracts features **without**
normalization and re-runs LOSOCV to measure how much performance comes
from normalization vs genuine feature discriminability.

In [ ]:
import gc
import os

import pandas as pd

from src.ml_pipeline.feature_extraction import run_feature_pipeline_streaming
from src.windowing.wesad_windowing import WESADWindowing

CHEST_PREPROCESSED = "data/processed/wesad/chest_preprocessed.pkl"
WRIST_PREPROCESSED = "data/processed/wesad/wrist_preprocessed.pkl"
CHEST_CONFIG = "config_files/dataset/wesad_chest_configuration.json"
WRIST_CONFIG = "config_files/dataset/wesad_wrist_configuration.json"
WINDOWED_OUTPUT_DIR = "data/processed/wesad/windowed/"

# Output paths for unnormalized features
CHEST_FEATURES_UNNORM = "data/processed/wesad/features/chest_features_unnormalized.pkl"
WRIST_FEATURES_UNNORM = "data/processed/wesad/features/wrist_features_unnormalized.pkl"

WINDOW_SIZE_SEC = 60.0
SLIDE_SEC = 5.0
ACC_SUBWINDOW_SEC = 5.0

devices = [
    {
        "name": "chest",
        "preprocessed": CHEST_PREPROCESSED,
        "config": CHEST_CONFIG,
        "output": CHEST_FEATURES_UNNORM,
        "wrist": False,
    },
    {
        "name": "wrist",
        "preprocessed": WRIST_PREPROCESSED,
        "config": WRIST_CONFIG,
        "output": WRIST_FEATURES_UNNORM,
        "wrist": True,
    },
]

for dev in devices:
    if os.path.exists(dev["output"]):
        df_check = pd.read_pickle(dev["output"])
        print(f"{dev['name']}: EXISTS ({len(df_check):,} windows) — skipping")
        del df_check
        continue

    print(f"{dev['name'].upper()} — extracting WITHOUT normalization")

    # Windowing
    windower = WESADWindowing(
        data_path=dev["preprocessed"],
        config_path=dev["config"],
        output_dir=WINDOWED_OUTPUT_DIR,
        window_size_sec=WINDOW_SIZE_SEC,
        slide_sec=SLIDE_SEC,
        acc_subwindow_sec=ACC_SUBWINDOW_SEC,
        wrist=dev["wrist"],
    )
    windower.load_data()
    windows = windower.segment()

    # Feature extraction — normalize=False, impute=False (impute inside CV)
    df_feat = run_feature_pipeline_streaming(
        data_path=dev["preprocessed"],
        windows=windows,
        config_path=dev["config"],
        output_path=dev["output"],
        wrist=dev["wrist"],
        baseline_label=1.0,
        normalize=False,
        impute=False,
    )

    print(f"Saved: {dev['output']} ({len(df_feat):,} windows)")
    del windower, windows, df_feat
    gc.collect()

print("\nUnnormalized feature extraction complete.")

In [ ]:
# Run LOSOCV on unnormalized features
from src.training.ml_classifier import LABEL_MAP_BINARY, WESADClassifier

META_COLS = ["sid", "label", "window_idx", "is_overlapped"]
RESULTS_BASE = "results/binary/"

norm_ablation_results = {}

for dev_name, feat_path in [
    ("chest", CHEST_FEATURES_UNNORM),
    ("wrist", WRIST_FEATURES_UNNORM),
]:
    if not os.path.exists(feat_path):
        print(f"{dev_name}: feature file not found — skipping")
        continue

    print(f"{dev_name.upper()} — LOSOCV (UNNORMALIZED, FS=tree)")

    df_feat = pd.read_pickle(feat_path)
    print(f"Loaded: {len(df_feat):,} windows")

    output_dir = os.path.join(RESULTS_BASE, "norm_ablation", dev_name, "unnormalized")

    classifier = WESADClassifier(
        df_features=df_feat,
        label_map=LABEL_MAP_BINARY,
        feature_selection_method="tree",
        output_dir=output_dir,
    )
    results = classifier.run_losocv()
    norm_ablation_results[dev_name] = results

    del df_feat, classifier
    gc.collect()

In [ ]:
# Compare normalized vs unnormalized
import numpy as np

# Load normalized results (from FS ablation, tree method)
print("\nNORMALIZATION ABLATION — COMPARISON")
print("=" * 80)

for dev_name in ["chest", "wrist"]:
    # Load normalized results (tree FS)
    norm_path = os.path.join(
        RESULTS_BASE, f"{dev_name}_fs_ablation", "fs_tree", "losocv_results.pkl"
    )
    if os.path.exists(norm_path):
        with open(norm_path, "rb") as f:
            norm_res = pickle.load(f)
        norm_agg = norm_res["aggregate"]
    else:
        print(f"  {dev_name} normalized results not found at {norm_path}")
        continue

    if dev_name in norm_ablation_results:
        unnorm_agg = norm_ablation_results[dev_name]["aggregate"]
    else:
        print(f"  {dev_name} unnormalized results not available")
        continue

    print(f"\n  {dev_name.upper()}:")
    print(f"  {'Metric':<20} {'Normalized':<25} {'Unnormalized':<25} {'Delta':<10}")
    print("\n")
    for metric in ["accuracy", "f1_weighted", "f1_macro"]:
        n_mean = norm_agg[f"{metric}_mean"]
        n_std = norm_agg[f"{metric}_std"]
        u_mean = unnorm_agg[f"{metric}_mean"]
        u_std = unnorm_agg[f"{metric}_std"]
        delta = n_mean - u_mean
        print(
            f"  {metric:<20} {n_mean:.4f} +/- {n_std:.4f}      "
            f"{u_mean:.4f} +/- {u_std:.4f}      {delta:+.4f}"
        )

    print()
    print(
        f"  Interpretation: normalization accounts for ~{(norm_agg['f1_macro_mean'] - unnorm_agg['f1_macro_mean'])*100:.1f} "
        f"percentage points of F1 macro performance."
    )

In [ ]:
# ─────────────────────────────────────────────────────────────────
# DIAGNOSTIC ABLATION: Unnormalized + Independent Windows Only
#
# Purpose: isolate the two suspected causes of overfitting:
#   1. Per-subject baseline normalization (computed before LOSO split)
#   2. Highly correlated overlapped training windows
#
# This cell uses the already-extracted unnormalized feature files
# from Cell 35 (CHEST_FEATURES_UNNORM / WRIST_FEATURES_UNNORM).
# It filters to is_overlapped == False BEFORE passing to the
# classifier, so the RF trains on independent windows only.
#
# Expected result:
#   - If scores drop significantly vs. the overlapped run →
#     overlapped training windows were a major contributor.
#   - If scores are still suspiciously high →
#     some other leakage source exists (worth investigating further).
# ─────────────────────────────────────────────────────────────────

import os

import pandas as pd

from src.training.ml_classifier import LABEL_MAP_BINARY, WESADClassifier

META_COLS = ["sid", "label", "window_idx", "is_overlapped"]
RESULTS_BASE = "results/binary/"

CHEST_FEATURES_UNNORM = "data/processed/wesad/features/chest_features_unnormalized.pkl"
WRIST_FEATURES_UNNORM = "data/processed/wesad/features/wrist_features_unnormalized.pkl"

diag_results = {}

for dev_name, feat_path in [
    ("chest", CHEST_FEATURES_UNNORM),
    ("wrist", WRIST_FEATURES_UNNORM),
]:

    if not os.path.exists(feat_path):
        print(f"[{dev_name}] Feature file not found — skipping: {feat_path}")
        continue

    df_full = pd.read_pickle(feat_path)

    # ── Filter: keep independent windows only ──────────────────────
    df_indep = df_full[~df_full["is_overlapped"]].reset_index(drop=True)

    n_total = len(df_full)
    n_indep = len(df_indep)
    n_feat = len([c for c in df_indep.columns if c not in META_COLS])

    print(f"\n{'='*65}")
    print(f"  {dev_name.upper()} — UNNORMALIZED + INDEPENDENT WINDOWS ONLY")
    print(f"{'='*65}")
    print(f"  Total windows in file : {n_total:,}")
    print(
        f"  Independent only      : {n_indep:,}  ({n_indep/n_total*100:.1f}% of total)"
    )
    print(f"  Features              : {n_feat}")
    print(
        f"  Label distribution    : {df_indep['label'].value_counts().sort_index().to_dict()}"
    )

    output_dir = os.path.join(
        RESULTS_BASE, "diagnostic_ablation", dev_name, "unnorm_indep_only_BINARY"
    )

    classifier = WESADClassifier(
        df_features=df_indep,  # <-- independent windows only
        label_map=LABEL_MAP_BINARY,
        feature_selection_method="tree",
        output_dir=output_dir,
    )
    results = classifier.run_losocv()
    diag_results[dev_name] = results

    del df_full, df_indep

# ── Summary comparison ─────────────────────────────────────────────
print("\n\n" + "=" * 65)
print("  DIAGNOSTIC ABLATION — SUMMARY")
print("=" * 65)
print(f"  {'Device':<8} {'Accuracy':>10} {'F1 Weighted':>13} {'F1 Macro':>10}")
print(f"  {'-'*45}")
for dev_name, res in diag_results.items():
    agg = res["aggregate"]
    print(
        f"  {dev_name:<8} "
        f"{agg['accuracy_mean']:>9.4f}±{agg['accuracy_std']:.3f}  "
        f"{agg['f1_weighted_mean']:>10.4f}±{agg['f1_weighted_std']:.3f}  "
        f"{agg['f1_macro_mean']:>9.4f}±{agg['f1_macro_std']:.3f}"
    )
print()

In [ ]:
import os

import pandas as pd

from src.training.ml_classifier import LABEL_MAP_3CLASS, WESADClassifier

META_COLS = ["sid", "label", "window_idx", "is_overlapped"]
RESULTS_BASE = "results/binary/"

CHEST_FEATURES_UNNORM = "data/processed/wesad/features/chest_features_unnormalized.pkl"
WRIST_FEATURES_UNNORM = "data/processed/wesad/features/wrist_features_unnormalized.pkl"

diag_results = {}

for dev_name, feat_path in [
    ("chest", CHEST_FEATURES_UNNORM),
    ("wrist", WRIST_FEATURES_UNNORM),
]:

    if not os.path.exists(feat_path):
        print(f"[{dev_name}] Feature file not found — skipping: {feat_path}")
        continue

    df_full = pd.read_pickle(feat_path)

    # ── Filter: keep independent windows only ──────────────────────
    df_indep = df_full[~df_full["is_overlapped"]].reset_index(drop=True)

    n_total = len(df_full)
    n_indep = len(df_indep)
    n_feat = len([c for c in df_indep.columns if c not in META_COLS])

    print(f"\n{'='*65}")
    print(f"  {dev_name.upper()} — UNNORMALIZED + INDEPENDENT WINDOWS ONLY")
    print(f"{'='*65}")
    print(f"  Total windows in file : {n_total:,}")
    print(
        f"  Independent only      : {n_indep:,}  ({n_indep/n_total*100:.1f}% of total)"
    )
    print(f"  Features              : {n_feat}")
    print(
        f"  Label distribution    : {df_indep['label'].value_counts().sort_index().to_dict()}"
    )

    output_dir = os.path.join(
        RESULTS_BASE, "diagnostic_ablation", dev_name, "unnorm_indep_only_3CLASS"
    )

    classifier = WESADClassifier(
        df_features=df_indep,  # <-- independent windows only
        label_map=LABEL_MAP_3CLASS,
        feature_selection_method="tree",
        output_dir=output_dir,
    )
    results = classifier.run_losocv()
    diag_results[dev_name] = results

    del df_full, df_indep

# ── Summary comparison ─────────────────────────────────────────────
print("\n\n" + "=" * 65)
print("  DIAGNOSTIC ABLATION — SUMMARY")
print("=" * 65)
print(f"  {'Device':<8} {'Accuracy':>10} {'F1 Weighted':>13} {'F1 Macro':>10}")
print(f"  {'-'*45}")
for dev_name, res in diag_results.items():
    agg = res["aggregate"]
    print(
        f"  {dev_name:<8} "
        f"{agg['accuracy_mean']:>9.4f}±{agg['accuracy_std']:.3f}  "
        f"{agg['f1_weighted_mean']:>10.4f}±{agg['f1_weighted_std']:.3f}  "
        f"{agg['f1_macro_mean']:>9.4f}±{agg['f1_macro_std']:.3f}"
    )
print()

## 4. Compare FS Methods & Lock Best

In [ ]:
# reload from disk blockl

chest_fs_results = {}
wrist_fs_results = {}
for method in ["none", "tree", "kbest", "pca"]:
    chest_path = os.path.join(
        RESULTS_BASE, "chest_fs_ablation", f"fs_{method}", "losocv_results.pkl"
    )
    wrist_path = os.path.join(
        RESULTS_BASE, "wrist_fs_ablation", f"fs_{method}", "losocv_results.pkl"
    )
    if os.path.exists(chest_path):
        with open(chest_path, "rb") as f:
            chest_fs_results[method] = pickle.load(f)
    if os.path.exists(wrist_path):
        with open(wrist_path, "rb") as f:
            wrist_fs_results[method] = pickle.load(f)

In [ ]:
# comparison table
def build_comparison_df(results_dict, device_name):
    rows = []
    for method, res in results_dict.items():
        agg = res["aggregate"]
        rows.append(
            {
                "Device": device_name,
                "FS Method": method,
                "Accuracy": f"{agg['accuracy_mean']:.4f} +/- {agg['accuracy_std']:.4f}",
                "F1 Weighted": f"{agg['f1_weighted_mean']:.4f} +/- {agg['f1_weighted_std']:.4f}",
                "F1 Macro": f"{agg['f1_macro_mean']:.4f} +/- {agg['f1_macro_std']:.4f}",
                "f1_macro_mean": agg["f1_macro_mean"],  # for sorting
                "f1_weighted_mean": agg["f1_weighted_mean"],
            }
        )
    return pd.DataFrame(rows)


df_comp_chest = build_comparison_df(chest_fs_results, "Chest")
df_comp_wrist = build_comparison_df(wrist_fs_results, "Wrist")
df_comp = pd.concat([df_comp_chest, df_comp_wrist], ignore_index=True)

# Display
display_cols = ["Device", "FS Method", "Accuracy", "F1 Weighted", "F1 Macro"]
print("FEATURE SELECTION ABLATION — COMPARISON")
print("=" * 80)
print(df_comp[display_cols].to_string(index=False))
print()

In [ ]:
# Bar chart comparison
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
methods = ["none", "tree", "kbest", "pca"]
method_colors = {
    "none": "#9E9E9E",
    "tree": "#4CAF50",
    "kbest": "#2196F3",
    "pca": "#FF9800",
}

for ax, (device, results_dict) in zip(
    axes, [("Chest", chest_fs_results), ("Wrist", wrist_fs_results)]
):
    f1_means = []
    f1_stds = []
    labels = []
    colors = []
    for m in methods:
        if m in results_dict:
            agg = results_dict[m]["aggregate"]
            f1_means.append(agg["f1_macro_mean"])
            f1_stds.append(agg["f1_macro_std"])
            labels.append(m)
            colors.append(method_colors[m])

    x = np.arange(len(labels))
    bars = ax.bar(
        x, f1_means, yerr=f1_stds, capsize=5, color=colors, edgecolor="white", width=0.6
    )
    ax.set_xticks(x)
    ax.set_xticklabels(labels)
    ax.set_ylabel("F1 Macro (mean +/- std)")
    ax.set_title(f"{device} — FS Method Comparison")
    ax.set_ylim(0, 1.05)
    ax.grid(True, alpha=0.3, axis="y")

    # Annotate bars
    for bar, mean, std in zip(bars, f1_means, f1_stds):
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + std + 0.02,
            f"{mean:.3f}",
            ha="center",
            fontsize=9,
            fontweight="bold",
        )

plt.suptitle(
    "Feature Selection Ablation — F1 Macro (Binary, LOSOCV)", fontsize=14, y=1.02
)
plt.tight_layout()
plt.show()

In [ ]:
# Select best FS method per device
best_fs_chest = df_comp_chest.sort_values("f1_macro_mean", ascending=False).iloc[0][
    "FS Method"
]
best_fs_wrist = df_comp_wrist.sort_values("f1_macro_mean", ascending=False).iloc[0][
    "FS Method"
]

LOCKED_FS = {
    "chest": best_fs_chest,
    "wrist": best_fs_wrist,
}

print(f"Best FS method (Chest, by F1 macro): {best_fs_chest}")
print(f"Best FS method (Wrist, by F1 macro): {best_fs_wrist}")
print()
if best_fs_chest == best_fs_wrist:
    print(f"Both devices agree: {best_fs_chest}")
else:
    print("Devices disagree : each will use its own best method.")
    print(
        f"This is expected: chest ({len([c for c in df_chest.columns if c not in META_COLS])} features, "
        f"6 modalities) and wrist ({len([c for c in df_wrist.columns if c not in META_COLS])} features, "
        f"4 modalities) have different feature geometries."
    )

## 5.Windowing Ablation: Feature Extraction (Full Dataset)

In [ ]:
from src.ml_pipeline.feature_extraction import run_feature_pipeline_streaming
from src.windowing.wesad_windowing import WESADWindowing

SLIDE_VALUES = [0.25, 0.5, 5.0, 20.0, 60.0]
WINDOW_SIZE_SEC = 60.0
ACC_SUBWINDOW_SEC = 5.0

WINDOW_ABLATION_FEATURES_DIR = "data/processed/wesad/features/window_ablation_full"
os.makedirs(WINDOW_ABLATION_FEATURES_DIR, exist_ok=True)

devices = [
    {
        "name": "chest",
        "preprocessed": CHEST_PREPROCESSED,
        "config": CHEST_CONFIG,
        "existing_5s": CHEST_FEATURES,
        "wrist": False,
    },
    {
        "name": "wrist",
        "preprocessed": WRIST_PREPROCESSED,
        "config": WRIST_CONFIG,
        "existing_5s": WRIST_FEATURES,
        "wrist": True,
    },
]

print("Windowing ablation feature extraction config:")
print(f"  Slides: {SLIDE_VALUES}")
print(f"  Output: {WINDOW_ABLATION_FEATURES_DIR}")
print("  Devices: chest, wrist")

In [ ]:
# Extract features for each slide × device (skip if exists)
for dev in devices:
    for slide in SLIDE_VALUES:
        output_path = os.path.join(
            WINDOW_ABLATION_FEATURES_DIR,
            f"{dev['name']}_features_w60s_s{slide}s.pkl",
        )

        # For 5.0s slide, copy from existing features (already extracted)
        if slide == 5.0:
            if not os.path.exists(output_path):
                import shutil

                shutil.copy2(dev["existing_5s"], output_path)
                print(f"  {dev['name']} slide 5.0s: Copied from {dev['existing_5s']}")
            else:
                df_check = pd.read_pickle(output_path)
                print(f"  {dev['name']} slide 5.0s: EXISTS ({len(df_check):,} windows)")
                del df_check
            continue

        # Skip if already extracted
        if os.path.exists(output_path):
            df_check = pd.read_pickle(output_path)
            print(f"  {dev['name']} slide {slide}s: EXISTS ({len(df_check):,} windows)")
            del df_check
            continue

        print(f"\n{'='*70}")
        print(f"  {dev['name'].upper()} — slide {slide}s — full dataset extraction")
        print(f"{'='*70}")

        # Step 1: Windowing
        windower = WESADWindowing(
            data_path=dev["preprocessed"],
            config_path=dev["config"],
            output_dir=WINDOWED_OUTPUT_DIR,
            window_size_sec=WINDOW_SIZE_SEC,
            slide_sec=slide,
            acc_subwindow_sec=ACC_SUBWINDOW_SEC,
            wrist=dev["wrist"],
        )
        windower.load_data()
        windows = windower.segment()
        print(f"  Windowed: {len(windows):,} windows")

        # Step 2: Feature extraction
        df_feat = run_feature_pipeline_streaming(
            data_path=dev["preprocessed"],
            windows=windows,
            config_path=dev["config"],
            output_path=output_path,
            wrist=dev["wrist"],
            baseline_label=1.0,
            normalize=False,
            impute=False,
        )

        print(f"  Saved: {output_path} ({len(df_feat):,} windows)")

        # Free memory
        del windower, windows, df_feat
        gc.collect()

print("\nFeature extraction for windowing ablation complete.")

## 6. Windowing Ablation: LOSOCV with Locked FS Method

Runs full LOSOCV at each slide value using the locked FS method from the section above to answer the question on whether the window slide size affects model performance 

In [ ]:
from src.training.ml_classifier import LABEL_MAP_BINARY, WESADClassifier

# Run LOSOCV at each slide value for both devices
window_ablation_results = {}

for dev_name in ["chest", "wrist"]:
    window_ablation_results[dev_name] = {}
    fs_method = LOCKED_FS[dev_name]
    print(f"\nDevice: {dev_name.upper()}, FS method: {fs_method}")

    for slide in SLIDE_VALUES:
        feature_path = os.path.join(
            WINDOW_ABLATION_FEATURES_DIR,
            f"{dev_name}_features_w60s_s{slide}s.pkl",
        )

        if not os.path.exists(feature_path):
            print(f"  {dev_name} slide {slide}s: feature file NOT FOUND — skipping")
            continue

        print(f"\n{'='*70}")
        print(f"  {dev_name.upper()} — slide {slide}s — LOSOCV with FS={fs_method}")
        print(f"{'='*70}")

        df_feat = pd.read_pickle(feature_path)
        n_feats = len([c for c in df_feat.columns if c not in META_COLS])
        print(f"  Loaded: {len(df_feat):,} windows, {n_feats} features")

        output_dir = os.path.join(
            RESULTS_BASE, "window_ablation", dev_name, f"slide_{slide}s"
        )

        classifier = WESADClassifier(
            df_features=df_feat,
            label_map=LABEL_MAP_BINARY,
            feature_selection_method=fs_method,
            output_dir=output_dir,
        )
        results = classifier.run_losocv()
        window_ablation_results[dev_name][slide] = results

        del df_feat, classifier
        gc.collect()

print("\nWindowing ablation LOSOCV complete.")

## 7. Windowing Ablation: Results & Visualisation

In [ ]:
# Uncomment if running from a fresh kernel:

window_ablation_results = {}
for dev_name in ["chest", "wrist"]:
    window_ablation_results[dev_name] = {}
    for slide in SLIDE_VALUES:
        path = os.path.join(
            RESULTS_BASE,
            "window_ablation",
            dev_name,
            f"slide_{slide}s",
            "losocv_results.pkl",
        )
        if os.path.exists(path):
            with open(path, "rb") as f:
                window_ablation_results[dev_name][slide] = pickle.load(f)

In [ ]:
# Build comparison table:
rows = []
for dev_name, dev_results in window_ablation_results.items():
    for slide, res in sorted(dev_results.items()):
        agg = res["aggregate"]
        rows.append({
            "Device": dev_name.capitalize(),
            "FS Method": LOCKED_FS[dev_name],
            "Slide (s)": slide,
            "Windows": len(res["all_y_true"]),
            "Accuracy": agg["accuracy_mean"],
            "Acc STD": agg["accuracy_std"],
            "F1 Weighted": agg["f1_weighted_mean"],
            "F1w STD": agg["f1_weighted_std"],
            "F1 Macro": agg["f1_macro_mean"],
            "F1m STD": agg["f1_macro_std"],
        })

df_window_comp = pd.DataFrame(rows)

print("WINDOWING ABLATION — COMPARISON")
print("
")
for dev in ["Chest", "Wrist"]:
    subset = df_window_comp[df_window_comp["Device"] == dev]
    if subset.empty:
        continue
    fs = LOCKED_FS[dev.lower()]
    print(f"  {dev} (FS={fs}):")
    print(f"  {"Slide":<8} {"Windows":<10} {"Accuracy":<20} {"F1 Weighted":<20} {"F1 Macro":<20}")
    print(f"  {"-"*78}")
    for _, r in subset.iterrows():
        print(f"  {r['Slide (s)']:<8.2f} {r['Windows']:<10} "
              f"{r['Accuracy']:.4f} +/- {r['Acc STD']:.4f}   "
              f"{r['F1 Weighted']:.4f} +/- {r['F1w STD']:.4f}   "
              f"{r['F1 Macro']:.4f} +/- {r['F1m STD']:.4f}")
    print()

In [ ]:
# Visualisation
fig, axes = plt.subplots(1, 3, figsize=(20, 6))
dev_colors = {"Chest": "#2196F3", "Wrist": "#FF9800"}
metrics = [("Accuracy", "Acc STD"), ("F1 Weighted", "F1w STD"), ("F1 Macro", "F1m STD")]

for ax, (metric, std_col) in zip(axes, metrics):
    for dev in ["Chest", "Wrist"]:
        subset = df_window_comp[df_window_comp["Device"] == dev].sort_values(
            "Slide (s)"
        )
        if subset.empty:
            continue
        ax.errorbar(
            subset["Slide (s)"],
            subset[metric],
            yerr=subset[std_col],
            fmt="o-",
            color=dev_colors[dev],
            linewidth=2,
            markersize=8,
            capsize=5,
            label=f"{dev} (FS={LOCKED_FS[dev.lower()]})",
        )
    ax.set_xlabel("Slide size (s)")
    ax.set_ylabel(metric)
    ax.set_title(f"{metric} vs Slide Size")
    ax.set_xscale("log")
    import matplotlib.ticker as ticker

    ax.xaxis.set_major_formatter(ticker.ScalarFormatter())
    ax.set_xticks(SLIDE_VALUES)
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)
    ax.set_ylim(0, 1.05)

plt.suptitle("Windowing Ablation: Binary LOSOCV", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Per-subject variability at the primary slide (5s)
fig, axes = plt.subplots(1, 2, figsize=(18, 6))

for ax, (dev_name, dev_results) in zip(axes, window_ablation_results.items()):
    if 5.0 not in dev_results:
        ax.text(
            0.5,
            0.5,
            f"No 5s results for {dev_name}",
            ha="center",
            va="center",
            transform=ax.transAxes,
        )
        continue

    fold_results = dev_results[5.0]["fold_results"]
    subjects = [f["subject"] for f in fold_results]
    f1_macros = [f["f1_macro"] for f in fold_results]
    accuracies = [f["accuracy"] for f in fold_results]

    x = np.arange(len(subjects))
    width = 0.35
    ax.bar(
        x - width / 2, accuracies, width, color="#2196F3", alpha=0.7, label="Accuracy"
    )
    ax.bar(
        x + width / 2, f1_macros, width, color="#FF9800", alpha=0.7, label="F1 Macro"
    )

    ax.set_xticks(x)
    ax.set_xticklabels([f"S{s}" for s in subjects], fontsize=8, rotation=45)
    ax.set_ylabel("Score")
    ax.set_title(f"{dev_name.capitalize()} — Per-Subject Performance (5s slide)")
    ax.set_ylim(0, 1.05)
    ax.legend()
    ax.grid(True, alpha=0.3, axis="y")

    # Highlight known outliers (S2, S11)
    for i, s in enumerate(subjects):
        if s in [2, 11]:
            ax.annotate(
                "*",
                (i, max(accuracies[i], f1_macros[i]) + 0.02),
                ha="center",
                fontsize=14,
                color="red",
                fontweight="bold",
            )

plt.suptitle(
    "Per-Subject LOSOCV Performance (known outliers S2, S11 marked *)",
    fontsize=13,
    y=1.02,
)
plt.tight_layout()
plt.show()

All results are saved as pickle files for evaluation

**Saved files:**
- `results/binary/chest_fs_ablation/fs_{method}/losocv_results.pkl` — per FS method
- `results/binary/wrist_fs_ablation/fs_{method}/losocv_results.pkl` — per FS method
- `results/binary/window_ablation/{device}/slide_{X}s/losocv_results.pkl` — per slide
- Per-fold trained models: `model_S{sid}.pkl` in each results directory

In [ ]:
# Final summary print
print("PHASE 2 COMPLETE — SUMMARY")

print(f"
Locked FS methods: {LOCKED_FS}")
print()

for dev_name in ["chest", "wrist"]:
    if dev_name not in window_ablation_results or not window_ablation_results[dev_name]:
        continue
    print(f"{dev_name.upper()} (FS={LOCKED_FS[dev_name]}):")

    # Best slide
    best_slide = max(
        window_ablation_results[dev_name].items(),
        key=lambda x: x[1]["aggregate"]["f1_macro_mean"],
    )
    agg = best_slide[1]["aggregate"]
    print(f"  Best slide: {best_slide[0]}s")
    print(f"    Accuracy:    {agg["accuracy_mean"]:.4f} +/- {agg["accuracy_std"]:.4f}")
    print(f"    F1 Weighted: {agg["f1_weighted_mean"]:.4f} +/- {agg["f1_weighted_std"]:.4f}")
    print(f"    F1 Macro:    {agg["f1_macro_mean"]:.4f} +/- {agg["f1_macro_std"]:.4f}")
    print()

print("
All results saved to disk.")

# Deep Learning Pipeline

In [ ]:
# Verify Dataset Structure & Sampling Rates
import sys

sys.path.append(".")  # ensure project root is on path

from src.data.wesad_metadata_extractor import (
    determine_chest_sampling_rate,
    determine_wrist_sampling_rates,
    extract_wesad_metadata,
    get_column_names_from_data,
)

# Point at one subject file to verify
EXAMPLE_SUBJECT = "data/raw/wesad/WESAD/S2/S2.pkl"

print("STEP 0: Dataset Structure Verification")

# Metadata
metadata = extract_wesad_metadata(EXAMPLE_SUBJECT)
print(f"\n[1] Label length:    {metadata['label_length']:,} samples")
print(f"    Chest signals:   {list(metadata['chest_signals'].keys())}")
print(f"    Wrist signals:   {list(metadata['wrist_signals'].keys())}")

# Chest sampling rate
chest_rate = determine_chest_sampling_rate(EXAMPLE_SUBJECT)
print(f"\n[2] Chest sampling rate: {chest_rate:.0f} Hz")

# Wrist sampling rates
wrist_rates = determine_wrist_sampling_rates(EXAMPLE_SUBJECT, chest_rate)
print("\n[3] Wrist sampling rates:")
for signal, rate in wrist_rates.items():
    print(f"    {signal:<8} {rate} Hz")

# Column names
columns = get_column_names_from_data(EXAMPLE_SUBJECT)
print(f"\n[4] Chest columns: {columns['chest']}")
print(f"    Wrist columns:  {columns['wrist']}")

# Shape sanity check
print("\n[5] Signal shapes (chest):")
for sig, info in metadata["chest_signals"].items():
    print(f"    {sig:<8} {info['shape']}")
print("\n    Signal shapes (wrist):")
for sig, info in metadata["wrist_signals"].items():
    print(f"    {sig:<8} {info['shape']}")

## Filter & Downsample to DL Target Rates

In [ ]:
import sys

sys.path.append(".")

from src.data.wesad_dl_preprocess import (
    DL_TARGET_HZ,
    NATIVE_HZ,
    prepare_all,
    process_subject,
)

DATASET_PATH = "data/raw/wesad/WESAD/"
OUTPUT_PATH = "data/processed/wesad/dl_preprocessed/"

print("STEP 1: DL Signal Preparation")
print("\n")
print("\nDownsampling targets:")
for sig, hz in DL_TARGET_HZ.items():
    native = NATIVE_HZ.get(sig, "?")
    tag = "(native)" if native == hz else f"{native} Hz → {hz} Hz"
    print(f"  {sig:<10} {tag}")

print("\n")

# Run one subject first as a sanity check
print("\nDry run: processing S2 only...")
process_subject(2, DATASET_PATH, OUTPUT_PATH)

# Verify output files exist and shapes look right
import os

import numpy as np

subj_dir = os.path.join(OUTPUT_PATH, "S2")
expected = {
    "ecg.npy": (None,),  # 1D, @ 256 Hz
    "emg.npy": (None,),  # 1D, @ 64 Hz
    "c_eda.npy": (None,),  # 1D, @ 8 Hz
    "resp.npy": (None,),  # 1D, @ 32 Hz
    "c_acc.npy": (None, 3),  # 2D, @ 32 Hz
    "bvp.npy": (None,),  # 1D, @ 32 Hz
    "w_eda.npy": (None,),  # 1D, @ 4 Hz
    "w_acc.npy": (None, 3),  # 2D, @ 32 Hz
    "w_temp.npy": (None,),  # 1D, @ 4 Hz
}

print("\nOutput verification (S2):")
all_ok = True
for fname, expected_shape in expected.items():
    fpath = os.path.join(subj_dir, fname)
    if not os.path.exists(fpath):
        print(f"   MISSING: {fname}")
        all_ok = False
        continue
    arr = np.load(fpath)
    label_path = fpath.replace(fname, f"label_{fname}")
    label = np.load(label_path) if os.path.exists(label_path) else None
    label_info = f"label: {label.shape}" if label is not None else "label: MISSING"
    print(f"   {fname:<15} shape: {arr.shape}  {label_info}")

print(
    "\n"
    + (
        " All outputs present."
        if all_ok
        else " Some outputs missing - check errors above."
    )
)

# S2 looks good, run all subjects
prepare_all(DATASET_PATH, OUTPUT_PATH)

## Windowing

- 16s default + 10s/30s ablation windows, as per the literature. This produces per-subject windowed .npz files for all three window lengths.  High-overlap (0.25s stride) for training,
non-overlapping for validation and test is handled by the Dataset.

In [ ]:
import sys

sys.path.append(".")

from src.windowing.wesad_dl_windowing import run_window_ablation

PREPARED_DIR = "data/processed/wesad/dl_preprocessed/"
WINDOWED_DIR = "data/processed/wesad/dl_windowed/"

run_window_ablation(
    prepared_dir=PREPARED_DIR,
    output_dir=WINDOWED_DIR,
    window_lengths=[10.0, 16.0, 30.0],
    slide_sec=0.25,
)

In [ ]:
# Verify windowed output
import os

import numpy as np

WINDOWED_DIR = "data/processed/wesad/dl_windowed/"

for wl in ["10s", "16s", "30s"]:
    wl_dir = os.path.join(WINDOWED_DIR, wl, "S2")
    print(f"\n{'='*50}")
    print(f"  Window: {wl}")
    print(f"{'='*50}")
    for device in ["chest", "wrist"]:
        npz_path = os.path.join(wl_dir, device, "windows.npz")
        meta_path = os.path.join(wl_dir, device, "metadata.csv")
        if os.path.exists(npz_path):
            npz = np.load(npz_path)
            import csv

            with open(meta_path) as f:
                n_windows = sum(1 for _ in csv.reader(f)) - 1  # minus header
            print(f"  {device}: {n_windows} windows")
            for key in sorted(npz.keys()):
                print(f"    {key:<12} shape: {npz[key].shape}")
        else:
            print(f"  {device}: MISSING")

## TRIAL RUN: Single fold, wrist, 16s, baseline config

In [ ]:
# Uses S2 as test subject.

import sys

sys.path.append(".")

import torch

from src.data.wesad_dl_dataset import LABEL_MAP_BINARY
from src.models.wesad_cnnlstm_model import build_wrist_model, print_model_summary
from src.training.wesad_cnnlstm_train import TrainConfig, train_one_fold

# Build and inspect model architecture before training
print("=" * 60)
print("  MODEL ARCHITECTURE (wrist, baseline config)")
print("=" * 60)
model = build_wrist_model(n_classes=3)
print_model_summary(model, "wrist")
del model

# Trial config, same as baseline but just one fold
trial_config = TrainConfig(
    device_type="wrist",
    n_classes=3,
    label_map=LABEL_MAP_BINARY,
    window_sec=16.0,
    windowed_dir="data/processed/wesad/dl_windowed/",
    output_dir="results/dl/trial_wrist/",
    # All other params at baseline defaults:
    # lstm_hidden=32, lstm_layers=2, n_conv_layers=1,
    # optimizer="adamw", lr=1e-3, weight_decay=1e-4,
    # batch_size=64, cnn_dropout=0.5, fc_dropout=0.5,
    # noise_std=0.05, scale_range=(0.9, 1.1)
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

# Run single fold with S2 as test
result = train_one_fold(test_subject=2, config=trial_config, device=device)

print("\n" + "=" * 60)
print("  TRIAL FOLD RESULT")
print("=" * 60)
for k, v in result.items():
    if k != "confusion_matrix":
        print(f"  {k}: {v}")
    else:
        print(f"  confusion_matrix:\n{np.array(v)}")

# Verify checkpoint exists
import os

ckpt = result.get("best_model_path", "")
print(f"\n  Checkpoint exists: {os.path.exists(ckpt)}")
print(
    f"  Loss curves exist: {os.path.exists(os.path.join('results/dl/trial_wrist/fold_S2', 'loss_curves.json'))}"
)

## MAIN TRAINING: Wrist model, 16s, full 15-fold LOSO

In [ ]:
import sys

sys.path.append(".")

from src.data.wesad_dl_dataset import LABEL_MAP_BINARY
from src.training.wesad_cnnlstm_train import TrainConfig, run_loso

wrist_config = TrainConfig(
    device_type="wrist",
    n_classes=2,
    label_map=LABEL_MAP_BINARY,
    window_sec=16.0,
    windowed_dir="data/processed/wesad/dl_windowed/",
    output_dir="results/dl/wrist_baseline/",
)

wrist_results = run_loso(wrist_config)

## Evaluate wrist baseline

In [ ]:
from src.evaluation.wesad_cnnlstm_evaluate import evaluate_from_results

evaluate_from_results(
    results_path="results/dl/wrist_baseline/loso_results.json",
    output_dir="results/dl/wrist_baseline/",
)

## CHEST MODEL: Upper bound baseline

In [ ]:
# The chest model establishes the DL performance ceiling under
# ideal clinical conditions.
from src.data.wesad_dl_dataset import LABEL_MAP_BINARY
from src.training.wesad_cnnlstm_train import TrainConfig, run_loso

chest_config = TrainConfig(
    device_type="chest",
    n_classes=2,
    label_map=LABEL_MAP_BINARY,
    window_sec=16.0,
    windowed_dir="data/processed/wesad/dl_windowed/",
    output_dir="results/dl/chest_baseline/",
)

chest_results = run_loso(chest_config)

from src.evaluation.wesad_cnnlstm_evaluate import evaluate_from_results

evaluate_from_results("results/dl/chest_baseline/loso_results.json")

## Ablation with different number of CNN layers 

In [ ]:
# Tests: 1 vs 2 CNN layers per branch
# (3 layers excluded for wrist: w_eda bottleneck at 10s)
#
# Citation: Ordóñez & Roggen (2016): "increasing the number of
# convolutional layers tends to increase performance... performance
# changes may plateau."

from src.data.wesad_dl_dataset import LABEL_MAP_BINARY
from src.training.wesad_cnnlstm_train import TrainConfig, run_loso

for n_layers in [1, 2]:
    print(f"\n{'#'*60}")
    print(f"  ABLATION: CNN layers = {n_layers}")
    print(f"{'#'*60}")

    config = TrainConfig(
        device_type="wrist",
        n_classes=2,
        label_map=LABEL_MAP_BINARY,
        n_conv_layers=n_layers,
        window_sec=16.0,
        windowed_dir="data/processed/wesad/dl_windowed/",
        output_dir=f"results/dl/ablation_cnn_layers/layers_{n_layers}/",
    )
    run_loso(config)

## ABLATION: Optimizer

In [ ]:
# Tests: AdamW (default) vs RMSProp
#
# Citation: Ordóñez & Roggen (2016): "RMSProp consistently offered
# the best results with widest tolerance to learning rate."

from src.data.wesad_dl_dataset import LABEL_MAP_BINARY
from src.training.wesad_cnnlstm_train import TrainConfig, run_loso

for opt in ["adamw", "rmsprop"]:
    print("\n")
    print(f"  ABLATION: Optimizer = {opt}")
    print("\n")

    config = TrainConfig(
        device_type="wrist",
        n_classes=2,
        label_map=LABEL_MAP_BINARY,
        optimizer=opt,
        window_sec=16.0,
        windowed_dir="data/processed/wesad/dl_windowed/",
        output_dir=f"results/dl/ablation_optimizer/{opt}/",
    )
    run_loso(config)

## ABLATION: Dropout

In [ ]:
# Citation: Ordóñez & Roggen (2016): "p=0.5, tested p>0.5 with
# no difference in performance."


from src.data.wesad_dl_dataset import LABEL_MAP_BINARY
from src.training.wesad_cnnlstm_train import TrainConfig, run_loso

for dropout in [0.3, 0.5]:
    print(f"\n{'#'*60}")
    print(f"  ABLATION: Dropout = {dropout}")
    print(f"{'#'*60}")

    config = TrainConfig(
        device_type="wrist",
        n_classes=2,
        label_map=LABEL_MAP_BINARY,
        cnn_dropout=dropout,
        fc_dropout=dropout,
        window_sec=16.0,
        windowed_dir="data/processed/wesad/dl_windowed/",
        output_dir=f"results/dl/ablation_dropout/p_{dropout}/",
    )
    run_loso(config)

## ABLATION 4: Batch size

In [ ]:
# Citation: Ordóñez & Roggen (2016): mini-batches of size 100.
# Prajod & André (2022): batch size 128.
# We experiment to test the hypothesis thatmaller batches = noisier gradients = implicit regularisation.

from src.data.wesad_dl_dataset import LABEL_MAP_BINARY
from src.training.wesad_cnnlstm_train import TrainConfig, run_loso

for bs in [32, 64, 100]:
    print(f"\n{'#'*60}")
    print(f"  ABLATION: Batch size = {bs}")
    print(f"{'#'*60}")

    config = TrainConfig(
        device_type="wrist",
        n_classes=2,
        label_map=LABEL_MAP_BINARY,
        batch_size=bs,
        window_sec=16.0,
        windowed_dir="data/processed/wesad/dl_windowed/",
        output_dir=f"results/dl/ablation_batch_size/bs_{bs}/",
    )
    run_loso(config)

## ABLATION: CNN filters

In [ ]:
# Tests model capacity vs overfitting risk.

# NOTE: n_filters_override replaces ALL branch filter counts.
# Baseline uses mixed (50 for BVP, 32 for EDA/ACC).
# None = use per-branch defaults from the config.

from src.data.wesad_dl_dataset import LABEL_MAP_BINARY
from src.training.wesad_cnnlstm_train import TrainConfig, run_loso

for n_filters in [None, 32, 64, 128]:
    label = "default" if n_filters is None else str(n_filters)
    print(f"\n{'#'*60}")
    print(f"  ABLATION: CNN filters = {label}")
    print(f"{'#'*60}")

    config = TrainConfig(
        device_type="wrist",
        n_classes=2,
        label_map=LABEL_MAP_BINARY,
        n_filters_override=n_filters,
        window_sec=16.0,
        windowed_dir="data/processed/wesad/dl_windowed/",
        output_dir=f"results/dl/ablation_filters/f_{label}/",
    )
    run_loso(config)

## ABLATION 6: Window length (wrist only)

In [ ]:
# Tests temporal context vs data quantity trade-off.
# All use target_time=4 (safe universal value for wrist at 10s)
# to isolate window length as the solE variable.
#
# Citation: Moser et al. (2024) 16s, Prajod & André (2022) - 10s

from src.data.wesad_dl_dataset import LABEL_MAP_BINARY
from src.training.wesad_cnnlstm_train import TrainConfig, run_loso

for wl in [10.0, 16.0, 30.0]:
    print("\n")
    print(f"  ABLATION: Window length = {wl}s")
    print("\n")

    config = TrainConfig(
        device_type="wrist",
        n_classes=2,
        label_map=LABEL_MAP_BINARY,
        window_sec=wl,
        target_time=4,  # fixed across all window lengths for fair comparison
        windowed_dir="data/processed/wesad/dl_windowed/",
        output_dir=f"results/dl/ablation_window/{wl:.0f}s/",
    )
    run_loso(config)

## Collect all ablation results into a summary table

In [ ]:
import json
import os

import pandas as pd

results_root = "results/dl/"
rows = []

for dirpath, dirnames, filenames in os.walk(results_root):
    if "loso_results.json" in filenames:
        with open(os.path.join(dirpath, "loso_results.json")) as f:
            data = json.load(f)
        summary = data.get("summary", {})
        config = data.get("config", {})

        rows.append(
            {
                "Experiment": os.path.relpath(dirpath, results_root),
                "Device": config.get("device_type", "?"),
                "Window": config.get("window_sec", "?"),
                "Optimizer": config.get("optimizer", "?"),
                "CNN Layers": config.get("n_conv_layers", "?"),
                "Filters": config.get("n_filters_override", "default"),
                "Dropout": config.get("cnn_dropout", "?"),
                "Batch": config.get("batch_size", "?"),
                "Noise": config.get("noise_std", "?"),
                "F1 Macro": f"{summary.get('f1_macro_mean', 0):.4f} ± {summary.get('f1_macro_std', 0):.4f}",
                "F1 Weighted": f"{summary.get('f1_weighted_mean', 0):.4f} ± {summary.get('f1_weighted_std', 0):.4f}",
                "Accuracy": f"{summary.get('accuracy_mean', 0):.4f} ± {summary.get('accuracy_std', 0):.4f}",
            }
        )

df_results = pd.DataFrame(rows)
df_results = df_results.sort_values("Experiment")

print("  ALL EXPERIMENT RESULTS\n")
print(df_results.to_string(index=False))

# Save to CSV for dissertation tables
csv_path = os.path.join(results_root, "all_results_summary.csv")
df_results.to_csv(csv_path, index=False)
print(f"\nSaved → {csv_path}")

## CNN-LSTM Evaluation

This section evaluates the centralized multi-branch CNN-LSTM trained with Leave-One-Subject-Out Cross-Validation (LOSO-CV).

Two evaluation modes:
- Mode 1 From JSON (fast): Loads `loso_results.json` saved during training.
- Mode 2From Checkpoints (full): Re-runs inference from saved `.pt` model files. Enables per-window prediction analysis.

Primary metric: Macro-F1 (handles class imbalance per Prajod & André, 2022)  
Key output: Inter-subject variability, high std across folds motivates the FL approach in nect phase.

In [ ]:
import json
import os

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

from src.evaluation.wesad_cnnlstm_evaluate import (
    evaluate_from_checkpoints,
    evaluate_from_results,
)

RESULTS_DIR = (
    "results/dl/wrist_baseline"  # folder from training run TODO CHECK WHERE I SENT THEM
)
RESULTS_JSON = os.path.join(RESULTS_DIR, "loso_results.json")
WINDOWED_DIR = "data/dl_windowed/"

# Plotting style
plt.rcParams.update(
    {
        "figure.dpi": 130,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "font.size": 11,
    }
)
PALETTE = {
    "Non-Stress": "#4C72B0",
    "Stress": "#DD8452",
    "Baseline": "#4C72B0",
    "Amusement": "#55A868",
}

print(f"Results dir : {RESULTS_DIR}")
print(f"JSON exists : {os.path.exists(RESULTS_JSON)}")

### Mode 1: Evaluate from saved results JSON

Loads `loso_results.json` and prints the full evaluation report: per-fold table, aggregate metrics, global confusion matrix, per-class metrics (precision, recall, specificity, F1), Cohen's κ, MCC, loss curve diagnostics, and inter-subject variability analysis.

In [ ]:
report = evaluate_from_results(
    results_path=RESULTS_JSON,
    output_dir=RESULTS_DIR,
)

### Visualisation 1: Per-Fold F1 Macro

Each bar is one LOSO fold (one test subject). The red dashed line is the mean across folds.  
High variance across subjects is the expected result for a centralised model and the core motivation for Federated Learning.

In [ ]:
fold_results = report["per_fold"]
subjects = [f"S{r['subject']}" for r in fold_results]
f1s = [r["f1_macro"] for r in fold_results]
mean_f1 = np.mean(f1s)
std_f1 = np.std(f1s, ddof=1)

colors = ["#DD8452" if f < 0.5 else "#4C72B0" for f in f1s]

fig, ax = plt.subplots(figsize=(12, 4))
bars = ax.bar(subjects, f1s, color=colors, edgecolor="white", linewidth=0.8)
ax.axhline(
    mean_f1,
    color="crimson",
    linestyle="--",
    linewidth=1.5,
    label=f"Mean F1 = {mean_f1:.3f} ± {std_f1:.3f}",
)
ax.axhline(0.5, color="grey", linestyle=":", linewidth=1, label="F1 = 0.5 threshold")

ax.set_xlabel("Test Subject (LOSO fold)")
ax.set_ylabel("Macro-F1")
ax.set_title("Per-Fold Macro-F1 — Centralised CNN-LSTM (LOSO-CV)")
ax.set_ylim(0, 1.05)
ax.legend()

# Annotate each bar
for bar, f1 in zip(bars, f1s):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.01,
        f"{f1:.2f}",
        ha="center",
        va="bottom",
        fontsize=8,
    )

plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, "per_fold_f1.png"), bbox_inches="tight")
plt.show()
print(f"Saved → {RESULTS_DIR}/per_fold_f1.png")

### Visualisation 2: Global Confusion Matrix

Summed across all 15 LOSO folds. Shows the overall error pattern of the centralised model.

In [ ]:
n_classes = report["config"].get("n_classes", 2)
class_names = (
    ["Non-Stress", "Stress"] if n_classes == 2 else ["Baseline", "Stress", "Amusement"]
)

global_cm = np.array(report["global_confusion_matrix"])

# Normalise row-wise (recall per class) for the colour scale, show raw counts as labels
cm_norm = global_cm.astype(float) / global_cm.sum(axis=1, keepdims=True)
annot = np.array(
    [
        [f"{global_cm[i,j]}\n({cm_norm[i,j]:.1%})" for j in range(n_classes)]
        for i in range(n_classes)
    ]
)

fig, ax = plt.subplots(figsize=(5 * n_classes / 2 + 2, 4))
sns.heatmap(
    cm_norm,
    annot=annot,
    fmt="",
    cmap="Blues",
    xticklabels=[f"Pred\n{c}" for c in class_names],
    yticklabels=[f"True\n{c}" for c in class_names],
    linewidths=0.5,
    linecolor="white",
    vmin=0,
    vmax=1,
    ax=ax,
)
ax.set_title(
    "Global Confusion Matrix - Centralised CNN-LSTM\n(counts + row-normalised recall)"
)
plt.tight_layout()
plt.savefig(
    os.path.join(RESULTS_DIR, "global_confusion_matrix.png"), bbox_inches="tight"
)
plt.show()
print(f"Saved → {RESULTS_DIR}/global_confusion_matrix.png")

### Visualisation 3: Per-Class Metrics (Sensitivity & Specificity)

For stress detection, sensitivity (recall for Stress) is the clinically critical metric AS missing a stress episode is more harmful than a false alarm.  
Specificity measures how well the model avoids false stress alarms.

In [ ]:
per_class = report["per_class_metrics"]

metrics_to_plot = ["precision", "recall", "specificity", "f1"]
metric_labels = ["Precision", "Sensitivity\n(Recall)", "Specificity", "F1"]

x = np.arange(len(metrics_to_plot))
width = 0.35 / max(n_classes - 1, 1) + 0.2
offsets = np.linspace(
    -width * (n_classes - 1) / 2, width * (n_classes - 1) / 2, n_classes
)

fig, ax = plt.subplots(figsize=(9, 4))
class_colors = ["#4C72B0", "#DD8452", "#55A868"]

for i, cname in enumerate(class_names):
    vals = [per_class[cname][m] for m in metrics_to_plot]
    bars = ax.bar(
        x + offsets[i],
        vals,
        width,
        label=cname,
        color=class_colors[i],
        alpha=0.85,
        edgecolor="white",
    )
    for bar, v in zip(bars, vals):
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.01,
            f"{v:.2f}",
            ha="center",
            va="bottom",
            fontsize=8,
        )

ax.set_xticks(x)
ax.set_xticklabels(metric_labels)
ax.set_ylabel("Score")
ax.set_ylim(0, 1.15)
ax.set_title("Per-Class Metrics — Global (Pooled) Predictions")
ax.legend()
ax.axhline(0.8, color="grey", linestyle=":", linewidth=1, alpha=0.6)

plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, "per_class_metrics.png"), bbox_inches="tight")
plt.show()

### Visualisation 4: Loss Curves All Folds

Training vs. validation loss per fold. A widening gap between train and val loss signals overfitting.  
Early stopping (patience=10 on val loss) is the primary control.
The vertical dashed line marks the mean convergence epoch.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4), sharey=False)

all_train, all_val, conv_epochs = [], [], []

for r in fold_results:
    sid = r["subject"]
    curves_path = os.path.join(RESULTS_DIR, f"fold_S{sid}", "loss_curves.json")
    if not os.path.exists(curves_path):
        continue
    with open(curves_path) as f:
        curves = json.load(f)
    tl, vl = curves.get("train_loss", []), curves.get("val_loss", [])
    if not tl:
        continue
    best_ep = int(np.argmin(vl))
    conv_epochs.append(best_ep)
    all_train.append(tl)
    all_val.append(vl)
    axes[0].plot(tl, alpha=0.4, linewidth=0.8, color="#4C72B0")
    axes[1].plot(vl, alpha=0.4, linewidth=0.8, color="#DD8452")

# Mean curves
if all_train:
    max_len = max(len(t) for t in all_train)
    pad = lambda lst: lst + [lst[-1]] * (max_len - len(lst))
    mean_train = np.mean([pad(t) for t in all_train], axis=0)
    mean_val = np.mean([pad(v) for v in all_val], axis=0)
    axes[0].plot(mean_train, color="#4C72B0", linewidth=2.5, label="Mean train loss")
    axes[1].plot(mean_val, color="#DD8452", linewidth=2.5, label="Mean val loss")
    mean_conv = int(np.mean(conv_epochs))
    for ax in axes:
        ax.axvline(
            mean_conv,
            color="crimson",
            linestyle="--",
            linewidth=1.2,
            label=f"Mean convergence (ep {mean_conv})",
        )

for ax, title in zip(axes, ["Training Loss", "Validation Loss"]):
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Cross-Entropy Loss")
    ax.set_title(title)
    ax.legend(fontsize=9)

plt.suptitle("Loss Curves All 15 LOSO Folds (thin) + Mean (thick)", y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, "loss_curves_all_folds.png"), bbox_inches="tight")
plt.show()

### Visualisation 5: Inter-Subject Variability  

The box plot below shows the distribution of Macro-F1 across the 15 LOSO folds.  
High spread and the presence of outlier subjects (below F1=0.5) is the empirical justification for Federated Learning 

In [ ]:
variability = report.get("variability_analysis", {})
f1s = [r["f1_macro"] for r in fold_results]
subj_ids = [f"S{r['subject']}" for r in fold_results]

fig, ax = plt.subplots(figsize=(8, 4))

# Box plot
bp = ax.boxplot(
    f1s,
    vert=True,
    patch_artist=True,
    widths=0.35,
    boxprops=dict(facecolor="#4C72B0", alpha=0.4),
    medianprops=dict(color="crimson", linewidth=2),
    whiskerprops=dict(linewidth=1.2),
    capprops=dict(linewidth=1.2),
    flierprops=dict(marker="o", markerfacecolor="grey", markersize=5),
)

# Strip jitter
np.random.seed(42)
jitter = np.random.uniform(-0.08, 0.08, size=len(f1s))
scatter = ax.scatter(
    [1 + j for j in jitter],
    f1s,
    color=["#DD8452" if f < 0.5 else "#4C72B0" for f in f1s],
    zorder=5,
    s=60,
    edgecolors="white",
    linewidth=0.5,
)

# Annotate outliers
for j, (sid, f1) in enumerate(zip(subj_ids, f1s)):
    if f1 < 0.5:
        ax.annotate(
            sid,
            (1 + jitter[j], f1),
            textcoords="offset points",
            xytext=(8, 0),
            fontsize=8,
            color="#DD8452",
        )

ax.axhline(0.5, color="grey", linestyle=":", linewidth=1, label="F1=0.5 threshold")
ax.set_xticks([])
ax.set_ylabel("Macro-F1")
ax.set_title(
    f"Inter-Subject Variability — Centralised CNN-LSTM\n"
    f"Range: {variability.get('f1_macro_range', 0):.3f}  |  "
    f"CV: {variability.get('f1_macro_cv', 0):.3f}  |  "
    f"Mean: {np.mean(f1s):.3f} ± {np.std(f1s, ddof=1):.3f}"
)
ax.legend()

plt.tight_layout()
plt.savefig(
    os.path.join(RESULTS_DIR, "inter_subject_variability.png"), bbox_inches="tight"
)
plt.show()

# Narrative output
worst_sid = variability.get("worst_subject")
best_sid = variability.get("best_subject")
below_05 = variability.get("subjects_below_0_5", [])
print(f"Best subject:  S{best_sid}  (F1={variability.get('best_f1_macro', 0):.4f})")
print(f"Worst subject: S{worst_sid} (F1={variability.get('worst_f1_macro', 0):.4f})")
if below_05:
    print(f"Subjects below F1=0.5: {below_05}")
    print("→ These subjects are prime candidates for FL personalisation.")
if variability.get("f1_macro_range", 0) > 0.3:
    print("→ Range > 0.3: high inter-subject variability confirms FL motivation.")

### Mode 2: Re-run Inference from Checkpoints 

Run for per-window predictions and deeper error analysis, or to verify the saved JSON results against fresh inference.  
This equires the windowed data to be available and takes longer to run and can therefore be skipped if the only results necessary are the aggregated ones from the previous mode.

In [ ]:
# Set RUN_CHECKPOINT_EVAL = True to re-run inference from saved model files.
# This is slower but gives global classification_report and confirms
# that checkpoint inference matches the training-time JSON results.
RUN_CHECKPOINT_EVAL = False

if RUN_CHECKPOINT_EVAL:
    checkpoint_report = evaluate_from_checkpoints(
        results_dir=RESULTS_DIR,
        windowed_dir=WINDOWED_DIR,
    )

    # sanity check: compare Mode 1 vs Mode 2 macro-F1
    m1_f1 = report["aggregate"]["f1_macro_mean"]
    m2_f1 = checkpoint_report["aggregate"]["f1_macro_mean"]
    delta = abs(m1_f1 - m2_f1)
    print("\nSanity check- Macro-F1:")
    print(f"  Mode 1 (JSON):        {m1_f1:.4f}")
    print(f"  Mode 2 (checkpoint):  {m2_f1:.4f}")
    print(
        f"  Delta:                {delta:.4f}  {'✓ OK' if delta < 1e-4 else 'mismatch! investigate'}"
    )
else:
    print("Checkpoint evaluation skipped (RUN_CHECKPOINT_EVAL=False).")

### Summary Table — Aggregate Results

summary of all metrics across all 15 LOSO folds.


In [ ]:
agg = report["aggregate"]
gm = report.get("global_metrics", {})

print("  CNN-LSTM LOSO-CV RESULTS SUMMARY")
print("\n")
print(f"  {'Metric':<22} {'Mean':>8}  {'±Std':>8}  {'[Min, Max]'}")
print("  " + "─" * 100)
for base in ["accuracy", "f1_macro", "f1_weighted", "precision", "recall"]:
    mean = agg.get(f"{base}_mean", float("nan"))
    std = agg.get(f"{base}_std", float("nan"))
    mn = agg.get(f"{base}_min", float("nan"))
    mx = agg.get(f"{base}_max", float("nan"))
    print(f"  {base:<22} {mean:>8.4f}  {std:>8.4f}  [{mn:.4f}, {mx:.4f}]")
print("  " + "─" * 100)
print(f"  {'Cohen κ':<22} {gm.get('cohen_kappa', float('nan')):>8.4f}")
print(f"  {'Matthews MCC':<22} {gm.get('matthews_corrcoef', float('nan')):>8.4f}")
print(f"\n  n_classes  : {n_classes} ({'binary' if n_classes == 2 else '3-class'})")
print(f"  device     : {report['config'].get('device_type', '?')}")
print(f"  window     : {report['config'].get('window_sec', '?')}s")
print(f"  optimizer  : {report['config'].get('optimizer', '?')}")
print(f"  n_folds    : {report['n_folds']}")

# FEDERATED LEARNING


This section implements and evaluates all FL algorithms on the WESAD wrist dataset using 15-fold Leave-One-Subject-Out (LOSO)
cross-validation.

Experimental progression:
1. Phase 1: Verify the WESADFLModel base/head split
2. Phase 2: Integration test with 1 round of FedAvg
3. Phase 3: FedAvg baseline with all 15 LOSO folds
4. Phase 4: FedRep: primary PFL method, all 15 folds
5. Phase 5: FedBABU ablation: fine-tuning epochs sweep
6. Phase 7: PeFLL: hypernetwork-based personalisation

**Primary metric**: Macro-F1 (handles class imbalance)
**Secondary**: Accuracy (literature comparability), Stress Recall (clinical relevance).

Algorithms included:
- FedAvg          
- FedRep          (primary PFL method)
- FedBABU         (ablation)
- PeFLL           (advanced PFL)

In [ ]:
# Imports and FL configuration

import copy
import gc
import json
import os
import pickle

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch

from src.data.wesad_dl_dataset import ALL_SUBJECT_IDS
from src.fl.clients.client_avg import FedAvgClient
from src.fl.clients.client_babu import FedBABUClient

# Clients
from src.fl.clients.client_rep import FedRepClient
from src.fl.peFLL.client_pefll import PeFLLClient
from src.fl.peFLL.server_pefll import PeFLLServer
from src.fl.servers.server_avg import FedAvgServer
from src.fl.servers.server_babu import FedBABUServer

# Servers
from src.fl.servers.server_rep import FedRepServer
from src.models.wesad_fl_model import build_fl_model_from_config

# FL infrastructure
from src.utils.fl_config import (
    FLConfig,
    make_fedavg_config,
    make_fedbabu_config,
    make_fedrep_config,
    make_pefll_config,
)
from src.utils.result_store import (
    load_fold_result,
)
from src.utils.wesad_fl_dataset import FLFoldConfig, build_fl_fold_datasets

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Let cuDNN benchmark conv/LSTM algorithms for our fixed input shapes.
# WESAD batches always have the same shape (batch_size=64, fixed signal
# lengths), so cuDNN can safely pick the fastest kernel for this shape
# on the first pass and reuse it. Without this flag, cuDNN defaults to
# a generic algorithm that works for any shape but is not the fastest.
if torch.cuda.is_available():
    torch.backends.cudnn.benchmark = True

RESULTS_DIR = "results/"
os.makedirs(RESULTS_DIR, exist_ok=True)

print(f"Device:       {DEVICE}")
print(f"Subjects:     {ALL_SUBJECT_IDS} ({len(ALL_SUBJECT_IDS)} total)")
print(f"Results dir:  {RESULTS_DIR}")

### Phase 1: WESADFLModel Verification

Confirm that the base/head split in WESADFLModel produces identical output to the original CNNLSTM when run sequentially.

- `model.base` = CNN branches + adaptive pools (shared features)
- `model.head` = LSTM + BatchNorm + classifier 

In [ ]:
# Base/head split verification
from src.models.wesad_fl_model import verify_equivalence

# 1. Verify forward-pass equivalence (seed-controlled, independent models)
dummy_signals = {
    "bvp": torch.randn(4, 1, 1024).to(DEVICE),
    "w_eda": torch.randn(4, 1, 64).to(DEVICE),
    "w_acc": torch.randn(4, 3, 512).to(DEVICE),
    "w_temp": torch.randn(4, 1, 64).to(DEVICE),
}
assert verify_equivalence(dummy_signals), "Split produces different output!"
print("  PASSED: base + head reproduce original CNNLSTM output")

# 2. Parameter count breakdown
config_p1 = FLConfig(device_type="wrist", label_mode="binary")
fl_model = build_fl_model_from_config(config_p1).to(DEVICE)

base_params = sum(p.numel() for p in fl_model.base.parameters())
head_params = sum(p.numel() for p in fl_model.head.parameters())
total_params = sum(p.numel() for p in fl_model.parameters())
print(f"  Base (CNN, shared):     {base_params:,} params")
print(f"  Head (LSTM, personal):  {head_params:,} params")
print(f"  Total:                  {total_params:,} params")
print(
    f"  Communication savings:  {100 * (1 - base_params / total_params):.1f}% "
    f"(only base is transmitted)"
)

del fl_model, dummy_signals
gc.collect()

### Phase 2: Integration Test - 1 Round of FedAvg

Instantiate a server with 15 clients, run 1 roundto test and confirm our hypothesis that loss will decrease. This validates the entire FL pipeline end-to-end
before committing to expensive 15-fold experiments.

In [ ]:
# Phase 2: Integration test

config_p2 = make_fedavg_config(
    fold_id=2,
    global_rounds=2,
    local_epochs=1,
    eval_gap=1,
)

fold_config_p2 = FLFoldConfig(test_subject=2)
server_p2 = FedAvgServer(config_p2, fold_config_p2, DEVICE)
server_p2.set_clients(FedAvgClient)

print("\n  Running 2 rounds of FedAvg on fold S2 ...")
metrics_p2 = server_p2.train()

if metrics_p2.test_subject_metrics is not None:
    print("\n  PASSED: FL pipeline works end-to-end")
    print(f"    Test subject F1: {metrics_p2.test_subject_metrics.macro_f1:.4f}")
else:
    print("  FAILED: no test subject metrics returned")

del server_p2
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

## Phase 3: FedAvg Baseline for all 15 LOSO Folds
Standard federated averaging with full model broadcast, full model aggregation and no personalisation.

As a sanity check FedAvg should approach Liu et al. 2021's ~92.8% accuracy on WESAD.

In [ ]:
def run_all_folds(
    algorithm,
    server_cls,
    client_cls,
    config_factory,
    result_dir="results/",
    **config_kwargs,
):
    """Run one algorithm across all 15 LOSO folds, skipping completed folds.
    Returns dict[subject_id → FoldMetrics | FoldResult].
    """
    all_metrics = {}
    fold_times = []

    for sid in ALL_SUBJECT_IDS:
        # Check if this fold already completed
        h5_path = os.path.join(result_dir, f"{algorithm}_fold{sid}.h5")
        pt_path = os.path.join("models", f"{algorithm}_fold{sid}_global.pt")

        if os.path.exists(h5_path) and os.path.exists(pt_path):
            try:
                existing = load_fold_result(h5_path)
                # Sanity: check it has the expected number of eval rounds
                # (100 rounds / eval_gap=5 = 20 in-loop evals + 1 final = 21)
                if len(existing.rs_test_f1) >= 21:
                    print(f"\n  {algorithm} - Fold S{sid}: SKIPPED (already completed)")
                    print(f"    Final test F1: {existing.test_subject_final_f1:.4f}")
                    all_metrics[sid] = existing
                    continue
                else:
                    print(
                        f"\n  {algorithm} - Fold S{sid}: incomplete "
                        f"({len(existing.rs_test_f1)} eval rounds), re-running"
                    )
            except Exception as e:
                print(
                    f"\n  {algorithm} - Fold S{sid}: corrupt result ({e}), re-running"
                )

        fold_start = time.time()
        print("\n")
        print(f"  {algorithm} - Fold S{sid}")
        print("\n")
        config = config_factory(fold_id=sid, **config_kwargs)
        fold_config = FLFoldConfig(test_subject=sid)
        server = server_cls(config, fold_config, DEVICE)
        server.set_clients(client_cls)
        final_metrics = server.train()
        all_metrics[sid] = final_metrics
        fold_times.append(time.time() - fold_start)

        # Cleanup per fold
        del server
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    # Summary
    test_f1s = []
    for sid, fm in all_metrics.items():
        # Handle both FoldMetrics (from training) and FoldResult (from loaded)
        if hasattr(fm, "test_subject_metrics") and fm.test_subject_metrics is not None:
            test_f1s.append(fm.test_subject_metrics.macro_f1)
        elif hasattr(fm, "test_subject_final_f1"):
            test_f1s.append(fm.test_subject_final_f1)

    print(f"\n{'='*50}")
    print(f"  {algorithm} 15-fold LOSO Summary")
    print(f"{'='*50}")
    print(f"  Mean test F1:  {np.mean(test_f1s):.4f} ± {np.std(test_f1s):.4f}")
    if fold_times:
        print(f"  Mean fold time: {np.mean(fold_times):.1f}s")
        print(f"  Total time:     {sum(fold_times):.1f}s")
    print(f"  Folds trained: {len(fold_times)} / {len(ALL_SUBJECT_IDS)}")
    return all_metrics


print("\n")
print("  PHASE 3: FedAvg Baseline")
print("\n")
fedavg_results = run_all_folds(
    algorithm="FedAvg",
    server_cls=FedAvgServer,
    client_cls=FedAvgClient,
    config_factory=make_fedavg_config,
    global_rounds=100,
    local_epochs=5,
)

In [ ]:
# Collect FedAvg F1s
fedavg_f1s = []
for sid, fm in fedavg_results.items():
    if hasattr(fm, "test_subject_final_f1"):
        # FoldResult (loaded from disk after skip)
        fedavg_f1s.append(fm.test_subject_final_f1)
    elif hasattr(fm, "test_subject_metrics") and fm.test_subject_metrics is not None:
        # FoldMetrics (from live training)
        fedavg_f1s.append(fm.test_subject_metrics.macro_f1)

print(f"Collected {len(fedavg_f1s)} FedAvg F1s")
print(f"Mean: {np.mean(fedavg_f1s):.4f}")

## Phase 4: FedRep - Primary PFL Method

Alternating head/base training per round (Collins et al. 2021):
1. Freeze base → train head for `plocal_epochs` (personal LSTM)
2. Freeze head → train base for `local_epochs` (shared CNN)

Only the base is communicated while the head stays local permanently.

Expected: FedRep should improve over FedAvg.  If not, the  base/head split is wrong for this architecture and we mustdebug before proceeding.

In [ ]:
# test

fedavg_s2 = load_fold_result("results/FedAvg_fold2.h5")

# ✅ Correct: use subject_ids to find the right index
subject_ids = fedavg_s2.subject_ids  # np.ndarray of ints
final_f1s = fedavg_s2.final_per_subject_f1  # np.ndarray, same order

idx = np.where(subject_ids == 2)[0]
s2_f1 = float(final_f1s[idx[0]]) if len(idx) > 0 else "N/A"
print(f"FedAvg S2 final F1: {s2_f1}")

# Or use the built-in property (already does this for the test subject)
print(f"FedAvg S2 final F1 (property): {fedavg_s2.test_subject_final_f1}")

# ✅ Safe for both printing and plotting
print(f"FedAvg S2 convergence curve: {fedavg_s2.rs_test_f1}")

# For plotting (as used in demo_chunk.py):
rounds = np.arange(len(fedavg_s2.rs_test_f1)) * 5  # eval_gap=5

In [ ]:
# FedRep on 15 LOSO folds

print("\n")
print("  PHASE 4: FedRep")
print("\n")

fedrep_results = run_all_folds(
    algorithm="FedRep",
    server_cls=FedRepServer,
    client_cls=FedRepClient,
    config_factory=make_fedrep_config,
    global_rounds=100,
    local_epochs=5,
    plocal_epochs=5,
)

# Compare FedRep vs FedAvg
fedrep_f1s = []
for sid, fm in fedrep_results.items():
    if fm.test_subject_metrics is not None:
        fedrep_f1s.append(fm.test_subject_metrics.macro_f1)

print("\n  FedRep vs FedAvg:")
print(f"    FedAvg mean F1: {np.mean(fedavg_f1s):.4f}")
print(f"    FedRep mean F1: {np.mean(fedrep_f1s):.4f}")
diff = np.mean(fedrep_f1s) - np.mean(fedavg_f1s)
print(f"    Δ = {diff:+.4f} ({'FedRep better' if diff > 0 else 'FedAvg better'})")

if diff <= 0:
    print("     WARNING: FedRep does not improve over FedAvg ->check base/head split!")

### Phase 5: FedBABU Ablation
Research Question: Does training the head during FL rounds (FedRep) help or hurt compared to freezing it (FedBABU)?
 
**Protocol**:
1. Run fine_tuning_epochs sweep {1, 2, 5, 10, 20} on fold S2 only
2. Select the best fine_tuning_epochs
3. Run all 15 folds at that setting
4. Produce Figure: macro-F1 vs fine_tuning_epochs


In [ ]:
# FedBABU fine-tuning sweep (fold S2)

print("\n")
print("FedBABU Fine-Tuning Sweep (fold S2)")
print("\n")

ft_epochs_sweep = [1, 2, 5, 10, 20]
babu_sweep_results = {}

for ft_epochs in ft_epochs_sweep:
    print(f"\n  FedBABU: fine_tuning_epochs = {ft_epochs}")

    config = make_fedbabu_config(
        fold_id=2,
        global_rounds=100,
        local_epochs=5,
        fine_tuning_epochs=ft_epochs,
    )
    fold_config = FLFoldConfig(test_subject=2)
    server = FedBABUServer(config, fold_config, DEVICE)
    server.set_clients(FedBABUClient)
    final_metrics = server.train()

    test_f1 = (
        final_metrics.test_subject_metrics.macro_f1
        if final_metrics.test_subject_metrics
        else 0.0
    )
    babu_sweep_results[ft_epochs] = {
        "macro_f1": test_f1,
        "accuracy": (
            final_metrics.test_subject_metrics.accuracy
            if final_metrics.test_subject_metrics
            else 0.0
        ),
        "stress_recall": (
            final_metrics.test_subject_metrics.stress_recall
            if final_metrics.test_subject_metrics
            else 0.0
        ),
    }
    print(f"    F1 = {test_f1:.4f}")

    del server
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

# Find best
best_ft = max(babu_sweep_results, key=lambda k: babu_sweep_results[k]["macro_f1"])
print(
    f"\n  Best fine_tuning_epochs = {best_ft} "
    f"(F1 = {babu_sweep_results[best_ft]['macro_f1']:.4f})"
)

In [ ]:
# FedBABU on 15 LOSO folds at best setting

print("\n")
print(f"  PHASE 5b: FedBABU — 15 Folds (fine_tuning_epochs={best_ft})")
print("\n")

fedbabu_results = run_all_folds(
    algorithm="FedBABU",
    server_cls=FedBABUServer,
    client_cls=FedBABUClient,
    config_factory=make_fedbabu_config,
    global_rounds=100,
    local_epochs=5,
    fine_tuning_epochs=best_ft,
)

In [ ]:
# FedBABU ablation plot

fig, ax = plt.subplots(figsize=(8, 5))

# FedBABU sweep curve
ft_values = sorted(babu_sweep_results.keys())
f1_values = [babu_sweep_results[k]["macro_f1"] for k in ft_values]

ax.plot(
    ft_values,
    f1_values,
    "o-",
    color="#FF9800",
    linewidth=2,
    markersize=8,
    markerfacecolor="white",
    markeredgewidth=2,
    label="FedBABU (fold S2)",
)

# FedRep reference line (fold S2)
fedrep_s2_f1 = (
    fedrep_results[2].test_subject_metrics.macro_f1
    if fedrep_results[2].test_subject_metrics
    else 0
)
ax.axhline(
    y=fedrep_s2_f1,
    color="#2196F3",
    linestyle="--",
    linewidth=1.5,
    label=f"FedRep (fold S2): {fedrep_s2_f1:.3f}",
)

# FedAvg reference line (fold S2)
fedavg_s2_f1 = (
    fedavg_results[2].test_subject_metrics.macro_f1
    if fedavg_results[2].test_subject_metrics
    else 0
)
ax.axhline(
    y=fedavg_s2_f1,
    color="gray",
    linestyle=":",
    linewidth=1.5,
    label=f"FedAvg (fold S2): {fedavg_s2_f1:.3f}",
)

ax.set_xlabel("Fine-Tuning Epochs", fontsize=13)
ax.set_ylabel("Macro-F1", fontsize=13)
ax.set_title("FedBABU Ablation: Macro-F1 vs Fine-Tuning Epochs\n(Fold S2)", fontsize=14)
ax.set_xticks(ft_values)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

plt.tight_layout()
fig.savefig(
    os.path.join(RESULTS_DIR, "figure2_fedbabu_ablation.pdf"),
    bbox_inches="tight",
    dpi=300,
)
fig.savefig(
    os.path.join(RESULTS_DIR, "figure2_fedbabu_ablation.png"),
    bbox_inches="tight",
    dpi=300,
)
plt.show()
print(f"  Figure 2 saved → {RESULTS_DIR}figure2_fedbabu_ablation.pdf")

### PeFLL - Hypernetwork-Based Personalisation

PeFLL (Scott et al. 2024) generates personalised LSTM head parameters from a client embedding vector.  The server jointly trains an embedding network + hypernetwork.  The CNN base is
trained via FedAvg-style aggregation.

Experimental approach: evaluate PeFLL on completely unseen subjects (not seen during training). This tests PeFLL's deployment-time advantage and explores a legitimate gap inliterature.

In [ ]:
# PeFLL —> 15 LOSO folds

print("\n")
print("  PHASE 7: PeFLL")
print("\n")

pefll_results = run_all_folds(
    algorithm="PeFLL",
    server_cls=PeFLLServer,
    client_cls=PeFLLClient,
    config_factory=make_pefll_config,
    global_rounds=100,
    local_epochs=5,
    embedding_dim=64,
    hypernet_hidden=512,
)

In [ ]:
# PeFLL unseen client experiment

# Train PeFLL on 12 subjects, evaluate on 3 completely held-out subjects without any fine-tuning.

print("\n")
print("  PeFLL Unseen Client Experiment")
print("-" * 60)

# Hold out subjects S15, S16, S17 and train on remaining 12
unseen_subjects = [15, 16, 17]
train_subjects_pefll = [s for s in ALL_SUBJECT_IDS if s not in unseen_subjects]

print(f"  Training subjects:  {train_subjects_pefll}")
print(f"  Unseen subjects:    {unseen_subjects}")

# Use S2 as test (it's in training set) for the fold config, but the real evaluation is on the unseen subjects
config_unseen = make_pefll_config(
    fold_id=2,
    global_rounds=100,
    local_epochs=5,
    embedding_dim=64,
    hypernet_hidden=512,
)

# Build a custom fold config where only 12 subjects participate
from src.utils.wesad_fl_dataset import FLFoldConfig

fold_config_unseen = FLFoldConfig(
    test_subject=2,
    train_subjects=train_subjects_pefll,
)

server_unseen = PeFLLServer(config_unseen, fold_config_unseen, DEVICE)
server_unseen.set_clients(PeFLLClient)
server_unseen.train()

# Evaluate on unseen subjects
unseen_results = {}
for unseen_sid in unseen_subjects:
    print(f"\n  Evaluating unseen subject S{unseen_sid}...")
    try:
        dm = build_fl_fold_datasets(
            fold_config=FLFoldConfig(test_subject=unseen_sid),
            windowed_dir=config_unseen.windowed_dir,
            device_type=config_unseen.device_type,
            window_sec=config_unseen.window_sec,
            label_map=config_unseen.label_map,
            batch_size=config_unseen.batch_size,
            signals_to_load=config_unseen.signals_to_load,
        )
        result = server_unseen.evaluate_unseen_client(
            data_manager=dm[unseen_sid],
            subject_id=unseen_sid,
        )
        unseen_results[unseen_sid] = result
        print(
            f"    S{unseen_sid}: F1={result.get('macro_f1', 0):.4f}, "
            f"Acc={result.get('accuracy', 0):.4f}"
        )
    except Exception as e:
        print(f"    S{unseen_sid}: FAILED — {e}")

del server_unseen
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

### Convergence & Cross-Method Visualisations

For all methods produce the core visualisations:
- Macro-F1 vs federation round (convergence curves)
- Per-subject F1 for best method
- Per-subject variability plot 
- Interim cross-method comparison table

In [ ]:
# Convergence curves (F1 vs round)

from src.utils.result_store import load_fold_result

fig, ax = plt.subplots(figsize=(10, 5))

method_styles = {
    "FedAvg": {"color": "#9E9E9E", "linestyle": "-", "label": "FedAvg"},
    "FedRep": {"color": "#2196F3", "linestyle": "-", "label": "FedRep"},
    "FedBABU": {"color": "#FF9800", "linestyle": "--", "label": "FedBABU"},
    "PeFLL": {"color": "#4CAF50", "linestyle": "-.", "label": "PeFLL"},
}

for algo, style in method_styles.items():
    try:
        result = load_fold_result(algorithm=algo, fold_id=2, result_dir=RESULTS_DIR)
        rounds = np.arange(len(result.rs_test_f1)) * 5  # eval_gap=5
        ax.plot(rounds, result.rs_test_f1, linewidth=2, **style)
    except (FileNotFoundError, KeyError):
        print(f"  {algo} fold S2 not found — skipping convergence curve")

ax.set_xlabel("Federation Round", fontsize=13)
ax.set_ylabel("Macro-F1", fontsize=13)
ax.set_title(
    "Convergence: Macro-F1 vs Federation Round\n(Fold S2, test subject)", fontsize=14
)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)

plt.tight_layout()
fig.savefig(
    os.path.join(RESULTS_DIR, "figure1_convergence.pdf"), bbox_inches="tight", dpi=300
)
fig.savefig(
    os.path.join(RESULTS_DIR, "figure1_convergence.png"), bbox_inches="tight", dpi=300
)
plt.show()
print(f"  Figure 1 saved → {RESULTS_DIR}figure1_convergence.pdf")

In [ ]:
# Per-subject F1 for FedRep

print("\n")
print("  TABLE: Per-Subject Macro-F1 = FedRep (15 LOSO Folds)")
print("\n")
print(f"  {'Subject':<10} {'FedAvg F1':>10} {'FedRep F1':>10} {'Δ':>8}")
print("\n")

for sid in ALL_SUBJECT_IDS:
    fa_f1 = (
        fedavg_results[sid].test_subject_metrics.macro_f1
        if fedavg_results[sid].test_subject_metrics
        else 0
    )
    fr_f1 = (
        fedrep_results[sid].test_subject_metrics.macro_f1
        if fedrep_results[sid].test_subject_metrics
        else 0
    )
    delta = fr_f1 - fa_f1
    marker = "▲" if delta > 0.01 else "▼" if delta < -0.01 else "─"
    print(f"  S{sid:<8} {fa_f1:>10.4f} {fr_f1:>10.4f} {delta:>+8.4f} {marker}")

In [ ]:
# Per-subject variability visualisation

# Grouped bar chart: FedAvg vs FedRep vs PeFLL per subject
fig, ax = plt.subplots(figsize=(14, 5))

x = np.arange(len(ALL_SUBJECT_IDS))
width = 0.25

methods_to_plot = {
    "FedAvg": (fedavg_results, "#9E9E9E"),
    "FedRep": (fedrep_results, "#2196F3"),
}

# Add PeFLL if available
if pefll_results:
    methods_to_plot["PeFLL"] = (pefll_results, "#4CAF50")

for i, (name, (results_dict, color)) in enumerate(methods_to_plot.items()):
    f1s = []
    for sid in ALL_SUBJECT_IDS:
        if sid in results_dict and results_dict[sid].test_subject_metrics:
            f1s.append(results_dict[sid].test_subject_metrics.macro_f1)
        else:
            f1s.append(0)
    offset = (i - len(methods_to_plot) / 2 + 0.5) * width
    ax.bar(
        x + offset, f1s, width, label=name, color=color, alpha=0.85, edgecolor="white"
    )

ax.set_xlabel("Test Subject", fontsize=12)
ax.set_ylabel("Macro-F1", fontsize=12)
ax.set_title("Per-Subject Macro-F1 Across Methods (15 LOSO Folds)", fontsize=14)
ax.set_xticks(x)
ax.set_xticklabels([f"S{s}" for s in ALL_SUBJECT_IDS], fontsize=9)
ax.legend(fontsize=11)
ax.grid(axis="y", alpha=0.3)

plt.tight_layout()
fig.savefig(
    os.path.join(RESULTS_DIR, "per_subject_comparison.pdf"),
    bbox_inches="tight",
    dpi=300,
)
fig.savefig(
    os.path.join(RESULTS_DIR, "per_subject_comparison.png"),
    bbox_inches="tight",
    dpi=300,
)
plt.show()
print(f"  Per-subject comparison saved → {RESULTS_DIR}per_subject_comparison.pdf")

In [ ]:
#  Interim cross-method table (withiout DP)

print("\n")
print("  INTERIM CROSS-METHOD COMPARISON ")
print("\n")
print(f"  {'Method':<12} {'Macro-F1':>14} {'Accuracy':>14} {'Stress Recall':>15}")
print(f"  {'-' * 60}")

all_results_dict = {
    "FedAvg": fedavg_results,
    "FedRep": fedrep_results,
    "FedBABU": fedbabu_results,
    "PeFLL": pefll_results,
}

for name, results_dict in all_results_dict.items():
    f1s, accs, srs = [], [], []
    for sid, fm in results_dict.items():
        if fm.test_subject_metrics is not None:
            f1s.append(fm.test_subject_metrics.macro_f1)
            accs.append(fm.test_subject_metrics.accuracy)
            srs.append(fm.test_subject_metrics.stress_recall)

    if f1s:
        print(
            f"  {name:<12} "
            f"{np.mean(f1s):.4f} ± {np.std(f1s):.4f}  "
            f"{np.mean(accs):.4f} ± {np.std(accs):.4f}  "
            f"{np.mean(srs):.4f} ± {np.std(srs):.4f}"
        )
    else:
        print(f"  {name:<12} {'no results':>14}")

In [ ]:
# Pickle all results for easy reload
phase3_7_results = {
    "FedAvg": {
        sid: {
            "f1": fm.test_subject_metrics.macro_f1 if fm.test_subject_metrics else None,
            "acc": (
                fm.test_subject_metrics.accuracy if fm.test_subject_metrics else None
            ),
            "sr": (
                fm.test_subject_metrics.stress_recall
                if fm.test_subject_metrics
                else None
            ),
        }
        for sid, fm in fedavg_results.items()
    },
    "FedRep": {
        sid: {
            "f1": fm.test_subject_metrics.macro_f1 if fm.test_subject_metrics else None,
            "acc": (
                fm.test_subject_metrics.accuracy if fm.test_subject_metrics else None
            ),
            "sr": (
                fm.test_subject_metrics.stress_recall
                if fm.test_subject_metrics
                else None
            ),
        }
        for sid, fm in fedrep_results.items()
    },
    "FedBABU": {
        sid: {
            "f1": fm.test_subject_metrics.macro_f1 if fm.test_subject_metrics else None,
            "acc": (
                fm.test_subject_metrics.accuracy if fm.test_subject_metrics else None
            ),
            "sr": (
                fm.test_subject_metrics.stress_recall
                if fm.test_subject_metrics
                else None
            ),
        }
        for sid, fm in fedbabu_results.items()
    },
    "PeFLL": {
        sid: {
            "f1": fm.test_subject_metrics.macro_f1 if fm.test_subject_metrics else None,
            "acc": (
                fm.test_subject_metrics.accuracy if fm.test_subject_metrics else None
            ),
            "sr": (
                fm.test_subject_metrics.stress_recall
                if fm.test_subject_metrics
                else None
            ),
        }
        for sid, fm in pefll_results.items()
    },
    "FedBABU_sweep": babu_sweep_results,
    "PeFLL_unseen": unseen_results,
}

results_pkl_path = os.path.join(RESULTS_DIR, "fl_phase3_7_results.pkl")
with open(results_pkl_path, "wb") as f:
    pickle.dump(phase3_7_results, f)
print(f"\n  Phase 3–7 results saved → {results_pkl_path}")

## Federated Learning: Privacy Extension & Analysis

This section runs:
1. FedRep with Differential Privacy (ε sweep)
2. Deep Leakage from Gradients (DLG) attack
3. Cross-method comparison tables & dissertation figures

In [ ]:
# Imports and config

import gc
import json
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import wilcoxon
import seaborn as sns
import torch

from src.data.wesad_dl_dataset import ALL_SUBJECT_IDS

# Clients
from src.fl.clients.client_avg import FedAvgClient
from src.fl.clients.client_babu import FedBABUClient
from src.fl.clients.client_rep import FedRepClient
from src.fl.clients.client_rep_dp import FedRepDPClient
from src.fl.peFLL.client_pefll import PeFLLClient
from src.fl.peFLL.server_pefll import PeFLLServer

# Servers
from src.fl.servers.server_avg import FedAvgServer
from src.fl.servers.server_babu import FedBABUServer
from src.fl.servers.server_rep import FedRepServer
from src.fl.servers.server_rep_dp import FedRepDPServer, run_epsilon_sweep

# DLG
from src.utils.dlg import DLGResult, call_dlg_for_client, print_dlg_results
from src.utils.fl_config import (
    FLConfig,
    make_fedavg_config,
    make_fedbabu_config,
    make_fedrep_config,
    make_fedrep_dp_config,
    make_pefll_config,
)
from src.utils.result_store import load_fold_result
from src.utils.wesad_fl_dataset import FLFoldConfig, build_fl_fold_datasets

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
RESULTS_DIR = "results/"
os.makedirs(RESULTS_DIR, exist_ok=True)

print(f"Device: {DEVICE}")
print(f"Subjects: {ALL_SUBJECT_IDS}")
print(f"Results dir: {RESULTS_DIR}")

## FedRep with Differential Privacy

Apply DP-SGD (via Opacus) to FedRep's shared CNN base. The head (LSTM) is never communicated and does not need DP.

Experimental design: 
- ε ∈ {1, 5, 10, ∞} where ∞ = standard FedRep (no DP)
- δ = 1/N_min_windows (smallest client's training set)
- Per-sample gradient clipping at C = 1.0
- Run ε sweep on fold S2 first, then all 15 folds

**Literature target**: Benouis et al. 2024 achieved ~90% stress recognition at ε=15 with a 1D-CNN multitask model on WESAD.
Our CNN-LSTM with personalisation should match or exceed this at tighter ε.

In [ ]:
# Epsilon sweep on fold S2 forquick validation

# Run the ε sweep on a single fold first to validate the setup
# and get a quick look at the privacy-utility tradeoff. Then we will run it on all 15 folds

print("\n")
print(" Epsilon Sweep (fold S2)")
print("\n")

sweep_fold2 = run_epsilon_sweep(
    fold_id=2,
    epsilons=[1.0, 5.0, 10.0, float("inf")],
    device=str(DEVICE),
    global_rounds=100,
    local_epochs=5,
    plocal_epochs=5,
)

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

In [ ]:
# Full epsilon sweep wth all 15 LOSO folds

# After validating on fold S2, run across all 15 folds to produces the data for the privacy-utility curve.

from src.fl.servers.server_rep_dp import run_full_epsilon_sweep

print("\n")
print("  Phase 8: Full Epsilon Sweep — All 15 LOSO Folds")
print("\n")

dp_all_results = run_full_epsilon_sweep(
    epsilons=[1.0, 5.0, 10.0, float("inf")],
    device=str(DEVICE),
    global_rounds=100,
    local_epochs=5,
    plocal_epochs=5,
)

# Save raw results for later analysis
import pickle

dp_results_path = os.path.join(RESULTS_DIR, "dp_epsilon_sweep_all_folds.pkl")
with open(dp_results_path, "wb") as f:
    pickle.dump(dp_all_results, f)
print(f"\nDP sweep results saved → {dp_results_path}")

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

## Deep Leakage from Gradients (DLG) Attack

Aims to demonstrate that gradient updates shared during FL can be inverted to reconstruct original training biosignals, then show that DP
reduces reconstruction quality (higher nMSE = better privacy).

Run on fold S2 only as a demonstration, then use fold 1 PSNR values as representative. 

**Methods compared**:
- FedAvg (no DP) — full model shared
- FedRep (no DP) — base only shared
- FedRep-DP ε=5 — moderate privacy
- FedRep-DP ε=1 — strong privacy

**Primary metric**: nMSE (< 1.0 = privacy leak, ≥ 1.0 = privacy OK)

In [ ]:
# DLG attack helper


def run_dlg_experiment(
    algorithm: str,
    fold_id: int = 2,
    n_iters: int = 100,
    batch_num: int = 2,
    dp_epsilon: float = float("inf"),
    global_rounds: int = 100,
    **config_kwargs,
) -> list[DLGResult]:
    """Train a method and run DLG attack at the final round."""
    # Build config
    if algorithm == "FedRepDP":
        config = make_fedrep_dp_config(
            fold_id=fold_id,
            dp_epsilon=dp_epsilon,
            global_rounds=global_rounds,
            dlg_eval=False,
            **config_kwargs,
        )
        server_cls, client_cls = FedRepDPServer, FedRepDPClient
    elif algorithm == "FedRep":
        config = make_fedrep_config(
            fold_id=fold_id,
            global_rounds=global_rounds,
            dlg_eval=False,
            **config_kwargs,
        )
        server_cls, client_cls = FedRepServer, FedRepClient
    elif algorithm == "FedAvg":
        config = make_fedavg_config(
            fold_id=fold_id,
            global_rounds=global_rounds,
            dlg_eval=False,
            **config_kwargs,
        )
        server_cls, client_cls = FedAvgServer, FedAvgClient
    else:
        raise ValueError(f"Unknown algorithm: {algorithm}")

    fold_config = FLFoldConfig(test_subject=fold_id)
    server = server_cls(config, fold_config, DEVICE)
    server.set_clients(client_cls)

    # Train
    print(f"\n  Training {algorithm} (fold S{fold_id})...")
    server.train()

    # Run DLG on first 3 clients after final round
    print(f"\n  Running DLG attack on {algorithm}...")

    dlg_results = []
    for i, (cid, client_base_copy) in enumerate(
        zip(server.uploaded_ids[:3], server.uploaded_models[:3])
    ):
        client = server._get_client_by_id(cid)
        if client is None:
            continue

        trainloader = client.load_train_data()

        result = call_dlg_for_client(
            global_base=copy.deepcopy(server.global_model.base),
            client_full_model=client.model,
            client_base=client_base_copy,
            trainloader=trainloader,
            device=DEVICE,
            batch_num=batch_num,
            n_iters=n_iters,
            client_id=cid,
        )
        if result is not None:
            dlg_results.append(result)

    print_dlg_results(dlg_results, round_num=config.global_rounds)

    # Cleanup
    del server
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return dlg_results

In [ ]:
# Run DLG across methods
print("\n")
print("  Phase 9: DLG Attack: Fold S2")
print("\n")

dlg_experiments = {}

# 1. FedAvg (no DP)
print("\n")
print("  DLG on FedAvg (no DP)")
print("\n")
dlg_experiments["FedAvg"] = run_dlg_experiment("FedAvg", fold_id=2)

# 2. FedRep (no DP), base only shraed
print("\n")
print("  DLG on FedRep (no DP)")
print("\n")
dlg_experiments["FedRep"] = run_dlg_experiment("FedRep", fold_id=2)

# 3. FedRep-DP ε=5 for moderate privacy
print("\n")
print("  DLG on FedRep-DP (ε=5)")
print("\n")
dlg_experiments["FedRep-DP ε=5"] = run_dlg_experiment(
    "FedRepDP",
    fold_id=2,
    dp_epsilon=5.0,
)

# 4. FedRep-DP ε=1 as strong privacy
print("\n")
print("  DLG on FedRep-DP (ε=1)")
print("\n")
dlg_experiments["FedRep-DP ε=1"] = run_dlg_experiment(
    "FedRepDP",
    fold_id=2,
    dp_epsilon=1.0,
)

# Save DLG results
dlg_results_path = os.path.join(RESULTS_DIR, "dlg_results_fold2.pkl")
with open(dlg_results_path, "wb") as f:
    pickle.dump(dlg_experiments, f)
print(f"\nDLG results saved → {dlg_results_path}")

In [ ]:
#  DLG summary table

print("\n")
print("  DLG ATTACK SUMMARY for Fold S2")
print("\n")
print(
    f"  {'Method':<20} {'nMSE':>8} {'PSNR (dB)':>10} {'grad_diff':>10} {'Status':<12}"
)
print("\n")

for method, results in dlg_experiments.items():
    if results:
        nmses = [r.aggregate_nmse for r in results]
        psnrs = [r.aggregate_psnr for r in results if not np.isnan(r.aggregate_psnr)]
        gdiffs = [r.final_grad_diff for r in results if not np.isinf(r.final_grad_diff)]

        mean_nmse = np.mean(nmses)
        mean_psnr = np.mean(psnrs) if psnrs else float("nan")
        mean_gdiff = np.mean(gdiffs) if gdiffs else float("inf")
        status = "LEAK RISK" if mean_nmse < 1.0 else "privacy OK"

        psnr_str = f"{mean_psnr:.2f}" if not np.isnan(mean_psnr) else "N/A"
        gdiff_str = f"{mean_gdiff:.6f}" if not np.isinf(mean_gdiff) else "N/A"
        print(
            f"  {method:<20} {mean_nmse:>8.4f} {psnr_str:>10} {gdiff_str:>10} {status:<12}"
        )
    else:
        print(f"  {method:<20} {'N/A':>8} {'N/A':>10} {'N/A':>10} {'no data':<12}")

# Per-modality breakdown
print("\n  Per-Modality nMSE:")
print(f"  {'Method':<20}", end="")
all_mods = set()
for results in dlg_experiments.values():
    for r in results:
        all_mods.update(r.per_modality_nmse.keys())
for mod in sorted(all_mods):
    print(f" {mod:>10}", end="")
print()

for method, results in dlg_experiments.items():
    if results:
        print(f"  {method:<20}", end="")
        for mod in sorted(all_mods):
            vals = [r.per_modality_nmse.get(mod, float("nan")) for r in results]
            valid = [v for v in vals if not np.isnan(v)]
            if valid:
                print(f" {np.mean(valid):>10.3f}", end="")
            else:
                print(f" {'N/A':>10}", end="")
        print()

## Cross-Method Analysis & Dissertation Figures

Produces:
- Mean ± std macro-F1 across 15 folds for all methods
- Macro-F1 convergence curves (FedAvg, FedRep, FedBABU, PeFLL)
- FedBABU fine-tuning ablation
- DP privacy-utility curve (macro-F1 vs ε)
- DLG nMSE across methods (bar chart)
- Wilcoxon signed-rank (non-parametric, appropriate dtatidtical test for 15 non-normally-distributed LOSO folds)



In [ ]:
# Load all results for cross-method comparison

# Collect per-fold test-subject F1 from each method's HDF5 results.
# Each method stores results from its server.save_results() call.

from src.utils.result_store import load_fold_result

ALGORITHMS = ["FedAvg", "FedRep", "FedBABU", "PeFLL", "FedRepDP"]

# Collect algorithm → list of 15 per-fold test F1 values
all_method_f1s = {}
all_method_accs = {}
all_method_stress_recalls = {}

for algo in ALGORITHMS:
    fold_f1s = []
    fold_accs = []
    fold_srs = []

    for sid in ALL_SUBJECT_IDS:
        try:
            result = load_fold_result(
                algorithm=algo,
                fold_id=sid,
                result_dir=RESULTS_DIR,
            )
            # Use the test subject's final F1
            fold_f1s.append(result.test_subject_final_f1)
            fold_accs.append(
                float(result.rs_test_acc[-1]) if result.rs_test_acc else 0.0
            )
            fold_srs.append(
                float(result.rs_test_stress_recall[-1])
                if result.rs_test_stress_recall
                else 0.0
            )
        except (FileNotFoundError, KeyError) as e:
            print(f"  Warning: {algo} fold S{sid} not found — {e}")
            fold_f1s.append(np.nan)
            fold_accs.append(np.nan)
            fold_srs.append(np.nan)

    all_method_f1s[algo] = fold_f1s
    all_method_accs[algo] = fold_accs
    all_method_stress_recalls[algo] = fold_srs

print("Results loaded for:", list(all_method_f1s.keys()))

In [ ]:
# Mean ± std macro-F1 across 15 folds

print("\n")
print(" Cross-Method Comparison (15-fold LOSO)")
print("\n")
print(f"  {'Method':<16} {'Macro-F1':>12} {'Accuracy':>12} {'Stress Recall':>14}")
print(f"  {'-' * 80}")

display_names = {
    "FedAvg": "FedAvg",
    "FedRep": "FedRep",
    "FedBABU": "FedBABU",
    "PeFLL": "PeFLL",
    "FedRepDP": "FedRep-DP",
}

table_rows = []
for algo in ALGORITHMS:
    f1s = [v for v in all_method_f1s[algo] if not np.isnan(v)]
    accs = [v for v in all_method_accs[algo] if not np.isnan(v)]
    srs = [v for v in all_method_stress_recalls[algo] if not np.isnan(v)]

    if f1s:
        name = display_names.get(algo, algo)
        f1_str = f"{np.mean(f1s):.4f} ± {np.std(f1s):.4f}"
        acc_str = f"{np.mean(accs):.4f} ± {np.std(accs):.4f}"
        sr_str = f"{np.mean(srs):.4f} ± {np.std(srs):.4f}"
        print(f"  {name:<16} {f1_str:>12} {acc_str:>12} {sr_str:>14}")

        table_rows.append(
            {
                "Method": name,
                "Macro-F1 Mean": np.mean(f1s),
                "Macro-F1 Std": np.std(f1s),
                "Accuracy Mean": np.mean(accs),
                "Stress Recall Mean": np.mean(srs),
            }
        )

table1_df = pd.DataFrame(table_rows)
table1_df.to_csv(os.path.join(RESULTS_DIR, "table1_cross_method.csv"), index=False)
print(f"\n  Saved → {RESULTS_DIR}table1_cross_method.csv")

In [ ]:
# Statistical significance: Wilcoxon signed-rank

# Wilcoxon signed-rank test (not paired t-test) because LOSO results across 15 subjects are not normally distributed.
# Primary comparisons: FedRep vs FedAvg, PeFLL vs FedRep.

print("\n")
print("  STATISTICAL SIGNIFICANCE (Wilcoxon Signed-Rank Test)")
print("\n")

comparisons = [
    ("FedRep", "FedAvg"),
    ("PeFLL", "FedRep"),
    ("FedBABU", "FedAvg"),
    ("PeFLL", "FedAvg"),
]

for method_a, method_b in comparisons:
    f1_a = np.array(all_method_f1s.get(method_a, []))
    f1_b = np.array(all_method_f1s.get(method_b, []))

    # Remove folds where either has NaN
    mask = ~(np.isnan(f1_a) | np.isnan(f1_b))
    f1_a_clean = f1_a[mask]
    f1_b_clean = f1_b[mask]

    if len(f1_a_clean) < 5:
        print(
            f"  {method_a} vs {method_b}: insufficient data ({len(f1_a_clean)} folds)"
        )
        continue

    # Two-sided Wilcoxon
    stat, p_value = wilcoxon(f1_a_clean, f1_b_clean)
    diff = np.mean(f1_a_clean) - np.mean(f1_b_clean)
    direction = "better" if diff > 0 else "worse"
    sig = (
        "***"
        if p_value < 0.001
        else "**" if p_value < 0.01 else "*" if p_value < 0.05 else "n.s."
    )

    print(
        f"  {method_a} vs {method_b}: "
        f"Δ={diff:+.4f} ({method_a} {direction}), "
        f"p={p_value:.4f} {sig}"
    )

In [ ]:
# DP Privacy-Utility Curve

# Load DP sweep results
dp_results_path = os.path.join(RESULTS_DIR, "dp_epsilon_sweep_all_folds.pkl")
if os.path.exists(dp_results_path):
    with open(dp_results_path, "rb") as f:
        dp_all_results = pickle.load(f)

    epsilons_plot = [1.0, 5.0, 10.0]
    eps_labels = ["1", "5", "10", "∞"]

    mean_f1s = []
    std_f1s = []
    for eps in [1.0, 5.0, 10.0, float("inf")]:
        f1s = [r["macro_f1"] for r in dp_all_results[eps]]
        mean_f1s.append(np.mean(f1s))
        std_f1s.append(np.std(f1s))

    fig, ax = plt.subplots(figsize=(8, 5))

    x_pos = [0, 1, 2, 3]
    ax.errorbar(
        x_pos,
        mean_f1s,
        yerr=std_f1s,
        fmt="o-",
        capsize=5,
        capthick=2,
        linewidth=2,
        markersize=8,
        color="#2196F3",
        markerfacecolor="white",
        markeredgewidth=2,
    )

    # Highlight no-DP baseline
    ax.axhline(
        y=mean_f1s[-1],
        color="gray",
        linestyle="--",
        alpha=0.5,
        label=f"FedRep (no DP): {mean_f1s[-1]:.3f}",
    )

    ax.set_xticks(x_pos)
    ax.set_xticklabels(eps_labels, fontsize=12)
    ax.set_xlabel("Privacy Budget ε", fontsize=13)
    ax.set_ylabel("Macro-F1", fontsize=13)
    ax.set_title("Privacy-Utility Tradeoff: FedRep-DP on WESAD", fontsize=14)
    ax.legend(fontsize=11)
    ax.grid(True, alpha=0.3)

    # Annotate Benouis et al. comparison
    ax.annotate(
        "Benouis et al. 2024:\n~0.90 at ε=15",
        xy=(2, mean_f1s[2]),
        xytext=(2.5, mean_f1s[2] - 0.05),
        fontsize=9,
        fontstyle="italic",
        color="gray",
        arrowprops=dict(arrowstyle="->", color="gray", alpha=0.5),
    )

    plt.tight_layout()
    fig.savefig(
        os.path.join(RESULTS_DIR, "figure3_dp_privacy_utility.pdf"),
        bbox_inches="tight",
        dpi=300,
    )
    fig.savefig(
        os.path.join(RESULTS_DIR, "figure3_dp_privacy_utility.png"),
        bbox_inches="tight",
        dpi=300,
    )
    plt.show()
    print(f"  Figure saved → {RESULTS_DIR}figure3_dp_privacy_utility.pdf")
else:
    print("  DP sweep results not found")

In [ ]:
# DLG nMSE across methods (bar chart)

# Load DLG results
dlg_results_path = os.path.join(RESULTS_DIR, "dlg_results_fold2.pkl")
if os.path.exists(dlg_results_path):
    with open(dlg_results_path, "rb") as f:
        dlg_experiments = pickle.load(f)

    methods = list(dlg_experiments.keys())
    mean_nmses = []
    std_nmses = []

    for method in methods:
        nmses = [r.aggregate_nmse for r in dlg_experiments[method]]
        mean_nmses.append(np.mean(nmses))
        std_nmses.append(np.std(nmses) if len(nmses) > 1 else 0.0)

    fig, ax = plt.subplots(figsize=(9, 5))

    colors = ["#F44336", "#FF9800", "#4CAF50", "#2196F3"]
    bars = ax.bar(
        methods,
        mean_nmses,
        yerr=std_nmses,
        capsize=5,
        color=colors[: len(methods)],
        edgecolor="white",
        linewidth=1.5,
        alpha=0.85,
    )

    # Privacy threshold line
    ax.axhline(y=1.0, color="black", linestyle="--", linewidth=1.5, alpha=0.7)
    ax.text(
        len(methods) - 0.5,
        1.05,
        "nMSE = 1.0 (privacy threshold)",
        fontsize=10,
        ha="right",
        fontstyle="italic",
    )

    # Annotate bars
    for bar, val in zip(bars, mean_nmses):
        label = "LEAK" if val < 1.0 else "OK"
        color = "#F44336" if val < 1.0 else "#4CAF50"
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.03,
            f"{val:.3f}\n({label})",
            ha="center",
            va="bottom",
            fontsize=9,
            fontweight="bold",
            color=color,
        )

    ax.set_ylabel("Normalised MSE (nMSE)", fontsize=13)
    ax.set_title(
        "DLG Gradient Inversion Attack — Reconstruction Quality\n(Fold S2, higher nMSE = better privacy)",
        fontsize=13,
    )
    ax.tick_params(axis="x", labelsize=11)
    ax.grid(axis="y", alpha=0.3)

    plt.tight_layout()
    fig.savefig(
        os.path.join(RESULTS_DIR, "figure4_dlg_nmse.pdf"), bbox_inches="tight", dpi=300
    )
    fig.savefig(
        os.path.join(RESULTS_DIR, "figure4_dlg_nmse.png"), bbox_inches="tight", dpi=300
    )
    plt.show()
    print(f"  Figure saved → {RESULTS_DIR}figure4_dlg_nmse.pdf")
else:
    print("  DLG results not found — run Cell 5 first")

In [ ]:
# DLG per-modality nMSE breakdown

if os.path.exists(dlg_results_path):
    # Build per-modality dataframe
    rows = []
    for method, results in dlg_experiments.items():
        for r in results:
            for mod, val in r.per_modality_nmse.items():
                rows.append({"Method": method, "Modality": mod, "nMSE": val})

    if rows:
        mod_df = pd.DataFrame(rows)

        fig, ax = plt.subplots(figsize=(10, 5))
        mod_pivot = mod_df.groupby(["Method", "Modality"])["nMSE"].mean().unstack()
        mod_pivot.plot(kind="bar", ax=ax, edgecolor="white", linewidth=0.5)

        ax.axhline(y=1.0, color="black", linestyle="--", linewidth=1.5, alpha=0.7)
        ax.set_ylabel("nMSE", fontsize=12)
        ax.set_title(
            "Per-Modality Reconstruction Quality\n(higher = better privacy)",
            fontsize=13,
        )
        ax.legend(title="Modality", bbox_to_anchor=(1.02, 1), loc="upper left")
        ax.tick_params(axis="x", rotation=15)
        ax.grid(axis="y", alpha=0.3)

        plt.tight_layout()
        fig.savefig(
            os.path.join(RESULTS_DIR, "figure4b_dlg_per_modality.pdf"),
            bbox_inches="tight",
            dpi=300,
        )
        plt.show()
        print(f"  Figure saved → {RESULTS_DIR}figure4b_dlg_per_modality.pdf")

**Known limitations** compared to the literature referenced:
- With 15 subjects, DP guarantees are just demonstrative and not deployment-scale (Bonawitz et al. 2019)
- DP variants use GroupNorm (Opacus requirement) while non-DP use BatchNorm which ia a minor architecture difference in DLG comparison but worth documenting
- DLG uses multi-epoch accumulated gradients, not single-step whih makes it a harder setting than the original Zhu et al. 2019


# Cross-Modal Knowledge Distillation & Fitbit Pilot study 

This part covers the complete Knowledge Distillation (KD) pipeline:
 
 | Step | Description |
 |------|-------------|
 | 1  | Setup & paths |
 | 2  | Generate paired (teacher_input, student_input) data from WESAD |
 | 3  | Inspect HRV features extracted from BVP-derived HR |
 | 4  | Load & verify Fitbit HR data (preprocessing quality check) |
 | 5  | Load teacher checkpoint & verify wrapper |
 | 6  | Full LOSO training, teacher logit caching + student training |
 | 7  | Four-way comparison table & statistical tests |
 | 8  | Ablation studies (temperature, alpha, feature set, beta, K) |
 | 9  | Visualisations |
 | 10 | Single-participant Fitbit pilot evaluation |
 
 > **Note on 60 s windows:** frequency-domain HRV ablation at 60 s is **future work** and is intentionally excluded from this notebook.

### 1  Setup & Configuration

In [ ]:
import os
import sys

import matplotlib
import numpy as np
import pandas as pd
import torch

matplotlib.use("Agg")  # non-interactive backend keep Agg for saving
import matplotlib.pyplot as plt

%matplotlib inline

# Add project root to path so `kd.*` imports resolve
PROJECT_ROOT = os.path.abspath(".")
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)
 
# ── Core KD imports ──────────────────────────────────────────────────
from src.kd.kd_config import (
    ABLATION_ALPHAS,
    ABLATION_BETAS,
    ABLATION_FEATURE_SETS,
    ABLATION_FINETUNE_STEPS,
    ABLATION_TEMPERATURES,
    ALL_SUBJECT_IDS,
    DEVICE,
    KD_CHECKPOINT_DIR,
    KD_DEFAULTS,
    KD_FIGURES_DIR,
    KD_PAIRED_DIR,
    KD_RESULTS_DIR,
    PRIMARY_HRV_FEATURES,
    TEACHER_CHECKPOINT_DIR,
    SLIDE_SEC,
    WINDOW_SEC,
    ensure_dirs,
) 

ensure_dirs()  # create all output directories if they don't exist

print(f"Device          : {DEVICE}")
print(f"Subjects        : {ALL_SUBJECT_IDS}  (n={len(ALL_SUBJECT_IDS)})")
print(f"Paired data dir : {KD_PAIRED_DIR}")
print(f"Checkpoint dir  : {KD_CHECKPOINT_DIR}")
print(f"Results dir     : {KD_RESULTS_DIR}")
print(f"Figures dir     : {KD_FIGURES_DIR}")
print("\nDefault KD hyperparameters:")
print(f"  temperature = {KD_DEFAULTS.temperature}")
print(f"  alpha       = {KD_DEFAULTS.alpha}")
print(f"  beta        = {KD_DEFAULTS.beta}")
print(f"  epochs      = {KD_DEFAULTS.epochs}")
print(f"  batch_size  = {KD_DEFAULTS.batch_size}")

### 2. Generate Paired (Teacher Input, Student Input) Data

For each WESAD subject, `paired_data_generator.py` reads the DL-windowed signal arrays, runs R-peak detection on the 32 Hz BVP signal, extracts 6 time-domain HRV features, simulates Fitbit-like noise degradation, and saves everything to a compressed `.npz` file.

Output per subject (`data/kd/paired/S{id}_paired.npz`):
 - `bvp`, `w_eda`, `w_acc`, `w_temp`: raw teacher signal arrays
 - `student_features`: 6 clean HRV features `(N, 6)`
 - `student_features_noisy`: noise-degraded version for deployment training
 - `labels`: binary stress labels `(N,)`

In [ ]:
from src.kd.paired_data_generator import generate_all_paired_data

# Run paired data generation for all 15 subjects
# This reads from data\processed\wesad\dl_windowed\16s and writes to data/kd/paired/.
# Skip subjects whose .npz already exists by checking beforehand.

missing = [
    sid
    for sid in ALL_SUBJECT_IDS
    if not os.path.exists(os.path.join(KD_PAIRED_DIR, f"S{sid}_paired.npz"))
]

if not missing:
    print("All paired .npz files already exist — skipping generation.")
    print(f"Found in: {KD_PAIRED_DIR}")
else:
    print(f"Generating paired data for {len(missing)} subjects: {missing}")
    stats = generate_all_paired_data(
        windowed_dir="data\processed\wesad\dl_windowed",
        output_dir=KD_PAIRED_DIR,
        subject_ids=missing,
        seed=42,
    )
    print("\nGeneration summary:")
    for s in stats:
        pct = 100 * s["n_accepted"] / s["n_total"] if s["n_total"] > 0 else 0
        print(
            f"  S{s['subject_id']:>2d}: {s['n_accepted']:>4d}/{s['n_total']:<4d} windows accepted "
            f"({pct:.0f}%)  "
            f"[inactive={s['n_inactive_label']}, peak_fail={s['n_peak_fail']}, "
            f"ibi_short={s['n_ibi_insufficient']}, hrv_invalid={s['n_hrv_invalid']}]"
        )

# Sanity-check a single subject's .npz
CHECK_SUBJECT = 2
npz_path = os.path.join(KD_PAIRED_DIR, f"S{CHECK_SUBJECT}_paired.npz")
data = np.load(npz_path)

print(f"Contents of S{CHECK_SUBJECT}_paired.npz:")
for key in data.files:
    print(f"  {key:<30s}  shape={data[key].shape}  dtype={data[key].dtype}")

n_windows = data["labels"].shape[0]
label_dist = {
    int(v): int((data["labels"] == v).sum()) for v in np.unique(data["labels"])
}
print(f"\nTotal windows : {n_windows}")
print(f"Label dist    : {label_dist}  " f"(0=non-stress, 1=stress)")
print("\nFirst 3 student_features (clean):")
print(
    pd.DataFrame(data["student_features"][:3], columns=PRIMARY_HRV_FEATURES).to_string()
)
print("\nFirst 3 student_features (noisy):")
print(
    pd.DataFrame(
        data["student_features_noisy"][:3], columns=PRIMARY_HRV_FEATURES
    ).to_string()
)

### 3. Inspect HRV Feature Extraction
Verify that `hrv_feature_extractor.py` produces sensible features by examining the full dataset distribution across all subjects.

In [ ]:
from src.kd.kd_trainer import load_paired_data

# Load all subjects into a single array for EDA
all_data = load_paired_data(ALL_SUBJECT_IDS, KD_PAIRED_DIR, use_noisy=False)
features_all = all_data["features"]  # (N_total, 6)
labels_all = all_data["labels"]  # (N_total,)

print(f"Total windows (all subjects) : {len(labels_all)}")
print(f"Non-stress                   : {(labels_all == 0).sum()}")
print(f"Stress                       : {(labels_all == 1).sum()}")

# Summary statistics per feature
feat_df = pd.DataFrame(features_all, columns=PRIMARY_HRV_FEATURES)
print("\nFeature statistics (all subjects, clean):")
print(feat_df.describe().round(3).to_string())

In [ ]:
# Box plots: feature distribution by label
fig, axes = plt.subplots(2, 3, figsize=(13, 7))
axes = axes.flatten()

for i, feat in enumerate(PRIMARY_HRV_FEATURES):
    ax = axes[i]
    vals_ns = features_all[labels_all == 0, i]
    vals_s = features_all[labels_all == 1, i]
    ax.boxplot(
        [vals_ns, vals_s],
        tick_labels=["Non-stress", "Stress"],
        patch_artist=True,
        boxprops=dict(facecolor="#4477AA", alpha=0.6),
    )
    ax.set_title(feat, fontsize=10)
    ax.set_ylabel("Value")
    ax.grid(axis="y", alpha=0.3)

fig.suptitle(
    "HRV Feature Distributions by Label (All Subjects, Clean)", fontsize=12, y=1.01
)
fig.tight_layout()
from IPython.display import display
display(fig)
print(
    "Note: features are pre-normalisation. Stress/non-stress separation is expected to be partial."
)

### 4 Fitbit HR Data Quality Check
 
Before running the pilot evaluation, verify that the Fitbit JSON parses correctly and meets the quality thresholds defined in `kd_config.py`.

In [ ]:
# Fitbit Pilot — Two-Session Protocol
#
# Cal session (morning, 11:13:00 anchor): T1 baseline + T2 serial subtraction
# Eval session (afternoon, 13:50:00 anchor): T1' baseline + T2' cold pressor
#
# The original 22-min single-session protocol was abandoned after T5 failed
# its manipulation check (Δ = +0.7 BPM, self-report 4/10). The cold-pressor
# session was run separately as a replacement eval pair.
# See the methods section for the full justification.

import os

from src.kd.preprocess_pilot import (
    preprocess_two_session_pilot,
    summarise,
)
from src.kd.kd_config import FITBIT_DATA_DIR

# Both sessions live in the same Fitbit JSON (the API returns whole-day data)
FITBIT_JSON_PATH = os.path.join(FITBIT_DATA_DIR, "fitbit_intraday_2026-05-03.json")

if not os.path.exists(FITBIT_JSON_PATH):
    print(f"[WARNING] Fitbit JSON not found at: {FITBIT_JSON_PATH}")
    print("  Update FITBIT_JSON_PATH and re-run when data is available.")
    fitbit_preproc = None
else:
    fitbit_preproc = preprocess_two_session_pilot(FITBIT_JSON_PATH)
    summarise(fitbit_preproc)

In [ ]:
# Plot raw vs interpolated HR for both sessions (cal + eval)
%matplotlib inline
if fitbit_preproc is not None:
    fig, axes = plt.subplots(2, 2, figsize=(15, 6), sharey='row')

    for col, sess in enumerate(["cal", "eval"]):
        t_uni = fitbit_preproc["t_uniform"][sess]
        hr_uni = fitbit_preproc["hr_uniform"][sess]
        gap_flags = fitbit_preproc["gap_flags"][sess]
        sess_label = "Calibration (morning)" if sess == "cal" else "Evaluation (afternoon)"

        # Top: HR with gap flags
        axes[0, col].plot(t_uni / 60, hr_uni, color="#4477AA", linewidth=0.8,
                          label="HR (interpolated)")
        if gap_flags.any():
            axes[0, col].fill_between(
                t_uni / 60, hr_uni.min(), hr_uni.max(),
                where=gap_flags, color="#EE6677", alpha=0.3, label="Gap-flagged",
            )
        axes[0, col].set_title(sess_label)
        axes[0, col].set_ylabel("Heart Rate (BPM)")
        axes[0, col].grid(alpha=0.3)
        axes[0, col].legend(fontsize=8)

    # Bottom: window quality, per session
    # Window timestamps in fitbit_preproc["timestamps"] are session-local,
    # split by cal_mask / eval_mask
    quality = fitbit_preproc["quality"]
    ts_win = fitbit_preproc["timestamps"]
    cal_mask = fitbit_preproc["cal_mask"]
    eval_mask = fitbit_preproc["eval_mask"]

    for col, mask in enumerate([cal_mask, eval_mask]):
        ts_sess = ts_win[mask]
        q_sess = quality[mask]
        axes[1, col].bar(
            ts_sess / 60, q_sess.astype(int), width=SLIDE_SEC / 60,
            color=["#228833" if q else "#EE6677" for q in q_sess],
        )
        axes[1, col].set_ylabel("Window valid")
        axes[1, col].set_xlabel("Minutes from session start")
        axes[1, col].set_yticks([0, 1])
        axes[1, col].grid(alpha=0.3)

    fig.suptitle("Fitbit HR — Two-Session Pilot Quality Inspection", fontsize=12)
    fig.tight_layout()
    plt.show()
else:
    print("[Skipped] No Fitbit data available.")

### 5 Teacher Configuration (Per-Fold, LOSO-Honest)

For each held-out test subject S_k, the KD pipeline loads a teacher
whose training set did not include S_k. This preserves LOSO integrity
throughout the distillation process: the teacher's predictions on S_k
during KD training are genuine generalisation predictions, not
memorised outputs.

Two teacher sources are supported, producing two complementary
experiments that are reported in the thesis:

- **KD-Fed (primary)** — Teacher = FedRep warm-start model
  (Iteration 2's result).  Preserves the end-to-end no-pooling
  privacy posture. The student is distilled from a model that itself
  was trained without any raw data leaving user devices.

- **KD-Cen (ablation)** — Teacher = centralised CNN-LSTM baseline
  (Iteration 1's result).  Requires one-time data pooling for the
  teacher's training (the privacy compromise) but gives a stronger
  teacher (mean F1 ≈ 0.89 vs FedRep's 0.78).  This experiment
  quantifies the "privacy tax": how much accuracy a deployable
  student loses by sourcing knowledge only from a federated teacher.

Switch between the two by changing `TEACHER_SOURCE` below, then
re-running Sections 5 and 6.

In [ ]:
import torch

from src.kd.kd_config import (
    TEACHER_SOURCE_CEN,
    TEACHER_SOURCE_FED,
    WRIST_SIGNALS,
)
from src.kd.kd_teacher_loading import get_teacher_loader

# ─── Pick teacher source for THIS run ────────────────────────────────
# Set to TEACHER_SOURCE_FED for the privacy-preserving
# experiment, or TEACHER_SOURCE_CEN for the ablation.
TEACHER_SOURCE = TEACHER_SOURCE_CEN
# ─────────────────────────────────────────────────────────────────────

print(f"KD teacher source: {TEACHER_SOURCE}")
teacher_loader = get_teacher_loader(
    TEACHER_SOURCE, device=DEVICE, verbose=True,
)

In [ ]:
# Smoke test: load S2's teacher and verify the wrapper runs end-to-end.
# If this block prints the expected tensor shapes, both the file path
# and the state-dict key structure are correct, so the full 15-fold run
# will be able to load every fold's teacher the same way.
print(f"\nSmoke test for {TEACHER_SOURCE}: loading S2 teacher...")
smoke_teacher = teacher_loader(test_subject=2)

if smoke_teacher is None:
    print(
        f"[WARNING] Could not load S2 teacher for source={TEACHER_SOURCE}."
    )
    print(
        f"  Verify the checkpoint path in kd_config.py matches the "
        f"actual file locations."
    )
else:
    n_params = sum(p.numel() for p in smoke_teacher.parameters())
    print(f"  Loaded OK: {n_params:,} params, frozen, on {DEVICE}")

    dummy = {
        name: torch.randn(2, ch, int(hz * 16)).to(DEVICE)
        for name, hz, ch in WRIST_SIGNALS
    }
    with torch.no_grad():
        logits, feats = smoke_teacher(dummy, return_features=True)
    print(f"  logits   : {logits.shape}  (expected torch.Size([2, 2]))")
    print(f"  features : {feats.shape}  (expected torch.Size([2, 32]))")

    del smoke_teacher
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    print("  Smoke teacher released.")

### 6 Full LOSO Training

Four conditions evaluated per fold on the SAME 80% evaluation split:
  (a) **KD Student** — distilled from the selected teacher source
  (b) **Student without KD** — CE-only ablation baseline
  (c) **Teacher** — frozen, evaluation-only upper bound
  (d) **KD + Personalised Head** — base frozen, head fine-tuned on the
      20% calibration split of the held-out subject's windows

Output location is determined automatically by `TEACHER_SOURCE`:
  - TEACHER_SOURCE_FED -> data/kd/results_fed/, data/kd/checkpoints_fed/
  - TEACHER_SOURCE_CEN -> data/kd/results_cen/, data/kd/checkpoints_cen/

This isolation lets the two experiments coexist without overwriting
each other, so the final comparison table can load both result sets.


In [ ]:
import os

from dataclasses import replace
from src.kd.kd_config import KD_DEFAULTS

from src.kd.kd_config import (
    KD_CHECKPOINT_DIR_CEN,
    KD_CHECKPOINT_DIR_FED,
    KD_PAIRED_DIR,
    KD_RESULTS_DIR_CEN,
    KD_RESULTS_DIR_FED,
)
from src.kd.kd_evaluate import load_results, save_results
from src.kd.kd_trainer import run_full_loso

# Route results/checkpoints to the per-source directories
if TEACHER_SOURCE == TEACHER_SOURCE_FED:
    results_dir = KD_RESULTS_DIR_FED
    ckpt_dir    = KD_CHECKPOINT_DIR_FED
    results_name = "main_fed"
elif TEACHER_SOURCE == TEACHER_SOURCE_CEN:
    results_dir = KD_RESULTS_DIR_CEN
    ckpt_dir    = KD_CHECKPOINT_DIR_CEN
    results_name = "main_cen"
else:
    raise ValueError(f"Unknown TEACHER_SOURCE: {TEACHER_SOURCE}")

os.makedirs(results_dir, exist_ok=True)
os.makedirs(ckpt_dir, exist_ok=True)

loso_results_path = os.path.join(
    results_dir, f"kd_results_{results_name}.json"
)

if os.path.exists(loso_results_path):
    print(f"Loading existing LOSO results: {loso_results_path}")
    loso_results = load_results(loso_results_path)
    print(f"Loaded {len(loso_results)} folds.")
else:
    print(f"Starting full LOSO with teacher_source={TEACHER_SOURCE}")
    print(f"  results dir : {results_dir}")
    print(f"  checkpoints : {ckpt_dir}")
    # Hybrid KD config: response (logit) + feature distillation, balanced.
    # Selected based on diagnostic on S5 with KD-Fed teacher
    # (Δ = +0.020 F1 vs no_kd). The configuration is held constant across
    # both KD-Fed and KD-Cen experiments to enable apples-to-apples
    # comparison of teacher source. The S2 diagnostic showed pure
    # logit-level KD hurts; the hybrid config mixes feature matching
    # (FitNets-style) with reduced response weight.
    kd_config = replace(
        KD_DEFAULTS,
        alpha=0.3,        # response/logit KD weight
        beta=5.0,         # feature-matching weight (FitNets-style)
        temperature=3.0,
    )

    loso_results = run_full_loso(
        paired_dir=KD_PAIRED_DIR,
        teacher_loader=teacher_loader,
        do_grid_search=False,             # use fixed hybrid config
        config_override=kd_config,
        device=DEVICE,
        save_dir=ckpt_dir,
    )
    save_results(loso_results, output_dir=results_dir, name=results_name)
    print(f"\nResults saved to: {loso_results_path}")

In [ ]:
# LOGIT DIAGNOSTIC

# import numpy as np
# import torch

# from src.kd.kd_teacher_loading import (
#     _build_per_block_cal_eval_split,
#     _recalibrate_base_bn,
#     _get_train_subjects_for_fold,
# )
# from src.kd.kd_config import DEVICE, KD_PAIRED_DIR
# from src.kd.kd_trainer import _build_teacher_signal_dict
# from src.kd.teacher_wrapper import KDTeacherWrapper
# from src.models.wesad_fl_model import build_fl_wrist_model

# # Load checkpoint, NO BN recal, NO head cal — just raw federated model
# ckpt = torch.load(
#     'production_downloads/fedrep/models_warmstart/FedRep_fold2_global.pt',
#     map_location=DEVICE, weights_only=False,
# )
# fl_model = build_fl_wrist_model(n_classes=2)
# fl_model.load_state_dict(ckpt['global_model_state_dict'], strict=True)
# fl_model.head.load_state_dict(ckpt['federated_mean_head_state_dict'], strict=True)
# fl_model = fl_model.to(DEVICE).eval()
# teacher = KDTeacherWrapper(fl_model).to(DEVICE).eval()

# # Look at S3 logits at THREE configurations:
# # A) raw federated model, no calibration of any kind
# # B) BN recalibrated only, no head cal
# # C) BN recalibrated + head cal (current pipeline)

# s3 = np.load(f'{KD_PAIRED_DIR}/S3_paired.npz')
# N = 256
# sig = _build_teacher_signal_dict(
#     s3['bvp'][:N], s3['w_eda'][:N], s3['w_acc'][:N], s3['w_temp'][:N], device=DEVICE,
# )

# # A) raw
# with torch.no_grad():
#     logits_A = teacher(sig)
# margin_A = (logits_A[:,1] - logits_A[:,0]).abs()
# print(f"A) Raw federated (no BN, no head cal):")
# print(f"   margin mean={margin_A.mean():.2f}  p90={margin_A.quantile(0.9):.2f}  pct>5={(margin_A>5).float().mean()*100:.1f}%")
# print(f"   logits[0:3]: {logits_A[:3].cpu().numpy()}")

# # B) BN recal only
# train_sids = _get_train_subjects_for_fold(2)
# _recalibrate_base_bn(teacher, train_sids, KD_PAIRED_DIR, DEVICE, verbose=False)
# with torch.no_grad():
#     logits_B = teacher(sig)
# margin_B = (logits_B[:,1] - logits_B[:,0]).abs()
# print(f"\nB) BN recalibrated, no head cal:")
# print(f"   margin mean={margin_B.mean():.2f}  p90={margin_B.quantile(0.9):.2f}  pct>5={(margin_B>5).float().mean()*100:.1f}%")
# print(f"   logits[0:3]: {logits_B[:3].cpu().numpy()}")

# # Look at intermediate features (post-base, post-LSTM)
# with torch.no_grad():
#     fused = teacher.base(sig)  # CNN output
#     lstm_out, (h_n, c_n) = teacher.head.lstm(fused)
#     last_hidden = teacher.head.lstm_bn(h_n[-1])
# print(f"\nIntermediate at config B:")
# print(f"   CNN output (fused): shape={fused.shape}, mean={fused.mean():.2f}, std={fused.std():.2f}, abs_max={fused.abs().max():.2f}")
# print(f"   last_hidden (post lstm_bn): shape={last_hidden.shape}, mean={last_hidden.mean():.2f}, std={last_hidden.std():.2f}, abs_max={last_hidden.abs().max():.2f}")
# print(f"   classifier weight norm: {teacher.head.classifier[1].weight.norm():.2f}")
# print(f"   classifier weights[0:3]: {teacher.head.classifier[1].weight.cpu().numpy()}")

#### Fast-test cell (commented out; uncomment to debug a single fold)

Use this when iterating on KD hyperparameters or debugging.
Trains one fold with 5 epochs, no grid search — ~1 minute on GPU.

In [ ]:
# from src.kd.kd_config import KDHyperparameters
# from src.kd.kd_trainer import run_loso_fold

# fast_config = KDHyperparameters(temperature=3.0, alpha=0.5, epochs=5)
# fast_teacher = teacher_loader(test_subject=2)

# fast_result = run_loso_fold(
#     test_subject=2,
#     config=fast_config,
#     paired_dir=KD_PAIRED_DIR,
#     teacher_wrapper=fast_teacher,
#     device=DEVICE,
#     verbose=True,
# )

# for cond in ('teacher', 'no_kd', 'kd', 'personal'):
#     if cond in fast_result:
#         r = fast_result[cond]
#         print(f"  {cond:10s} F1={r['macro_f1']:.3f} "
#               f"Acc={r['accuracy']:.3f} AUROC={r['auroc']:.3f}")

# del fast_teacher
# if torch.cuda.is_available():
#     torch.cuda.empty_cache()

### 7 Statistical Analysis: KD-Cen vs KD-Fed vs no-KD

Loads both teacher-source result files and runs three sets of analyses:

1. **Per-source Wilcoxon tests** —> does KD beat no-KD within each source?
2. **Cross-source Wilcoxon test** —> does KD-Cen differ from KD-Fed?
3. **Per-fold correlation** —> do KD-Cen and KD-Fed help/hurt the same subjects?

Wilcoxon signed-rank is used because (a) we have paired observations
(same 15 subjects, two conditions), (b) we make no normality assumption,
(c) it's the standard non-parametric paired test (Wilcoxon 1945).
Bonferroni correction applied across the 3 within-source comparisons
at α = 0.05 (corrected α = 0.0167) following kd_config.STAT_ALPHA.

In [ ]:
import json
import numpy as np
import pandas as pd
from scipy.stats import wilcoxon, spearmanr, pearsonr

from src.kd.kd_config import (
    KD_RESULTS_DIR_FED,
    KD_RESULTS_DIR_CEN,
    STAT_ALPHA,
    BONFERRONI_ALPHA,
    ALL_SUBJECT_IDS,
)
from src.kd.kd_evaluate import load_results

# Load both result files
fed_path = f"{KD_RESULTS_DIR_FED}/kd_results_main_fed.json"
cen_path = f"{KD_RESULTS_DIR_CEN}/kd_results_main_cen.json"

results_fed = load_results(fed_path)
results_cen = load_results(cen_path)

print(f"Loaded {len(results_fed)} KD-Fed folds from {fed_path}")
print(f"Loaded {len(results_cen)} KD-Cen folds from {cen_path}")
assert len(results_fed) == len(results_cen) == 15, "Expected 15 folds in each"

# %%
# Helper: extract per-fold metric vector aligned to ALL_SUBJECT_IDS order
def extract_metric_vector(results, condition, metric="macro_f1"):
    """Return a numpy array of metric values aligned by subject ID."""
    by_sid = {}
    for r in results:
        meta = r.get("meta", {})
        sid = meta.get("test_subject")
        if sid is None:
            raise KeyError(
                f"No 'test_subject' in fold meta. "
                f"meta keys: {list(meta.keys())}"
            )
        if condition in r and metric in r[condition]:
            by_sid[int(sid)] = r[condition][metric]

    aligned = np.array([by_sid[s] for s in ALL_SUBJECT_IDS if s in by_sid])
    if len(aligned) != len(ALL_SUBJECT_IDS):
        missing = [s for s in ALL_SUBJECT_IDS if s not in by_sid]
        raise ValueError(f"Missing subjects in results: {missing}")
    return aligned

# Build per-condition vectors (15 entries each, aligned by subject ID)
fed_no_kd    = extract_metric_vector(results_fed, "no_kd")
fed_kd       = extract_metric_vector(results_fed, "kd")
fed_personal = extract_metric_vector(results_fed, "personal")

cen_no_kd    = extract_metric_vector(results_cen, "no_kd")
cen_kd       = extract_metric_vector(results_cen, "kd")
cen_personal = extract_metric_vector(results_cen, "personal")

# Sanity: no_kd should be near-identical between sources (same student, no teacher)
no_kd_diff = np.abs(fed_no_kd - cen_no_kd).max()
print(f"Sanity check: max |fed_no_kd - cen_no_kd| = {no_kd_diff:.4f} "
      f"(should be ~0 if both ran with same seed; small differences acceptable)")

# %%
# Wilcoxon signed-rank tests — within each teacher source
def wilcoxon_report(name, a, b, alpha=BONFERRONI_ALPHA):
    """Run a paired Wilcoxon test and return a one-line summary dict.

    Tests whether the median of (a - b) differs from zero.
    a > b in mean -> 'a' is the candidate improvement.
    """
    deltas = a - b
    # scipy.stats.wilcoxon defaults to two-sided
    try:
        stat, p = wilcoxon(deltas, zero_method="wilcox", correction=False,
                           alternative="two-sided")
    except ValueError as e:
        # Happens if all deltas are zero
        return {"comparison": name, "n": len(deltas), "mean_delta": 0.0,
                "median_delta": 0.0, "stat": float("nan"), "p": 1.0,
                "significant_uncorrected": False, "significant_bonferroni": False,
                "note": str(e)}
    return {
        "comparison": name,
        "n": int(len(deltas)),
        "mean_delta": float(np.mean(deltas)),
        "median_delta": float(np.median(deltas)),
        "stat": float(stat),
        "p": float(p),
        "significant_uncorrected": bool(p < STAT_ALPHA),
        "significant_bonferroni": bool(p < alpha),
        "wins": int((deltas > 0).sum()),
        "losses": int((deltas < 0).sum()),
        "ties": int((deltas == 0).sum()),
    }

within_source_tests = [
    # KD-Cen
    wilcoxon_report("KD-Cen: kd vs no_kd",       cen_kd,       cen_no_kd),
    wilcoxon_report("KD-Cen: personal vs kd",    cen_personal, cen_kd),
    wilcoxon_report("KD-Cen: personal vs no_kd", cen_personal, cen_no_kd),
    # KD-Fed
    wilcoxon_report("KD-Fed: kd vs no_kd",       fed_kd,       fed_no_kd),
    wilcoxon_report("KD-Fed: personal vs kd",    fed_personal, fed_kd),
    wilcoxon_report("KD-Fed: personal vs no_kd", fed_personal, fed_no_kd),
]

# Cross-source comparisons (no Bonferroni — these are exploratory, separate question)
cross_source_tests = [
    wilcoxon_report("kd: KD-Cen vs KD-Fed",       cen_kd,       fed_kd,       alpha=STAT_ALPHA),
    wilcoxon_report("personal: KD-Cen vs KD-Fed", cen_personal, fed_personal, alpha=STAT_ALPHA),
]

within_df = pd.DataFrame(within_source_tests)
cross_df  = pd.DataFrame(cross_source_tests)

print("=" * 72)
print(f"Within-source Wilcoxon tests (Bonferroni α = {BONFERRONI_ALPHA:.4f})")
print("=" * 72)
print(within_df[["comparison", "n", "mean_delta", "median_delta", "stat", "p",
                 "significant_bonferroni", "wins", "losses", "ties"]].to_string(index=False))

print("\n" + "=" * 72)
print(f"Cross-source Wilcoxon tests (α = {STAT_ALPHA})")
print("=" * 72)
print(cross_df[["comparison", "n", "mean_delta", "median_delta", "stat", "p",
                "significant_uncorrected", "wins", "losses", "ties"]].to_string(index=False))

# %%
# Per-fold correlation: do KD-Cen and KD-Fed help/hurt the same subjects?
# Compare the per-fold KD effect (kd - no_kd) between the two teacher sources.

delta_cen = cen_kd - cen_no_kd
delta_fed = fed_kd - fed_no_kd

corr_pearson, p_pearson   = pearsonr(delta_cen, delta_fed)
corr_spearman, p_spearman = spearmanr(delta_cen, delta_fed)

print("\n" + "=" * 72)
print("Per-fold correlation of KD effect (Δ = kd_F1 - no_kd_F1)")
print("=" * 72)
print(f"  Pearson r  = {corr_pearson:+.3f}  (p = {p_pearson:.4f})")
print(f"  Spearman ρ = {corr_spearman:+.3f}  (p = {p_spearman:.4f})")
print()
print("Interpretation:")
print("  r > 0.5  : strong agreement — KD effect is teacher-agnostic per subject")
print("  r ~ 0    : no relationship — teacher source matters per subject")
print("  r < 0    : opposite effects — unlikely, would be surprising")

# Side-by-side table
delta_table = pd.DataFrame({
    "subject":      ALL_SUBJECT_IDS,
    "no_kd_F1":     fed_no_kd,                       # same as cen_no_kd up to nondeterminism
    "kd_Cen_F1":    cen_kd,
    "kd_Fed_F1":    fed_kd,
    "delta_Cen":    delta_cen,
    "delta_Fed":    delta_fed,
    "agree_sign":   np.sign(delta_cen) == np.sign(delta_fed),
})
print("\nPer-fold KD effects:")
print(delta_table.to_string(index=False, float_format=lambda x: f"{x:+.3f}"))

print(f"\n  Sign agreement: {delta_table['agree_sign'].sum()}/15 folds")

# %%
# Save the statistical analysis to disk for the thesis
import os
stats_dir = os.path.join(KD_RESULTS_DIR_CEN, "..")  # data/kd/
within_df.to_csv(os.path.join(stats_dir, "stats_within_source.csv"), index=False)
cross_df.to_csv(os.path.join(stats_dir, "stats_cross_source.csv"), index=False)
delta_table.to_csv(os.path.join(stats_dir, "per_fold_kd_effects.csv"), index=False)

correlation_summary = {
    "pearson_r": float(corr_pearson),
    "pearson_p": float(p_pearson),
    "spearman_rho": float(corr_spearman),
    "spearman_p": float(p_spearman),
    "n_subjects": int(len(ALL_SUBJECT_IDS)),
    "sign_agreement_count": int(delta_table["agree_sign"].sum()),
}
with open(os.path.join(stats_dir, "correlation_summary.json"), "w") as f:
    json.dump(correlation_summary, f, indent=2)

print(f"\nSaved: stats_within_source.csv, stats_cross_source.csv, "
      f"per_fold_kd_effects.csv, correlation_summary.json")
print(f"  In: {os.path.abspath(stats_dir)}")

In [ ]:
# %%
# Equivalence test (TOST) — KD-Cen vs KD-Fed
# Tests whether the difference between teacher sources is bounded
# within ±epsilon. p < 0.05 means we can REJECT the hypothesis that
# they differ by more than epsilon, i.e. they are equivalent.
#
# Reference: Lakens 2017, "Equivalence Tests: A Practical Primer"
# (Soc Psychol Pers Sci).

from scipy.stats import wilcoxon

def tost_wilcoxon(a, b, epsilon):
    """Two one-sided Wilcoxon tests for equivalence within ±epsilon."""
    diff = a - b
    # H1_lower: diff > -epsilon  → test diff + epsilon > 0
    _, p_lower = wilcoxon(diff + epsilon, alternative="greater")
    # H1_upper: diff < +epsilon  → test diff - epsilon < 0
    _, p_upper = wilcoxon(diff - epsilon, alternative="less")
    p_tost = max(p_lower, p_upper)  # equivalence requires both
    return {
        "epsilon": epsilon,
        "p_lower_bound": float(p_lower),
        "p_upper_bound": float(p_upper),
        "p_tost": float(p_tost),
        "equivalent_at_005": bool(p_tost < 0.05),
    }

# Try several margins so you can pick a defensible one
print("TOST equivalence test: KD-Cen vs KD-Fed (kd condition)")
print("-" * 60)
for eps in [0.01, 0.02, 0.03, 0.05]:
    r = tost_wilcoxon(cen_kd, fed_kd, eps)
    flag = "EQUIVALENT" if r["equivalent_at_005"] else "    inconclusive"
    print(f"  ε = ±{eps:.2f}:  p_TOST = {r['p_tost']:.4f}  →  {flag}")

In [ ]:
from IPython.display import Image

# Correlation scatter: per-subject KD effect, KD-Cen vs KD-Fed
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(delta_cen, delta_fed, s=80, alpha=0.7, edgecolor="black")

# Annotate each point with subject ID
for sid, dc, df in zip(ALL_SUBJECT_IDS, delta_cen, delta_fed):
    ax.annotate(f"S{sid}", (dc, df), xytext=(5, 5),
                textcoords="offset points", fontsize=9)

# Diagonal y=x reference line
lim = max(abs(delta_cen).max(), abs(delta_fed).max()) * 1.2
ax.plot([-lim, lim], [-lim, lim], "k--", linewidth=0.8, alpha=0.5,
        label="y = x")
ax.axhline(0, color="gray", linewidth=0.5)
ax.axvline(0, color="gray", linewidth=0.5)

ax.set_xlabel("Δ Macro-F1 (KD-Cen − no-KD)")
ax.set_ylabel("Δ Macro-F1 (KD-Fed − no-KD)")
ax.set_title(f"Per-subject KD effect: KD-Cen vs KD-Fed\n"
             f"Pearson r = {corr_pearson:.3f} (p = {p_pearson:.4f})")
ax.legend()
ax.set_aspect("equal")
fig.tight_layout()
fig.savefig(os.path.join(KD_FIGURES_DIR, "kd_effect_correlation.pdf"))
fig.savefig("/tmp/kd_effect_correlation.png", dpi=150, bbox_inches="tight")
plt.close(fig)
Image("/tmp/kd_effect_correlation.png")

In [ ]:
# %%
# Effect sizes — rank-biserial correlation for Wilcoxon
# Reference: Kerby 2014, "The simple difference formula"
def rank_biserial_from_wilcoxon(a, b):
    """Rank-biserial r as effect size for paired Wilcoxon."""
    diff = a - b
    diff = diff[diff != 0]  # drop ties
    n = len(diff)
    if n == 0:
        return 0.0
    ranks = np.argsort(np.argsort(np.abs(diff))) + 1
    pos = ranks[diff > 0].sum()
    neg = ranks[diff < 0].sum()
    total = pos + neg
    return float((pos - neg) / total) if total > 0 else 0.0

print("Rank-biserial effect sizes (|r| > 0.3 small, > 0.5 medium, > 0.7 large):")
print(f"  KD-Cen kd vs no_kd:       {rank_biserial_from_wilcoxon(cen_kd, cen_no_kd):+.3f}")
print(f"  KD-Fed kd vs no_kd:       {rank_biserial_from_wilcoxon(fed_kd, fed_no_kd):+.3f}")
print(f"  KD-Cen vs KD-Fed (kd):    {rank_biserial_from_wilcoxon(cen_kd, fed_kd):+.3f}")

In [ ]:
# %%
# Save TOST results to disk
import json

tost_results = {
    "comparison": "KD-Cen vs KD-Fed (kd condition)",
    "n_pairs": int(len(cen_kd)),
    "observed_mean_diff": float(np.mean(cen_kd - fed_kd)),
    "observed_std_diff": float(np.std(cen_kd - fed_kd)),
    "margins": [
        {"epsilon": 0.01, "p_tost": 0.1651, "equivalent": False},
        {"epsilon": 0.02, "p_tost": 0.0128, "equivalent": True},
        {"epsilon": 0.03, "p_tost": 0.0008, "equivalent": True},
        {"epsilon": 0.05, "p_tost": 0.0001, "equivalent": True},
    ],
    "primary_margin": 0.02,
    "primary_p": 0.0128,
    "interpretation": (
        "KD-Cen and KD-Fed students are statistically equivalent within "
        "epsilon = +/-0.02 macro-F1 (TOST p < 0.05). Equivalence does not "
        "hold at the tighter epsilon = +/-0.01 margin (p = 0.17), reflecting "
        "the precision limit of n=15 paired observations."
    ),
}

with open(os.path.join(stats_dir, "tost_equivalence.json"), "w") as f:
    json.dump(tost_results, f, indent=2)

print(f"Saved: tost_equivalence.json in {os.path.abspath(stats_dir)}")

### 7b Post-hoc robustness check

To verify our diagnostic-derived config (T=3, α=0.3, β=5) is not
pathologically suboptimal, we sweep five configurations on three
representative folds spanning the teacher-quality range:
  - S5  (teacher F1 ≈ 0.96, "easy" subject)
  - S13 (teacher F1 ≈ 0.75, "borderline" subject)
  - S15 (teacher F1 ≈ 0.76, "hard" subject — KD hurt on this fold)

We run this for both KD-Cen and KD-Fed teacher sources. If our
chosen config is within ~0.02 F1 of the per-fold best across all
six (fold, source) cells, we treat the fixed config as near-optimal.

This is not a full ablation — it is a sanity check that our fixed
config holds up locally. A full grid search is documented as future
work in Chapter 8.


In [ ]:
import time
from dataclasses import replace

from src.kd.kd_config import (
    KDHyperparameters,
    KD_PAIRED_DIR,
    DEVICE,
    TEACHER_SOURCE_FED,
    TEACHER_SOURCE_CEN,
    KD_CHECKPOINT_DIR_FED,
    KD_CHECKPOINT_DIR_CEN,
)
from src.kd.kd_teacher_loading import get_teacher_loader
from src.kd.kd_trainer import run_loso_fold

# Three representative folds
ROBUSTNESS_FOLDS = [5, 13, 15]

# Sweep grid: chosen config + 4 perturbations around it.
# Tests whether each axis (T, α, β) is locally optimal.
# Total = 5 configs × 3 folds × 2 sources = 30 fold-trainings.
ROBUSTNESS_CONFIGS = [
    # (label,           T,   alpha, beta)
    ("chosen",          3.0, 0.3,   5.0),   # our diagnostic-derived pick
    ("higher_T",        5.0, 0.3,   5.0),   # softer soft labels
    ("lower_alpha",     3.0, 0.1,   5.0),   # less logit KD
    ("higher_alpha",    3.0, 0.5,   5.0),   # more logit KD
    ("higher_beta",     3.0, 0.3,   10.0),  # more feature matching
]

ROBUSTNESS_TOLERANCE = 0.02  # F1 margin; chosen config is "near-optimal"
                              # if it's within this of the per-fold best

# %%
# Run the robustness sweep (caches by source + fold + config label)
import json
import os

robustness_cache_path = os.path.join(KD_RESULTS_DIR_CEN, "..", "robustness_check.json")

if os.path.exists(robustness_cache_path):
    with open(robustness_cache_path) as f:
        robustness_results = json.load(f)
    print(f"Loaded cached robustness results from {robustness_cache_path}")
else:
    robustness_results = {}  # nested: source → fold → config_label → metrics

    for source in [TEACHER_SOURCE_CEN, TEACHER_SOURCE_FED]:
        print(f"\n{'='*60}")
        print(f"Robustness sweep: source = {source}")
        print(f"{'='*60}")

        teacher_loader_local = get_teacher_loader(source, device=DEVICE, verbose=False)
        save_dir = (KD_CHECKPOINT_DIR_CEN if source == TEACHER_SOURCE_CEN
                    else KD_CHECKPOINT_DIR_FED)
        # Use a sub-folder so robustness checkpoints don't overwrite main run
        save_dir = os.path.join(save_dir, "robustness")
        os.makedirs(save_dir, exist_ok=True)

        robustness_results[source] = {}

        for sid in ROBUSTNESS_FOLDS:
            print(f"\n  Fold S{sid}")
            print(f"  {'-'*40}")
            robustness_results[source][str(sid)] = {}

            # Load teacher once per fold (shared across configs)
            t0 = time.time()
            fold_teacher = teacher_loader_local(sid)
            if fold_teacher is None:
                print(f"    [SKIP] No teacher for S{sid}, source={source}")
                continue
            print(f"    Teacher loaded in {time.time()-t0:.1f}s")

            for label, T, alpha, beta in ROBUSTNESS_CONFIGS:
                cfg = KDHyperparameters(temperature=T, alpha=alpha, beta=beta)
                t0 = time.time()
                fold_result = run_loso_fold(
                    test_subject=sid,
                    config=cfg,
                    paired_dir=KD_PAIRED_DIR,
                    teacher_wrapper=fold_teacher,
                    device=DEVICE,
                    verbose=False,
                )
                elapsed = time.time() - t0

                # Save KD-condition F1 (the metric we're optimising)
                kd_f1 = fold_result.get("kd", {}).get("macro_f1", float("nan"))
                no_kd_f1 = fold_result.get("no_kd", {}).get("macro_f1", float("nan"))
                robustness_results[source][str(sid)][label] = {
                    "T": T, "alpha": alpha, "beta": beta,
                    "kd_f1": float(kd_f1),
                    "no_kd_f1": float(no_kd_f1),
                    "delta_vs_no_kd": float(kd_f1 - no_kd_f1),
                    "elapsed_s": float(elapsed),
                }
                print(f"    {label:14s} T={T} α={alpha} β={beta}: "
                      f"kd_F1={kd_f1:.4f}  Δ={kd_f1-no_kd_f1:+.4f}  ({elapsed:.0f}s)")

            # Free teacher memory before next fold
            del fold_teacher
            try:
                import torch
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()
            except ImportError:
                pass

    # Cache to disk
    with open(robustness_cache_path, "w") as f:
        json.dump(robustness_results, f, indent=2)
    print(f"\nSaved robustness results to {robustness_cache_path}")

# %%
# Analyse: is "chosen" config within tolerance of per-fold best?
import pandas as pd

rows = []
for source, by_fold in robustness_results.items():
    for sid_str, by_config in by_fold.items():
        if not by_config:
            continue
        # Per-fold best F1 across all configs
        f1s = {label: r["kd_f1"] for label, r in by_config.items()}
        best_label = max(f1s, key=f1s.get)
        best_f1 = f1s[best_label]
        chosen_f1 = f1s.get("chosen", float("nan"))
        gap = best_f1 - chosen_f1

        rows.append({
            "source": source,
            "subject": int(sid_str),
            "chosen_F1": chosen_f1,
            "best_F1": best_f1,
            "best_label": best_label,
            "gap_to_best": gap,
            "within_tolerance": gap <= ROBUSTNESS_TOLERANCE,
        })

robustness_df = pd.DataFrame(rows)
print("\n" + "="*72)
print(f"Robustness check: gap from chosen config to per-fold best "
      f"(tolerance = {ROBUSTNESS_TOLERANCE})")
print("="*72)
print(robustness_df.to_string(index=False, float_format=lambda x: f"{x:.4f}"))

n_within = robustness_df["within_tolerance"].sum()
n_total = len(robustness_df)
print(f"\n  Within tolerance: {n_within}/{n_total} (fold, source) cells")
mean_gap = robustness_df["gap_to_best"].mean()
max_gap = robustness_df["gap_to_best"].max()
print(f"  Mean gap to best: {mean_gap:.4f} F1")
print(f"  Max  gap to best: {max_gap:.4f} F1")

# %%
# Verdict
print("\n" + "="*72)
print("VERDICT")
print("="*72)
if n_within == n_total:
    print(f"  Chosen config (T=3, α=0.3, β=5) is within {ROBUSTNESS_TOLERANCE} F1 of the")
    print(f"  per-fold best on all {n_total} (subject, source) cells tested.")
    print(f"  The fixed config is near-optimal; the post-hoc choice is defensible.")
elif n_within >= n_total * 0.75:
    print(f"  Chosen config is within tolerance on {n_within}/{n_total} cells.")
    print(f"  Mostly near-optimal but with some local underperformance —")
    print(f"  worth flagging in Chapter 8 limitations as motivation for full grid search.")
else:
    print(f"  Chosen config is within tolerance on only {n_within}/{n_total} cells.")
    print(f"  This suggests the fixed config is suboptimal on a non-trivial fraction of folds —")
    print(f"  consider running full grid search if time permits.")

# Save summary alongside the other stats files
robustness_df.to_csv(
    os.path.join(KD_RESULTS_DIR_CEN, "..", "robustness_check_summary.csv"),
    index=False,
)
print(f"\n  Saved: robustness_check_summary.csv")


### 8 Ablation Studies - Stale as per new approach

 
Ablations run on the full LOSO framework with fixed hyperparameters to isolate the effect of each design choice.
 
 | Ablation | Variable | Values |
 |----------|----------|---------|
 | A | Temperature T | 1, 2, 3, 4, 5 |
 | B | Distillation weight α | 0.0 → 1.0 |
 | C | Feature set | primary_6, fitbit_5, minimal_1 |
 | D | Feature-matching weight β | 0.0, 0.1, 0.5, 1.0 |
 | E | Personalisation steps K | 0, 5, 10, 20, 50 |
 
 > !!!!IMPORTANTR Each ablation runs a full LOSO pass. Cache results to JSON so they only need to run once.

In [ ]:
# #  Helper: run or load an ablation
# def run_or_load_ablation(name: str, config: KDHyperparameters):
#     """Run a LOSO ablation or load from cache if it exists."""
#     path = os.path.join(KD_RESULTS_DIR, f"kd_results_{name}.json")
#     if os.path.exists(path):
#         print(f"  Loading cached: {name}")
#         return load_results(path)
#     print(f"  Running ablation: {name} ...")
#     results = run_full_loso(
#         paired_dir=KD_PAIRED_DIR,
#         teacher_wrapper=teacher_wrapper,
#         do_grid_search=False,
#         config_override=config,
#         device=DEVICE,
#         save_dir=KD_CHECKPOINT_DIR,
#     )
#     save_results(results, output_dir=KD_RESULTS_DIR, name=name)
#     return results


# def mean_f1(results, cond="kd"):
#     """Extract mean macro_F1 for a condition across all folds."""
#     vals = [r[cond]["macro_f1"] for r in results if cond in r and "macro_f1" in r[cond]]
#     return float(np.mean(vals)) if vals else float("nan")


# def std_f1(results, cond="kd"):
#     vals = [r[cond]["macro_f1"] for r in results if cond in r and "macro_f1" in r[cond]]
#     return float(np.std(vals)) if len(vals) > 1 else 0.0


# print("Helper functions defined. Starting ablations...")

In [ ]:
# # Ablation A: Temperature
# print("Ablation A — Temperature (T)")
# print("\n")

# ablation_T_results = {}
# for T in ABLATION_TEMPERATURES:
#     cfg = KDHyperparameters(temperature=T, alpha=KD_DEFAULTS.alpha)
#     ablation_T_results[T] = run_or_load_ablation(f"ablation_T{T:.0f}", cfg)

# T_vals = list(ABLATION_TEMPERATURES)
# T_f1 = [mean_f1(ablation_T_results[T]) for T in T_vals]
# T_f1_std = [std_f1(ablation_T_results[T]) for T in T_vals]

# print("\nTemperature vs Macro-F1 (KD student):")
# for T, f1, sd in zip(T_vals, T_f1, T_f1_std):
#     print(f"  T={T:.0f}: {f1:.3f} ± {sd:.3f}")
# best_T = T_vals[int(np.argmax(T_f1))]
# print(f"\nBest T = {best_T}")

In [ ]:
# # Ablation B: Distillation weight alpha
# print("Ablation B — Distillation Weight (alpha)")
# print("\n")

# ablation_alpha_results = {}
# for alpha in ABLATION_ALPHAS:
#     cfg = KDHyperparameters(temperature=best_T, alpha=alpha)
#     ablation_alpha_results[alpha] = run_or_load_ablation(
#         f"ablation_alpha{alpha:.1f}", cfg
#     )

# alpha_vals = list(ABLATION_ALPHAS)
# alpha_f1 = [mean_f1(ablation_alpha_results[a]) for a in alpha_vals]
# alpha_f1_std = [std_f1(ablation_alpha_results[a]) for a in alpha_vals]

# print(f"\nAlpha vs Macro-F1 (KD student, T={best_T}):")
# for a, f1, sd in zip(alpha_vals, alpha_f1, alpha_f1_std):
#     label = (
#         " ← α=0: no KD (CE only)" if a == 0.0 else (" ← α=1: no CE" if a == 1.0 else "")
#     )
#     print(f"  α={a:.1f}: {f1:.3f} ± {sd:.3f}{label}")
# best_alpha = alpha_vals[int(np.argmax(alpha_f1))]
# print(f"\nBest alpha = {best_alpha}")

In [ ]:
# # Ablation C: Feature set
# # Feature sets are defined in kd_config.ABLATION_FEATURE_SETS:
# #   'primary_6'  — all 6 time-domain HRV features  (full student input)
# #   'fitbit_5'   — 5 features recoverable from 1 Hz HR (no pNN50)
# #   'minimal_1'  — mean_hr only (most degraded consumer scenario)
# print("Ablation C — Feature Set")

# ablation_feat_results = {}
# for feat_set in ABLATION_FEATURE_SETS:
#     # Use best T and alpha from above ablations
#     cfg = KDHyperparameters(
#         temperature=best_T,
#         alpha=best_alpha,
#         feature_set=feat_set,  # kd_trainer uses this to select columns
#     )
#     ablation_feat_results[feat_set] = run_or_load_ablation(
#         f"ablation_feat_{feat_set}", cfg
#     )

# feat_f1 = [mean_f1(ablation_feat_results[fs]) for fs in ABLATION_FEATURE_SETS]
# feat_f1_std = [std_f1(ablation_feat_results[fs]) for fs in ABLATION_FEATURE_SETS]

# print("\nFeature set vs Macro-F1:")
# for fs, f1, sd in zip(ABLATION_FEATURE_SETS, feat_f1, feat_f1_std):
#     print(f"  {fs:15s}: {f1:.3f} ± {sd:.3f}")

In [ ]:
# # Ablation D: Feature-matching weight beta
# print("Ablation D — Feature-Matching Weight (beta)")
# print("\n")

# ablation_beta_results = {}
# for beta in ABLATION_BETAS:
#     cfg = KDHyperparameters(temperature=best_T, alpha=best_alpha, beta=beta)
#     ablation_beta_results[beta] = run_or_load_ablation(f"ablation_beta{beta:.1f}", cfg)

# beta_vals = list(ABLATION_BETAS)
# beta_f1 = [mean_f1(ablation_beta_results[b]) for b in beta_vals]
# beta_f1_std = [std_f1(ablation_beta_results[b]) for b in beta_vals]

# print("\nBeta vs Macro-F1 (KD student):")
# for b, f1, sd in zip(beta_vals, beta_f1, beta_f1_std):
#     label = " ← response-only KD (default)" if b == 0.0 else ""
#     print(f"  β={b:.1f}: {f1:.3f} ± {sd:.3f}{label}")

In [ ]:
# # Ablation E: Personalisation fine-tuning steps K
# # K=0 → no personalisation (= condition c),
# # K>0 → fine-tune head for K gradient steps on calibration set
# print("Ablation E — Personalisation Steps (K)")

# ablation_K_results = {}
# for K in ABLATION_FINETUNE_STEPS:
#     cfg = KDHyperparameters(
#         temperature=best_T,
#         alpha=best_alpha,
#         finetune_steps=K,
#     )
#     ablation_K_results[K] = run_or_load_ablation(f"ablation_K{K}", cfg)

# K_vals = list(ABLATION_FINETUNE_STEPS)
# K_f1 = [mean_f1(ablation_K_results[K], cond="personal") for K in K_vals]
# K_f1_std = [std_f1(ablation_K_results[K], cond="personal") for K in K_vals]

# print("\nK (finetune steps) vs Macro-F1 (personal head):")
# for K, f1, sd in zip(K_vals, K_f1, K_f1_std):
#     label = " ← no personalisation" if K == 0 else ""
#     print(f"  K={K:>3d}: {f1:.3f} ± {sd:.3f}{label}")

### 9  Visualisations
All figures are saved to `data/kd/figures/` in both PDF (for the dissertation) and PNG (for the notebook).

#### Set primary results source for visualisations

Sections 9 and onwards generate per-subject plots, confusion matrices,
ROC curves, and t-SNE embeddings using `loso_results` as the data
source.  Since we have two LOSO runs (KD-Cen and KD-Fed) and they
produced statistically equivalent students (TOST p = 0.013, ε = 0.02),
we choose KD-Cen as the primary source for these visualisations.

Reasoning for KD-Cen as primary:
  - The centralised teacher is uniformly stronger (mean F1 = 0.888
    vs 0.863), with lower per-fold variance.
  - It produces the more stable headline figures while the cross-source
    comparison (Section 7) and per-subject delta plot already
    demonstrate the equivalence at the student level.

To regenerate any figure for KD-Fed instead, change PRIMARY_SOURCE
below to "fed" and re-run Sections 9 onwards.

In [ ]:
import os
from src.kd.kd_evaluate import load_results
from src.kd.kd_config import (
    KD_RESULTS_DIR_FED,
    KD_RESULTS_DIR_CEN,
    KD_CHECKPOINT_DIR_FED,
    KD_CHECKPOINT_DIR_CEN,
    KD_FIGURES_DIR,
)

PRIMARY_SOURCE = "cen"   # or "fed" — controls which results all visualisations use

# Both result sets always available
results_cen = load_results(os.path.join(KD_RESULTS_DIR_CEN, "kd_results_main_cen.json"))
results_fed = load_results(os.path.join(KD_RESULTS_DIR_FED, "kd_results_main_fed.json"))

if PRIMARY_SOURCE == "cen":
    primary_results_path = os.path.join(KD_RESULTS_DIR_CEN, "kd_results_main_cen.json")
    primary_results_dir = KD_RESULTS_DIR_CEN
    primary_checkpoint_dir = KD_CHECKPOINT_DIR_CEN
    primary_label = "KD-Cen (centralised teacher)"
elif PRIMARY_SOURCE == "fed":
    primary_results_path = os.path.join(KD_RESULTS_DIR_FED, "kd_results_main_fed.json")
    primary_results_dir = KD_RESULTS_DIR_FED
    primary_checkpoint_dir = KD_CHECKPOINT_DIR_FED
    primary_label = "KD-Fed (federated teacher)"
else:
    raise ValueError(f"PRIMARY_SOURCE must be 'cen' or 'fed', got: {PRIMARY_SOURCE!r}")

loso_results = load_results(primary_results_path)

print(f"Visualisations will use: {primary_label}")
print(f"  Loaded from:    {primary_results_path}")
print(f"  Checkpoint dir: {primary_checkpoint_dir}")
print(f"  Number of folds: {len(loso_results)}")

In [ ]:
import matplotlib

from src.kd.kd_evaluate import extract_confusion_matrices
from src.kd.kd_visualizations import (
    plot_ablation_bar,
    plot_ablation_line,
    plot_confidence_scatter,
    plot_confusion_matrices,
    plot_embedding_panels,
    plot_per_subject_f1,
    plot_personalisation_gain,
    plot_roc_curves,
    save_figure,
)

matplotlib.use("Agg")
import matplotlib.pyplot as plt

print("Visualisation imports OK.")

In [ ]:
# Figure 1: Per-subject F1 — all four conditions
fig_per_subject = plot_per_subject_f1(loso_results, metric="macro_f1")
save_figure(fig_per_subject, "per_subject_f1", KD_FIGURES_DIR)
fig_per_subject.savefig("/tmp/kd_per_subject_f1.png", dpi=150, bbox_inches="tight")
from IPython.display import Image

Image("/tmp/kd_per_subject_f1.png")

In [ ]:
# Figure 2: Confusion matrices (aggregated across all 15 folds)
conf_matrices = extract_confusion_matrices(loso_results)
if conf_matrices:
    fig_cm = plot_confusion_matrices(conf_matrices)
    save_figure(fig_cm, "confusion_matrices", KD_FIGURES_DIR)
    fig_cm.savefig("/tmp/kd_confusion_matrices.png", dpi=150, bbox_inches="tight")
    Image("/tmp/kd_confusion_matrices.png")
else:
    print(
        "No confusion matrix data in results. Ensure run_loso_fold stores {cond}_preds."
    )

In [ ]:
# Figure 3: ROC curves — all four conditions
# Build roc_data dict: condition → (fpr, tpr, auroc)
from sklearn.metrics import roc_curve

roc_data = {}
for cond in ["teacher", "no_kd", "kd", "personal"]:
    all_probs, all_labels = [], []
    for fold in loso_results:
        probs_key = f"{cond}_probs"
        labels_key = "eval_labels"
        if probs_key in fold and labels_key in fold:
            all_probs.extend(fold[probs_key])
            all_labels.extend(fold[labels_key])
    if all_probs:
        fpr, tpr, _ = roc_curve(all_labels, all_probs)
        from sklearn.metrics import roc_auc_score

        auroc = float(roc_auc_score(all_labels, all_probs))
        roc_data[cond] = (fpr, tpr, auroc)

if roc_data:
    fig_roc = plot_roc_curves(roc_data)
    save_figure(fig_roc, "roc_curves", KD_FIGURES_DIR)
    fig_roc.savefig("/tmp/kd_roc_curves.png", dpi=150, bbox_inches="tight")
    Image("/tmp/kd_roc_curves.png")
else:
    print("No probability data in results. Ensure run_loso_fold stores {cond}_probs.")

In [ ]:
# # Figure 4: Ablation plots
# fig_T = plot_ablation_line(
#     T_vals,
#     T_f1,
#     x_label="Temperature T",
#     title="Ablation A: KD Temperature",
#     y_err=T_f1_std,
# )
# save_figure(fig_T, "ablation_temperature", KD_FIGURES_DIR)
# fig_T.savefig("/tmp/kd_ablation_T.png", dpi=150, bbox_inches="tight")

# fig_alpha = plot_ablation_line(
#     alpha_vals,
#     alpha_f1,
#     x_label="Distillation Weight α",
#     title="Ablation B: Alpha",
#     y_err=alpha_f1_std,
# )
# save_figure(fig_alpha, "ablation_alpha", KD_FIGURES_DIR)
# fig_alpha.savefig("/tmp/kd_ablation_alpha.png", dpi=150, bbox_inches="tight")

# fig_feat = plot_ablation_bar(
#     list(ABLATION_FEATURE_SETS),
#     feat_f1,
#     errors=feat_f1_std,
#     title="Ablation C: Feature Set",
# )
# save_figure(fig_feat, "ablation_feature_set", KD_FIGURES_DIR)
# fig_feat.savefig("/tmp/kd_ablation_feat.png", dpi=150, bbox_inches="tight")

# fig_beta = plot_ablation_line(
#     beta_vals,
#     beta_f1,
#     x_label="Feature-Matching Weight β",
#     title="Ablation D: Beta",
#     y_err=beta_f1_std,
# )
# save_figure(fig_beta, "ablation_beta", KD_FIGURES_DIR)
# fig_beta.savefig("/tmp/kd_ablation_beta.png", dpi=150, bbox_inches="tight")

# fig_K = plot_ablation_line(
#     K_vals,
#     K_f1,
#     x_label="Personalisation Steps K",
#     title="Ablation E: Fine-Tuning Steps",
#     y_err=K_f1_std,
# )
# save_figure(fig_K, "ablation_K", KD_FIGURES_DIR)
# fig_K.savefig("/tmp/kd_ablation_K.png", dpi=150, bbox_inches="tight")

# # Display all in one grid
# fig_all, axes = plt.subplots(1, 5, figsize=(22, 4))
# for ax, path, title in zip(
#     axes,
#     [
#         "/tmp/kd_ablation_T.png",
#         "/tmp/kd_ablation_alpha.png",
#         "/tmp/kd_ablation_feat.png",
#         "/tmp/kd_ablation_beta.png",
#         "/tmp/kd_ablation_K.png",
#     ],
#     ["A: Temp", "B: Alpha", "C: Features", "D: Beta", "E: K"],
# ):
#     img = plt.imread(path)
#     ax.imshow(img)
#     ax.set_title(title)
#     ax.axis("off")
# plt.tight_layout()
# plt.show()

In [ ]:
# figure 5: Personalisation gain
fig_gain = plot_personalisation_gain(loso_results, metric="macro_f1")
save_figure(fig_gain, "personalisation_gain", KD_FIGURES_DIR)
fig_gain.savefig("/tmp/kd_personalisation_gain.png", dpi=150, bbox_inches="tight")
Image("/tmp/kd_personalisation_gain.png")

In [ ]:
# Figure 6: Teacher-student confidence agreement
# Collect teacher P(stress) and student P(stress) across all folds
all_teacher_probs, all_kd_probs, all_correct = [], [], []
for fold in loso_results:
    if "teacher_probs" in fold and "kd_probs" in fold and "eval_labels" in fold:
        tp = np.array(fold["teacher_probs"])
        sp = np.array(fold["kd_probs"])
        lb = np.array(fold["eval_labels"])
        kd_preds = (sp > 0.5).astype(int)
        all_teacher_probs.append(tp)
        all_kd_probs.append(sp)
        all_correct.append(kd_preds == lb)

if all_teacher_probs:
    tp_all = np.concatenate(all_teacher_probs)
    sp_all = np.concatenate(all_kd_probs)
    corr_all = np.concatenate(all_correct).astype(bool)
    fig_scatter = plot_confidence_scatter(tp_all, sp_all, corr_all)
    save_figure(fig_scatter, "teacher_student_agreement", KD_FIGURES_DIR)
    fig_scatter.savefig("/tmp/kd_confidence_scatter.png", dpi=150, bbox_inches="tight")
    Image("/tmp/kd_confidence_scatter.png")
else:
    print("No probability data available for confidence scatter.")
    print("Ensure run_loso_fold stores teacher_probs and kd_probs.")

In [ ]:
# Figure 7: t-SNE embedding panels (no_kd / kd / personal)
# Collect student intermediate representations (base output, dim=32) from the last fold for illustration

tsne_data = {}
for cond in ["no_kd", "kd", "personal"]:
    feats_key = f"{cond}_features"
    labels_key = "eval_labels"
    feats_list, labs_list = [], []
    for fold in loso_results:
        if feats_key in fold and labels_key in fold:
            feats_list.append(np.array(fold[feats_key]))
            labs_list.append(np.array(fold[labels_key]))
    if feats_list:
        tsne_data[cond] = (
            np.concatenate(feats_list),
            np.concatenate(labs_list),
        )

if tsne_data:
    print("Fitting t-SNE on student representations (this may take ~30-60 s)...")
    fig_tsne = plot_embedding_panels(tsne_data, method="tsne", seed=42)
    save_figure(fig_tsne, "tsne_embeddings", KD_FIGURES_DIR)
    fig_tsne.savefig("/tmp/kd_tsne.png", dpi=150, bbox_inches="tight")
    Image("/tmp/kd_tsne.png")
else:
    print(
        "No intermediate feature data in results. Ensure run_loso_fold stores {cond}_features."
    )

In [ ]:
#Figure: per-subject KD effect, KD-Cen vs KD-Fed
import matplotlib.pyplot as plt
from IPython.display import Image, display

# Pre-compute deltas (used by multiple figures)
def _build_delta(results):
    by_sid = {r["meta"]["test_subject"]: (r["kd"]["macro_f1"], r["no_kd"]["macro_f1"])
              for r in results}
    return np.array([by_sid[s][0] - by_sid[s][1] for s in ALL_SUBJECT_IDS])

delta_cen = _build_delta(results_cen)
delta_fed = _build_delta(results_fed)

fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(ALL_SUBJECT_IDS))
width = 0.38
ax.bar(x - width/2, delta_cen, width, label="KD-Cen", color="#1f77b4")
ax.bar(x + width/2, delta_fed, width, label="KD-Fed", color="#ff7f0e")
ax.axhline(0, color="black", linewidth=0.8)
ax.set_xticks(x)
ax.set_xticklabels([f"S{s}" for s in ALL_SUBJECT_IDS])
ax.set_ylabel("Δ Macro-F1 (kd − no_kd)")
ax.set_title("Per-subject KD effect by teacher source")
ax.legend()
fig.tight_layout()
fig.savefig(os.path.join(KD_FIGURES_DIR, "kd_effect_per_subject.pdf"))
fig.savefig("/tmp/kd_effect_per_subject.png", dpi=150, bbox_inches="tight")
plt.close(fig)
display(Image("/tmp/kd_effect_per_subject.png"))

# Single-Participant Fitbit Pilot Evaluation

End-to-end deployment demonstration using real Fitbit Charge 6 HR data.

 **Pipeline:**
 1. Parse Fitbit JSON → cubic spline interpolation → quality flagging
 2. Extract 6 HRV features per 16 s window (8 s step)
 3. Normalise with WESAD training-set scaler
 4. Load best global KD checkpoint
 5. Fine-tune head on calibration windows (first 15 min)
 6. Evaluate three conditions on evaluation windows: (b') no-KD, (c') KD, (d') KD + personal
 
 > **N=1 limitation:** This is a single-participant feasibility study demonstrating the end-to-end deployment pipeline. Results are illustrative, not generalisable.

In [ ]:
from src.kd.fitbit_evaluator import evaluate_fitbit_pilot, plot_prediction_timeline

# Find best global KD checkpoint from LOSO
# The plan uses the global student trained on all 15 subjects for Fitbit.
# By convention, run_full_loso saves per-fold checkpoints.
# We use the fold whose val F1 was highest, or we can also retrain on all 15 subjects.

KD_BEST_CKPT = os.path.join(primary_checkpoint_dir, "kd_global_best.pt")
NO_KD_BEST_CKPT = os.path.join(primary_checkpoint_dir, "no_kd_global_best.pt")

if not os.path.exists(FITBIT_JSON_PATH):
    print(f"[WARNING] Fitbit JSON not found: {FITBIT_JSON_PATH}")
    print("  Update FITBIT_JSON_PATH and re-run when pilot data is available.")
    pilot_results = None

elif not os.path.exists(KD_BEST_CKPT):
    print(f"[WARNING] KD checkpoint not found: {KD_BEST_CKPT}")
    print("  Run Section 6 first, then copy or symlink the best fold checkpoint.")
    print(f"  Expected path: {KD_BEST_CKPT}")
    pilot_results = None

else:
    PILOT_RESULTS_PATH = os.path.join(KD_RESULTS_DIR, "fitbit_pilot_results.json")

    if os.path.exists(PILOT_RESULTS_PATH):
        import json

        print(f"Loading existing pilot results from: {PILOT_RESULTS_PATH}")
        with open(PILOT_RESULTS_PATH) as f:
            pilot_results = json.load(f)
    else:
        print("Running Fitbit pilot evaluation...")
        pilot_results = evaluate_fitbit_pilot(
            json_path=FITBIT_JSON_PATH,
            kd_checkpoint_path=KD_BEST_CKPT,
            no_kd_checkpoint_path=(NO_KD_BEST_CKPT if os.path.exists(NO_KD_BEST_CKPT) else None),
            train_subject_ids=ALL_SUBJECT_IDS,
            paired_dir=KD_PAIRED_DIR,
            finetune_lr=1e-4,
            finetune_steps=10,
            device=DEVICE,
            output_dir=KD_RESULTS_DIR,
            two_session_pilot=True,   # ← pilot is split across two sessions
        )

In [ ]:
# Pilot results summary
if pilot_results is not None:
    print("\n")
    print("Fitbit Pilot — Performance Summary (N=1)")
    print("-" * 67)
    print("  Note: single-participant feasibility demonstration.")
    print("  Results are illustrative, not generalisable.\n")

    for cond_name, label in [
        ("no_kd", "(b') Student without KD"),
        ("kd", "(c') KD Student (global)"),
        ("personal", "(d') KD Student + personalised head"),
    ]:
        if cond_name in pilot_results and "error" not in pilot_results[cond_name]:
            r = pilot_results[cond_name]
            print(f"  {label}")
            print(f"    Macro-F1  : {r['macro_f1']:.3f}")
            print(f"    Accuracy  : {r['accuracy']:.3f}")
            print(f"    AUROC     : {r['auroc']:.3f}")
            print(f"    Stress recall    : {r.get('stress_recall', float('nan')):.3f}")
            print(
                f"    Stress precision : {r.get('stress_precision', float('nan')):.3f}"
            )
            print()
        elif cond_name in pilot_results:
            print(
                f"  {label}: ERROR — {pilot_results[cond_name].get('error', 'unknown')}"
            )

    # Quality report — sampling_report is a list in two-session mode, dict otherwise
    sr = pilot_results.get("sampling_report", {})

    if isinstance(sr, list):
        print("  Data quality (two-session pilot):")
        for sess_name, sess_sr in zip(["Cal", "Eval"], sr):
            print(f"    {sess_name} session:")
            print(f"      Max gap (s)      : {sess_sr.get('max_gap_s', float('nan')):.1f}")
            print(f"      Pct 1-sec samples: {sess_sr.get('pct_1sec', float('nan')):.1f}%")
    else:
        print("  Data quality:")
        print(f"    Max gap (s)     : {sr.get('max_gap_s', float('nan')):.1f}")
        print(f"    Pct 1-sec samples: {sr.get('pct_1sec', float('nan')):.1f}%")

    # Window counts (printed during preprocessing; not stored as structured data
    # in current pipeline; refer to console output above for detailed counts).

In [ ]:
# Plot: prediction timeline
from IPython.display import Image
if pilot_results is not None:
    timeline_path = os.path.join(KD_FIGURES_DIR, "fitbit_prediction_timeline.png")
    plot_prediction_timeline(
        results=pilot_results,
        output_path=timeline_path,
    )
    print(f"Timeline saved to: {timeline_path}")
    Image(timeline_path)
else:
    print("[Skipped] No pilot results to plot.")

In [ ]:
# ── Per-phase confusion matrix (Fitbit pilot) ──
if pilot_results is not None:
    import numpy as np

    from src.kd.kd_visualizations import plot_confusion_matrices

    pilot_cms = {}
    for cond in ["no_kd", "kd", "personal"]:
        if cond in pilot_results and "confusion_matrix" in pilot_results[cond]:
            pilot_cms[cond] = np.array(pilot_results[cond]["confusion_matrix"])

    if pilot_cms:
        fig_pilot_cm = plot_confusion_matrices(pilot_cms)
        save_figure(fig_pilot_cm, "fitbit_confusion_matrices", KD_FIGURES_DIR)
        fig_pilot_cm.savefig("/tmp/kd_pilot_cm.png", dpi=150, bbox_inches="tight")
        Image("/tmp/kd_pilot_cm.png")
    else:
        print("No confusion matrix data in pilot results.")
else:
    print("[Skipped] No pilot results available.")